In [ ]:

# =============================================================================
# TCN V2 COMPETING-RISK — CELL 1
# ENVIRONMENT, FROZEN RELEASE AND TRAIN DATA INTEGRITY
#
# NO TRAINING
# NO OFFICIAL VALIDATION OR TEST DATA LOADED
# NO MODIFICATIONS TO EXISTING MODELS OR DATA
# =============================================================================

from pathlib import Path
import hashlib
import json
import random
import platform

import numpy as np
import pandas as pd
import torch

from google.colab import drive

# -----------------------------------------------------------------------------
# 1. Mount Drive
# -----------------------------------------------------------------------------

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/network-attack-forecasting")

DATA_DIR = ROOT / "Synthetic_NAF_v2/release_candidate_corrected"
FREEZE_DIR = ROOT / "Synthetic_NAF_v2/FINAL_RELEASE_FREEZE"

TRAIN_PATH = DATA_DIR / "windows_W10_TRAIN.npz"

META_PATH = (
    DATA_DIR / "episode_metadata_DO_NOT_USE_AS_PREDICTORS.csv"
)

CONTRACT_PATH = DATA_DIR / "FEATURE_AND_DATASET_CONTRACT.json"

MANIFEST_PATH = (
    FREEZE_DIR / "Synthetic_NAF_v2_RELEASE_MANIFEST.json"
)

LOCK_PATH = (
    FREEZE_DIR / "Synthetic_NAF_v2_FREEZE_LOCK.json"
)

HASHES_PATH = DATA_DIR / "SHA256SUMS.json"

TCN_MODEL_ROOT = (
    ROOT / "Synthetic_NAF_v2/models/TCN_V2_CompetingRisk"
)

# Do not create or overwrite model directories in this setup cell.

# -----------------------------------------------------------------------------
# 2. Integrity helpers
# -----------------------------------------------------------------------------

def load_json(path):
    if not path.is_file():
        raise FileNotFoundError(f"Required file missing: {path}")

    return json.loads(path.read_text(encoding="utf-8"))


def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)

    return h.hexdigest()


def canonical_sha256(obj):
    payload = json.dumps(
        obj,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False
    ).encode("utf-8")

    return hashlib.sha256(payload).hexdigest()


# -----------------------------------------------------------------------------
# 3. Verify required files
# -----------------------------------------------------------------------------

required_files = [
    TRAIN_PATH,
    META_PATH,
    CONTRACT_PATH,
    MANIFEST_PATH,
    LOCK_PATH,
    HASHES_PATH
]

for path in required_files:
    if not path.is_file():
        raise FileNotFoundError(path)

MANIFEST = load_json(MANIFEST_PATH)
FREEZE_LOCK = load_json(LOCK_PATH)
CONTRACT = load_json(CONTRACT_PATH)

assert MANIFEST["release_version"] == "2.0.0"

assert (
    MANIFEST["release_status"]
    == "FROZEN_CONTROLLED_SYNTHETIC_BENCHMARK"
)

manifest_hash = canonical_sha256(MANIFEST)

assert (
    manifest_hash
    == FREEZE_LOCK["manifest_sha256_canonical"]
), "Frozen manifest hash mismatch."

# Verify all files in the release checksum inventory.
for filename, expected_hash in load_json(HASHES_PATH).items():
    actual_hash = sha256_file(DATA_DIR / filename)

    if actual_hash != expected_hash:
        raise RuntimeError(
            f"Release checksum mismatch: {filename}"
        )

# Independently verify all manifest-listed release files.
for filename, expected_hash in (
    MANIFEST["data_integrity"]["file_sha256"].items()
):
    actual_hash = sha256_file(DATA_DIR / filename)

    if actual_hash != expected_hash:
        raise RuntimeError(
            f"Manifest checksum mismatch: {filename}"
        )

# -----------------------------------------------------------------------------
# 4. Verify feature and prediction contract
# -----------------------------------------------------------------------------

FEATURES = list(CONTRACT["features"])
FAMILIES = list(CONTRACT["attack_families"])

HORIZONS = [5, 10, 15]
HISTORY_MINUTES = 10
NUM_FEATURES = 14
NUM_FAMILIES = 5
NUM_JOINT_CLASSES = 76

assert len(FEATURES) == NUM_FEATURES
assert len(FAMILIES) == NUM_FAMILIES

assert CONTRACT["history_minutes"] == HISTORY_MINUTES
assert CONTRACT["forecast_horizons"] == HORIZONS

assert FAMILIES == [
    "BRUTE_AUTH",
    "DOS_DDOS",
    "EXPLOIT_APP",
    "FUZZING",
    "RECON_SCAN"
]

FAMILY_TO_ID = {
    name: idx + 1
    for idx, name in enumerate(FAMILIES)
}

ID_TO_FAMILY = {
    idx + 1: name
    for idx, name in enumerate(FAMILIES)
}

# Joint target:
# 0 -> no onset within 15 minutes
# 1..75 -> 5 * (onset_minute - 1) + family_id

assert 1 + 15 * NUM_FAMILIES == NUM_JOINT_CLASSES

# -----------------------------------------------------------------------------
# 5. Verify TRAIN arrays without accessing VAL / TEST
# -----------------------------------------------------------------------------

with np.load(TRAIN_PATH, allow_pickle=False) as data:

    required_keys = {
        "X",
        "y_horizons",
        "episode_id",
        "minute",
        "onset_class",
        "family_class"
    }

    assert required_keys.issubset(set(data.files))

    train_shape = data["X"].shape
    target_shape = data["y_horizons"].shape

    onset = data["onset_class"]
    family = data["family_class"]
    horizon_y = data["y_horizons"]

    episode_ids = (
        pd.Series(data["episode_id"])
        .astype(str)
        .to_numpy()
    )

    assert train_shape == (88186, 10, 14)
    assert target_shape == (88186, 3)

    assert np.all((onset >= 0) & (onset <= 15))
    assert np.all((family >= 0) & (family <= 5))
    assert np.array_equal(onset == 0, family == 0)

    for col, horizon in enumerate(HORIZONS):
        expected_y = (
            (onset >= 1) & (onset <= horizon)
        ).astype(np.int8)

        assert np.array_equal(
            horizon_y[:, col],
            expected_y
        )

    joint_target = np.where(
        onset == 0,
        0,
        5 * (onset - 1) + family
    )

    assert joint_target.min() == 0
    assert joint_target.max() <= 75

    observed_classes = np.unique(joint_target)

    assert len(observed_classes) == 76
    assert len(np.unique(episode_ids)) == 1400

    no_event_count = int(np.sum(joint_target == 0))
    event_count = int(np.sum(joint_target > 0))

# -----------------------------------------------------------------------------
# 6. Environment and reproducibility
# -----------------------------------------------------------------------------

SEED = 20261024

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("=" * 100)
print("TCN V2 COMPETING-RISK — CELL 1 VERIFIED")
print("=" * 100)

print("Release:", MANIFEST["release_version"])
print("Manifest SHA256:", manifest_hash)

print("\nENVIRONMENT")
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: CUDA unavailable. Select a GPU before training.")

print("\nDATASET")
print("TRAIN shape:", train_shape)
print("TRAIN episodes:", len(np.unique(episode_ids)))
print("Input features:", len(FEATURES))
print("Attack families:", FAMILIES)

print("\nCOMPETING-RISK TARGET")
print("Joint classes:", len(observed_classes))
print("No-onset windows:", no_event_count)
print("Future-onset windows:", event_count)
print("Forecast horizons:", HORIZONS)

print("\nINTEGRITY")
print("Frozen release: PASS")
print("TRAIN dataset checksum: PASS")
print("Horizon target reconstruction: PASS")
print("76-class target coverage: PASS")

print("\nSAFETY")
print("Existing LSTM models: UNCHANGED")
print("Existing XGBoost models: UNCHANGED")
print("Official VALIDATION: NOT LOADED")
print("Official TEST: NOT LOADED")
print("Training: NOT STARTED")

print("=" * 100)


In [ ]:

# =============================================================================
# TCN V2 COMPETING-RISK — CELL 2
# TRAIN DATA, JOINT TARGETS, GROUP-DISJOINT CV
#
# NO TRAINING. NO OFFICIAL VALIDATION OR TEST ACCESS.
# =============================================================================

from sklearn.model_selection import GroupKFold

# 1. Load frozen TRAIN arrays
with np.load(TRAIN_PATH, allow_pickle=False) as pack:
    X_RAW = pack["X"].astype(np.float32)
    Y_HORIZONS = pack["y_horizons"].astype(np.int8)
    ONSET = pack["onset_class"].astype(np.int16)
    FAMILY = pack["family_class"].astype(np.int16)
    EPISODE_ID = pd.Series(
        pack["episode_id"]
    ).astype(str).to_numpy()
    MINUTE = pack["minute"].astype(np.int16)

assert X_RAW.shape == (88186, 10, 14)
assert Y_HORIZONS.shape == (88186, 3)
assert np.isfinite(X_RAW).all()
assert (X_RAW >= 0).all()

# 2. Same joint encoding as competing-risk LSTM
# 0 = no onset within 15 minutes
# 1..75 = 5 * (onset minute - 1) + family ID
JOINT_TARGET = np.where(
    ONSET == 0,
    0,
    5 * (ONSET - 1) + FAMILY
).astype(np.int64)

assert set(np.unique(JOINT_TARGET)) == set(range(76))
assert np.array_equal(ONSET == 0, FAMILY == 0)

for j, h in enumerate([5, 10, 15]):
    expected = (
        (ONSET >= 1) & (ONSET <= h)
    ).astype(np.int8)
    assert np.array_equal(Y_HORIZONS[:, j], expected)

# 3. Metadata is for grouping/auditing only, never predictors
META = pd.read_csv(META_PATH)
META["Episode_ID"] = META["Episode_ID"].astype(str)

assert META["Episode_ID"].is_unique

TRAIN_META = META.loc[
    META["Split"].astype(str) == "TRAIN"
].set_index("Episode_ID")

assert set(np.unique(EPISODE_ID)) == set(TRAIN_META.index)

SOURCE = TRAIN_META.loc[
    EPISODE_ID, "Source"
].astype(str).to_numpy()

CAL_GROUP = TRAIN_META.loc[
    EPISODE_ID, "Calibration_Group_ID"
].astype(str).to_numpy()

CV_GROUPS = np.array([
    f"{src}::{group}"
    for src, group in zip(SOURCE, CAL_GROUP)
])

assert len(np.unique(CV_GROUPS)) == 35
assert len(np.unique(EPISODE_ID)) == 1400

# 4. GroupKFold: only TRAIN windows are used
CV_FOLDS = list(
    GroupKFold(n_splits=5).split(
        X_RAW,
        JOINT_TARGET,
        groups=CV_GROUPS
    )
)

fold_summary = []

for fold_id, (fit_idx, hold_idx) in enumerate(
    CV_FOLDS, start=1
):
    fit_groups = set(CV_GROUPS[fit_idx])
    hold_groups = set(CV_GROUPS[hold_idx])

    fit_episodes = set(EPISODE_ID[fit_idx])
    hold_episodes = set(EPISODE_ID[hold_idx])

    assert fit_groups.isdisjoint(hold_groups)
    assert fit_episodes.isdisjoint(hold_episodes)

    assert len(hold_groups) == 7
    assert len(fit_groups) == 28
    assert set(np.unique(JOINT_TARGET[fit_idx])) == set(range(76))
    assert set(np.unique(JOINT_TARGET[hold_idx])) == set(range(76))

    row = {
        "Fold": fold_id,
        "Fit_Windows": len(fit_idx),
        "Holdout_Windows": len(hold_idx),
        "Fit_Groups": len(fit_groups),
        "Holdout_Groups": len(hold_groups),
        "Holdout_Event_Prevalence": float(
            (JOINT_TARGET[hold_idx] > 0).mean()
        )
    }

    fold_summary.append(row)

CV_SUMMARY = pd.DataFrame(fold_summary)

assert sum(
    len(hold_idx) for _, hold_idx in CV_FOLDS
) == len(X_RAW)

# 5. Training-only robust preprocessing
class TemporalRobustScaler:
    def __init__(self, clip_value=8.0):
        self.clip_value = float(clip_value)
        self.median_ = None
        self.scale_ = None

    def fit(self, X):
        assert X.ndim == 3 and X.shape[1:] == (10, 14)
        logged = np.log1p(X.astype(np.float32))
        flat = logged.reshape(-1, 14)

        q25, q50, q75 = np.quantile(
            flat, [0.25, 0.50, 0.75], axis=0
        )
        self.median_ = q50.astype(np.float32)
        self.scale_ = np.maximum(
            (q75 - q25).astype(np.float32), 0.05
        )
        return self

    def transform(self, X):
        if self.median_ is None:
            raise RuntimeError("Fit the scaler first.")

        logged = np.log1p(X.astype(np.float32))
        scaled = (
            logged - self.median_[None, None, :]
        ) / self.scale_[None, None, :]

        return np.ascontiguousarray(
            np.clip(scaled, -self.clip_value, self.clip_value),
            dtype=np.float32
        )

    def state_dict(self):
        return {
            "median": self.median_.tolist(),
            "scale": self.scale_.tolist(),
            "clip_value": self.clip_value,
            "feature_names": list(FEATURES)
        }

# Smoke-test scaler on fit-only data
fit_idx, hold_idx = CV_FOLDS[0]
PREVIEW_SCALER = TemporalRobustScaler().fit(X_RAW[fit_idx])

preview_fit = PREVIEW_SCALER.transform(X_RAW[fit_idx[:64]])
preview_hold = PREVIEW_SCALER.transform(X_RAW[hold_idx[:64]])

assert preview_fit.shape == (64, 10, 14)
assert preview_hold.shape == (64, 10, 14)
assert np.isfinite(preview_fit).all()
assert np.isfinite(preview_hold).all()

print("=" * 100)
print("TCN V2 COMPETING-RISK — CELL 2 COMPLETE")
print("=" * 100)
print("TRAIN shape:", X_RAW.shape)
print("Joint classes:", len(np.unique(JOINT_TARGET)))
print("No-onset windows:", int((JOINT_TARGET == 0).sum()))
print("Future-onset windows:", int((JOINT_TARGET > 0).sum()))
print("Calibration groups:", len(np.unique(CV_GROUPS)))

print("\nGROUP-DISJOINT FIVE-FOLD CV")
print(CV_SUMMARY.round(6).to_string(index=False))

print("\nChecks:")
print("✓ Episode and calibration-group overlap: ZERO")
print("✓ All 76 classes present in every fold")
print("✓ Fold-local scaling verified")
print("✓ No data leakage from holdout into scaler fitting")
print("✓ Official VALIDATION/TEST not loaded")
print("✓ Training not started")


In [ ]:

# =============================================================================
# TCN V2 COMPETING-RISK — CELL 3
# CAUSAL TEMPORAL CONVOLUTIONAL NETWORK
#
# NO TRAINING
# NO OFFICIAL VALIDATION OR TEST ACCESS
# =============================================================================

import torch
import torch.nn as nn
import torch.nn.functional as F

# ---------------------------------------------------------------------------
# 1. Strictly causal 1D convolution
# ---------------------------------------------------------------------------

class CausalConv1d(nn.Module):
    """
    Left-padded causal convolution.

    Output at timestep t depends only on inputs
    at timesteps <= t.
    """

    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size=3,
        dilation=1
    ):
        super().__init__()

        self.left_padding = (
            (kernel_size - 1) * dilation
        )

        self.conv = nn.Conv1d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=kernel_size,
            dilation=dilation,
            padding=0
        )

    def forward(self, x):
        x = F.pad(
            x,
            (self.left_padding, 0)
        )
        return self.conv(x)


# ---------------------------------------------------------------------------
# 2. Residual temporal block
# ---------------------------------------------------------------------------

class TemporalResidualBlock(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size,
        dilation,
        dropout
    ):
        super().__init__()

        self.conv1 = CausalConv1d(
            in_channels,
            out_channels,
            kernel_size,
            dilation
        )

        self.conv2 = CausalConv1d(
            out_channels,
            out_channels,
            kernel_size,
            dilation
        )

        self.relu1 = nn.ReLU()
        self.relu2 = nn.ReLU()

        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

        self.residual = (
            nn.Identity()
            if in_channels == out_channels
            else nn.Conv1d(
                in_channels,
                out_channels,
                kernel_size=1
            )
        )

    def forward(self, x):

        residual = self.residual(x)

        out = self.conv1(x)
        out = self.relu1(out)
        out = self.dropout1(out)

        out = self.conv2(out)
        out = self.relu2(out)
        out = self.dropout2(out)

        return F.relu(out + residual)


# ---------------------------------------------------------------------------
# 3. Competing-risk TCN
# ---------------------------------------------------------------------------

class TCNCompetingRisk(nn.Module):
    """
    Input:  (batch, 10, 14)
    Output: (batch, 76) raw logits

    Uses the final causal timestep representation
    for joint event-time and family classification.
    """

    def __init__(
        self,
        input_dim=14,
        channels=64,
        num_blocks=3,
        kernel_size=3,
        dropout=0.20,
        dense_dim=32,
        num_classes=76
    ):
        super().__init__()

        layers = []

        in_channels = input_dim

        for block_idx in range(num_blocks):

            dilation = 2 ** block_idx

            layers.append(
                TemporalResidualBlock(
                    in_channels=in_channels,
                    out_channels=channels,
                    kernel_size=kernel_size,
                    dilation=dilation,
                    dropout=dropout
                )
            )

            in_channels = channels

        self.tcn = nn.Sequential(*layers)

        self.projection = nn.Sequential(
            nn.Linear(channels, dense_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.classifier = nn.Linear(
            dense_dim,
            num_classes
        )

    def encode_sequence(self, x):
        """
        Returns representation at every input timestep.

        Useful for inspecting causality.
        """
        assert x.ndim == 3
        assert x.shape[-1] == 14

        x = x.transpose(1, 2)

        features = self.tcn(x)

        return features.transpose(1, 2)

    def forward(self, x):

        features = self.encode_sequence(x)

        last_step = features[:, -1, :]

        projected = self.projection(last_step)

        return self.classifier(projected)


# ---------------------------------------------------------------------------
# 4. Architecture configuration
# ---------------------------------------------------------------------------

TCN_MODEL_CONFIG = {
    "input_dim": 14,
    "channels": 64,
    "num_blocks": 3,
    "kernel_size": 3,
    "dropout": 0.20,
    "dense_dim": 32,
    "num_classes": 76
}

torch.manual_seed(20261024)

TCN_MODEL = TCNCompetingRisk(
    **TCN_MODEL_CONFIG
)

parameter_count = sum(
    p.numel()
    for p in TCN_MODEL.parameters()
    if p.requires_grad
)

# Receptive field:
# each temporal block has 2 convolutions.
# With dilations 1, 2, 4 and kernel size 3:
# RF = 1 + 2 * (3 - 1) * (1 + 2 + 4) = 29.
RECEPTIVE_FIELD = (
    1
    + 2
    * (TCN_MODEL_CONFIG["kernel_size"] - 1)
    * sum(
        2 ** i
        for i in range(TCN_MODEL_CONFIG["num_blocks"])
    )
)

assert RECEPTIVE_FIELD >= 10

# ---------------------------------------------------------------------------
# 5. Basic shape and numerical smoke tests
# ---------------------------------------------------------------------------

TCN_MODEL.eval()

x = torch.randn(8, 10, 14)

with torch.inference_mode():

    encoded = TCN_MODEL.encode_sequence(x)
    logits = TCN_MODEL(x)
    probabilities = torch.softmax(logits, dim=1)

assert encoded.shape == (8, 10, 64)
assert logits.shape == (8, 76)
assert probabilities.shape == (8, 76)

assert torch.isfinite(logits).all()
assert torch.isfinite(probabilities).all()

assert torch.allclose(
    probabilities.sum(dim=1),
    torch.ones(8),
    atol=1e-6
)

# ---------------------------------------------------------------------------
# 6. Causality test
#
# Modify the future half of an input sequence.
# Earlier representations MUST remain unchanged.
# ---------------------------------------------------------------------------

x_original = torch.randn(4, 10, 14)
x_modified = x_original.clone()

# Change minutes 6..10 (zero-based positions 5..9).
x_modified[:, 5:, :] = (
    torch.randn_like(x_modified[:, 5:, :]) * 100
)

with torch.inference_mode():

    encoded_original = TCN_MODEL.encode_sequence(
        x_original
    )

    encoded_modified = TCN_MODEL.encode_sequence(
        x_modified
    )

# The first 5 representations cannot depend on later inputs.
assert torch.allclose(
    encoded_original[:, :5, :],
    encoded_modified[:, :5, :],
    atol=1e-6,
    rtol=0
), "CAUSALITY VIOLATION: future inputs changed earlier outputs."

# ---------------------------------------------------------------------------
# 7. Joint probability marginalization
# ---------------------------------------------------------------------------

with torch.inference_mode():

    p = probabilities

    no_event_p = p[:, 0]

    event_joint = p[:, 1:].reshape(
        -1, 15, 5
    )

    onset_p = event_joint.sum(dim=2)
    family_p = event_joint.sum(dim=1)

    horizon_p = onset_p.cumsum(dim=1)[:, [4, 9, 14]]

    assert horizon_p.shape == (8, 3)
    assert family_p.shape == (8, 5)
    assert onset_p.shape == (8, 15)

    assert torch.allclose(
        no_event_p + horizon_p[:, 2],
        torch.ones(8),
        atol=1e-6
    )

    assert torch.allclose(
        family_p.sum(dim=1),
        horizon_p[:, 2],
        atol=1e-6
    )

    assert torch.all(
        horizon_p[:, 0] <= horizon_p[:, 1] + 1e-7
    )

    assert torch.all(
        horizon_p[:, 1] <= horizon_p[:, 2] + 1e-7
    )

# ---------------------------------------------------------------------------
# 8. Architecture report
# ---------------------------------------------------------------------------

print("=" * 100)
print("TCN V2 COMPETING-RISK — CELL 3 VERIFIED")
print("=" * 100)

print("\nARCHITECTURE")
print(TCN_MODEL)

print("\nPARAMETERS")
print("Trainable parameters:", parameter_count)
print("Temporal blocks:", TCN_MODEL_CONFIG["num_blocks"])
print("Kernel size:", TCN_MODEL_CONFIG["kernel_size"])
print("Dilations:", [1, 2, 4])
print("Theoretical receptive field:", RECEPTIVE_FIELD)
print("History window:", 10)

print("\nSHAPES")
print("Input:", tuple(x.shape))
print("Encoded sequence:", tuple(encoded.shape))
print("Output logits:", tuple(logits.shape))

print("\nVERIFICATION")
print("✓ Causal convolution test: PASS")
print("✓ 76-class probability normalization: PASS")
print("✓ H5/H10/H15 temporal monotonicity: PASS")
print("✓ Family probability marginalization: PASS")
print("✓ No-event complement: PASS")

print("\nTraining: NOT STARTED")
print("Official VALIDATION/TEST: NOT LOADED")
print("Frozen dataset: UNCHANGED")
print("=" * 100)


In [ ]:

# =============================================================================
# TCN V2 COMPETING-RISK — CELL 4
# FOLD 1 TRAINING DIAGNOSTIC
#
# TRAIN-ONLY GroupKFold fold 1
# 8 epochs, unweighted 76-class cross-entropy
# Fresh TCN and fold-local scaler
#
# NO official VALIDATION or TEST access
# NO final model saved or trained
# =============================================================================

import random
import time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import average_precision_score, f1_score

# ---------------------------------------------------------------------------
# 1. Configuration
# ---------------------------------------------------------------------------

SANITY_SEED = 20261024
SANITY_EPOCHS = 8
SANITY_BATCH_SIZE = 256

SANITY_DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

if SANITY_DEVICE.type != "cuda":
    raise RuntimeError(
        "CUDA unavailable. Enable T4 GPU and rerun Cells 1-3. "
        "Training has not started."
    )

random.seed(SANITY_SEED)
np.random.seed(SANITY_SEED)
torch.manual_seed(SANITY_SEED)
torch.cuda.manual_seed_all(SANITY_SEED)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

# ---------------------------------------------------------------------------
# 2. Verify data and group-disjoint fold
# ---------------------------------------------------------------------------

assert X_RAW.shape == (88186, 10, 14)
assert JOINT_TARGET.shape == (88186,)
assert Y_HORIZONS.shape == (88186, 3)
assert len(CV_FOLDS) == 5

fit_idx, hold_idx = CV_FOLDS[0]

assert set(CV_GROUPS[fit_idx]).isdisjoint(
    set(CV_GROUPS[hold_idx])
)

assert set(EPISODE_ID[fit_idx]).isdisjoint(
    set(EPISODE_ID[hold_idx])
)

assert set(np.unique(JOINT_TARGET[fit_idx])) == set(range(76))
assert set(np.unique(JOINT_TARGET[hold_idx])) == set(range(76))

# ---------------------------------------------------------------------------
# 3. Fit a new scaler on fold 1 FIT ONLY
# ---------------------------------------------------------------------------

SANITY_SCALER = TemporalRobustScaler().fit(
    X_RAW[fit_idx]
)

X_FIT = SANITY_SCALER.transform(
    X_RAW[fit_idx]
)

X_HOLD = SANITY_SCALER.transform(
    X_RAW[hold_idx]
)

Y_FIT = np.ascontiguousarray(
    JOINT_TARGET[fit_idx],
    dtype=np.int64
)

Y_HOLD = np.ascontiguousarray(
    JOINT_TARGET[hold_idx],
    dtype=np.int64
)

assert np.isfinite(X_FIT).all()
assert np.isfinite(X_HOLD).all()

train_dataset = TensorDataset(
    torch.from_numpy(X_FIT),
    torch.from_numpy(Y_FIT)
)

hold_dataset = TensorDataset(
    torch.from_numpy(X_HOLD),
    torch.from_numpy(Y_HOLD)
)

generator = torch.Generator().manual_seed(SANITY_SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=SANITY_BATCH_SIZE,
    shuffle=True,
    generator=generator,
    num_workers=0,
    pin_memory=True
)

hold_loader = DataLoader(
    hold_dataset,
    batch_size=1024,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

# ---------------------------------------------------------------------------
# 4. Initialize NEW TCN
# ---------------------------------------------------------------------------

SANITY_TCN = TCNCompetingRisk(
    **TCN_MODEL_CONFIG
).to(SANITY_DEVICE)

optimizer = torch.optim.AdamW(
    SANITY_TCN.parameters(),
    lr=0.001,
    weight_decay=0.001
)

print("=" * 100)
print("TCN V2 COMPETING-RISK — FOLD 1 DIAGNOSTIC")
print("=" * 100)

print("Device:", SANITY_DEVICE)
print("GPU:", torch.cuda.get_device_name(0))
print("Fit windows:", len(fit_idx))
print("Holdout windows:", len(hold_idx))
print("Epochs:", SANITY_EPOCHS)
print("Trainable parameters:", sum(
    p.numel() for p in SANITY_TCN.parameters()
))
print("Official VALIDATION/TEST: NOT ACCESSED")
print("Training: STARTING")

# ---------------------------------------------------------------------------
# 5. Diagnostic training
# ---------------------------------------------------------------------------

SANITY_HISTORY = []
start = time.perf_counter()

for epoch in range(1, SANITY_EPOCHS + 1):

    SANITY_TCN.train()

    running_loss = 0.0
    samples_seen = 0

    for xb, yb in train_loader:

        xb = xb.to(SANITY_DEVICE, non_blocking=True)
        yb = yb.to(SANITY_DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        logits = SANITY_TCN(xb)
        loss = F.cross_entropy(logits, yb)

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Nonfinite training loss at epoch {epoch}"
            )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            SANITY_TCN.parameters(),
            max_norm=1.0,
            error_if_nonfinite=True
        )

        optimizer.step()

        running_loss += float(loss.item()) * len(xb)
        samples_seen += len(xb)

    assert samples_seen == len(Y_FIT)

    train_nll = running_loss / samples_seen

    # -----------------------------------------------------------------------
    # TRAIN fold 1 holdout evaluation
    # -----------------------------------------------------------------------

    SANITY_TCN.eval()

    hold_loss_sum = 0.0
    hold_seen = 0
    all_logits = []

    with torch.inference_mode():

        for xb, yb in hold_loader:

            xb = xb.to(SANITY_DEVICE, non_blocking=True)
            yb = yb.to(SANITY_DEVICE, non_blocking=True)

            logits = SANITY_TCN(xb)

            hold_loss_sum += float(
                F.cross_entropy(
                    logits,
                    yb,
                    reduction="sum"
                ).item()
            )

            hold_seen += len(yb)

            all_logits.append(
                logits.cpu()
            )

    assert hold_seen == len(Y_HOLD)

    hold_nll = hold_loss_sum / hold_seen

    logits_all = torch.cat(all_logits, dim=0)

    with torch.inference_mode():

        joint_probs = torch.softmax(
            logits_all,
            dim=1
        ).numpy()

    assert joint_probs.shape == (len(Y_HOLD), 76)

    assert np.allclose(
        joint_probs.sum(axis=1),
        1.0,
        atol=1e-6
    )

    # Event-time x family joint probability matrix
    event_matrix = joint_probs[:, 1:].reshape(
        -1, 15, 5
    )

    onset_probs = event_matrix.sum(axis=2)
    family_probs = event_matrix.sum(axis=1)

    horizon_probs = (
        onset_probs.cumsum(axis=1)[:, [4, 9, 14]]
    )

    assert np.all(
        horizon_probs[:, 0]
        <= horizon_probs[:, 1] + 1e-7
    )

    assert np.all(
        horizon_probs[:, 1]
        <= horizon_probs[:, 2] + 1e-7
    )

    # H5/H10/H15 Average Precision
    true_horizons = Y_HORIZONS[hold_idx]

    ap = [
        float(
            average_precision_score(
                true_horizons[:, j],
                horizon_probs[:, j]
            )
        )
        for j in range(3)
    ]

    # -----------------------------------------------------------------------
    # Conditional attack-family and onset-time diagnostics
    #
    # Evaluation subset: known true onset within 15 minutes.
    # Not an end-to-end alert performance measure.
    # -----------------------------------------------------------------------

    true_onset = ONSET[hold_idx]
    true_family = FAMILY[hold_idx]

    event_mask = true_onset > 0

    event_probability = np.maximum(
        horizon_probs[:, 2],
        1e-12
    )

    conditional_family_probs = (
        family_probs / event_probability[:, None]
    )

    conditional_onset_probs = (
        onset_probs / event_probability[:, None]
    )

    predicted_family = (
        conditional_family_probs.argmax(axis=1) + 1
    )

    family_acc = float(
        np.mean(
            predicted_family[event_mask]
            == true_family[event_mask]
        )
    )

    family_macro_f1 = float(
        f1_score(
            true_family[event_mask],
            predicted_family[event_mask],
            labels=[1, 2, 3, 4, 5],
            average="macro",
            zero_division=0
        )
    )

    onset_minutes = np.arange(
        1, 16,
        dtype=np.float64
    )

    expected_eta = (
        conditional_onset_probs @ onset_minutes
    )

    eta_mae = float(
        np.mean(
            np.abs(
                expected_eta[event_mask]
                - true_onset[event_mask]
            )
        )
    )

    result = {
        "epoch": epoch,
        "train_nll": float(train_nll),
        "hold_nll": float(hold_nll),
        "H5_AP": ap[0],
        "H10_AP": ap[1],
        "H15_AP": ap[2],
        "family_accuracy": family_acc,
        "family_macro_f1": family_macro_f1,
        "eta_mae": eta_mae
    }

    SANITY_HISTORY.append(result)

    print(
        f"Epoch {epoch:02d}/{SANITY_EPOCHS} | "
        f"Train NLL={train_nll:.5f} | "
        f"Hold NLL={hold_nll:.5f} | "
        f"H5 AP={ap[0]:.5f} | "
        f"H10 AP={ap[1]:.5f} | "
        f"H15 AP={ap[2]:.5f} | "
        f"Family Acc={family_acc:.4f} | "
        f"Family F1={family_macro_f1:.4f} | "
        f"ETA MAE={eta_mae:.3f}m"
    )

SANITY_HISTORY_DF = pd.DataFrame(
    SANITY_HISTORY
)

print("\n" + "=" * 100)
print("TCN V2 COMPETING-RISK — SANITY FIT COMPLETE")
print("=" * 100)

print(
    SANITY_HISTORY_DF.round(6).to_string(index=False)
)

print(
    "\nElapsed seconds:",
    round(time.perf_counter() - start, 2)
)

print("Optimizer training epochs:", SANITY_EPOCHS)
print("Official VALIDATION: NOT ACCESSED")
print("Official TEST: NOT ACCESSED")
print("Final TCN model: NOT TRAINED")
print("Frozen dataset: UNCHANGED")


In [ ]:

# =============================================================================
# TCN V2 COMPETING-RISK — CELL 5
# FULL FIVE-FOLD TRAIN-ONLY HYPERPARAMETER SEARCH
#
# 8 configurations x 5 GroupKFold splits = 40 training runs
#
# PRIMARY SELECTION:
#   Lowest mean group-disjoint holdout joint NLL
#
# FOLD CHECKPOINT SELECTION:
#   Lowest holdout joint NLL
#
# No official VALIDATION or TEST access.
# No final full-TRAIN fitting.
# No modification of frozen data or earlier models.
# =============================================================================

import os
import gc
import json
import time
import random
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import average_precision_score, f1_score

# =============================================================================
# 5A — SEARCH CONFIGURATIONS
# =============================================================================

TCN_SEARCH_CONFIGS = [
    {
        "id": "C01",
        "channels": 64,
        "num_blocks": 3,
        "dropout": 0.20,
        "learning_rate": 0.001
    },
    {
        "id": "C02",
        "channels": 32,
        "num_blocks": 3,
        "dropout": 0.20,
        "learning_rate": 0.001
    },
    {
        "id": "C03",
        "channels": 128,
        "num_blocks": 3,
        "dropout": 0.20,
        "learning_rate": 0.001
    },
    {
        "id": "C04",
        "channels": 64,
        "num_blocks": 2,
        "dropout": 0.20,
        "learning_rate": 0.001
    },
    {
        "id": "C05",
        "channels": 64,
        "num_blocks": 3,
        "dropout": 0.10,
        "learning_rate": 0.001
    },
    {
        "id": "C06",
        "channels": 64,
        "num_blocks": 3,
        "dropout": 0.35,
        "learning_rate": 0.001
    },
    {
        "id": "C07",
        "channels": 64,
        "num_blocks": 3,
        "dropout": 0.20,
        "learning_rate": 0.0003
    },
    {
        "id": "C08",
        "channels": 64,
        "num_blocks": 3,
        "dropout": 0.20,
        "learning_rate": 0.002
    }
]

TCN_SEARCH_SEED = 20261024

TCN_MAX_EPOCHS = 35
TCN_PATIENCE = 7
TCN_MIN_DELTA = 0.0001

TCN_BATCH_SIZE = 256
TCN_WEIGHT_DECAY = 0.001
TCN_GRAD_CLIP = 1.0

TCN_DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

TCN_CV_ROOT = (
    ROOT
    / "Synthetic_NAF_v2"
    / "models"
    / "TCN_V2_CompetingRisk"
    / "train_cv_selection_GPU"
)

# =============================================================================
# 5B — PREREQUISITES AND LEAKAGE CHECKS
# =============================================================================

required_variables = [
    "X_RAW",
    "Y_HORIZONS",
    "ONSET",
    "FAMILY",
    "JOINT_TARGET",
    "EPISODE_ID",
    "CV_GROUPS",
    "CV_FOLDS",
    "FEATURES",
    "FAMILIES",
    "TemporalRobustScaler",
    "TCNCompetingRisk",
    "MANIFEST",
    "TRAIN_PATH"
]

missing = [
    name for name in required_variables
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Run TCN preparation Cells 1–3 first. "
        f"Missing: {missing}"
    )

if TCN_DEVICE.type != "cuda":
    raise RuntimeError(
        "CUDA unavailable. Select T4 GPU and "
        "rerun Cells 1–3 before training."
    )

assert X_RAW.shape == (88186, 10, 14)
assert Y_HORIZONS.shape == (88186, 3)
assert JOINT_TARGET.shape == (88186,)
assert len(CV_FOLDS) == 5

assert set(np.unique(JOINT_TARGET)) == set(range(76))
assert len(np.unique(EPISODE_ID)) == 1400
assert len(np.unique(CV_GROUPS)) == 35

assert np.isfinite(X_RAW).all()

expected_train_hash = (
    MANIFEST["data_integrity"]["file_sha256"]
    ["windows_W10_TRAIN.npz"]
)

assert sha256_file(TRAIN_PATH) == expected_train_hash

fold_descriptions = []

for fold_id, (fit_idx, hold_idx) in enumerate(
    CV_FOLDS, start=1
):

    assert set(CV_GROUPS[fit_idx]).isdisjoint(
        set(CV_GROUPS[hold_idx])
    )

    assert set(EPISODE_ID[fit_idx]).isdisjoint(
        set(EPISODE_ID[hold_idx])
    )

    assert len(np.unique(CV_GROUPS[hold_idx])) == 7

    assert set(
        np.unique(JOINT_TARGET[hold_idx])
    ) == set(range(76))

    fold_descriptions.append({
        "fold": fold_id,
        "fit_windows": int(len(fit_idx)),
        "holdout_windows": int(len(hold_idx)),
        "holdout_groups": sorted(
            set(CV_GROUPS[hold_idx].tolist())
        ),
        "holdout_indices_sha256": hashlib.sha256(
            np.asarray(
                hold_idx,
                dtype="<i8"
            ).tobytes()
        ).hexdigest()
    })

# All configurations must retain adequate receptive field.
for config in TCN_SEARCH_CONFIGS:

    receptive_field = (
        1
        + 2 * (3 - 1)
        * sum(
            2 ** i
            for i in range(config["num_blocks"])
        )
    )

    assert receptive_field >= 10

assert len(TCN_SEARCH_CONFIGS) == 8
assert len({
    c["id"] for c in TCN_SEARCH_CONFIGS
}) == 8

print("=" * 100)
print("TCN V2 COMPETING-RISK — FULL FIVE-FOLD MODEL SELECTION")
print("=" * 100)

print("Device:", TCN_DEVICE)
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)

print("\nConfigurations:", len(TCN_SEARCH_CONFIGS))
print("Group-disjoint folds:", len(CV_FOLDS))
print("Total experiments:", 40)
print("Maximum epochs:", TCN_MAX_EPOCHS)
print("Early stopping patience:", TCN_PATIENCE)
print("Batch size:", TCN_BATCH_SIZE)

print("\nPrimary selection: MINIMUM MEAN HOLDOUT JOINT NLL")
print("Official VALIDATION: NOT ACCESSED")
print("Official TEST: NOT ACCESSED")

# =============================================================================
# 5C — REPRODUCIBILITY AND RESUME PROTOCOL
# =============================================================================

def tcn_canonical_hash(obj):
    payload = json.dumps(
        obj,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False
    ).encode("utf-8")

    return hashlib.sha256(payload).hexdigest()


def tcn_atomic_json(path, data):
    path = Path(path)

    tmp = path.with_name(path.name + ".tmp")

    tmp.write_text(
        json.dumps(
            data,
            sort_keys=True,
            indent=2,
            allow_nan=False
        ),
        encoding="utf-8"
    )

    os.replace(tmp, path)


def tcn_load_json(path):
    return json.loads(
        Path(path).read_text(
            encoding="utf-8"
        )
    )


def tcn_seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


TCN_PROTOCOL = {
    "experiment": "TCN_V2_COMPETING_RISK_CV",
    "dataset": "Synthetic_NAF_v2",
    "release_version": "2.0.0",
    "train_sha256": expected_train_hash,
    "feature_names": list(FEATURES),
    "family_names": list(FAMILIES),
    "num_joint_classes": 76,
    "joint_target_encoding": (
        "CLASS_0_NO_EVENT_ELSE_5_TIMES_ONSET_MINUS_1_PLUS_FAMILY"
    ),
    "candidate_configurations": TCN_SEARCH_CONFIGS,
    "architecture": {
        "input_dim": 14,
        "kernel_size": 3,
        "dense_dim": 32,
        "num_classes": 76,
        "dilation_rule": "2**block_index",
        "convolutions_per_block": 2,
        "residual_blocks": True,
        "causal_left_padding": True,
        "final_timestep_pooling": True
    },
    "fold_strategy": "GROUP_K_FOLD_SOURCE_CALIBRATION_GROUP",
    "folds": fold_descriptions,
    "max_epochs": TCN_MAX_EPOCHS,
    "patience": TCN_PATIENCE,
    "min_delta": TCN_MIN_DELTA,
    "batch_size": TCN_BATCH_SIZE,
    "weight_decay": TCN_WEIGHT_DECAY,
    "gradient_clip_norm": TCN_GRAD_CLIP,
    "loss": "UNWEIGHTED_76_CLASS_CROSS_ENTROPY",
    "optimizer": "AdamW",
    "scaler": "FOLD_LOCAL_LOG1P_MEDIAN_IQR_CLIP8",
    "checkpoint_rule": "MINIMUM_FOLD_HOLDOUT_JOINT_NLL",
    "selection_rule": "MINIMUM_MEAN_FOLD_HOLDOUT_JOINT_NLL",
    "final_epoch_rule": "MEDIAN_SELECTED_FOLD_EPOCH",
    "seed": TCN_SEARCH_SEED,
    "device": str(TCN_DEVICE),
    "torch_version": torch.__version__,
    "official_validation_used": False,
    "official_test_used": False
}

TCN_PROTOCOL_HASH = tcn_canonical_hash(
    TCN_PROTOCOL
)

TCN_CV_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

protocol_file = (
    TCN_CV_ROOT / "SEARCH_PROTOCOL.json"
)

if protocol_file.exists():

    saved_protocol = tcn_load_json(
        protocol_file
    )

    if saved_protocol != TCN_PROTOCOL:
        raise RuntimeError(
            "Existing TCN CV results have a different "
            "protocol. Refusing to mix experiments."
        )

else:

    existing_fold_results = list(
        TCN_CV_ROOT.glob("C*/FOLD_*_RESULT.json")
    )

    if existing_fold_results:
        raise RuntimeError(
            "Fold results exist but SEARCH_PROTOCOL.json "
            "is missing. Refusing unsafe resume."
        )

    tcn_atomic_json(
        protocol_file,
        TCN_PROTOCOL
    )

print("\n✓ Locked search protocol")
print("Protocol hash:", TCN_PROTOCOL_HASH)
print("Output directory:", TCN_CV_ROOT)

# =============================================================================
# 5D — PRECOMPUTE FOLD-LOCAL SCALED ARRAYS
# =============================================================================

TCN_FOLD_DATA = []

for fold_id, (fit_idx, hold_idx) in enumerate(
    CV_FOLDS, start=1
):

    scaler = TemporalRobustScaler().fit(
        X_RAW[fit_idx]
    )

    x_fit = scaler.transform(
        X_RAW[fit_idx]
    )

    x_hold = scaler.transform(
        X_RAW[hold_idx]
    )

    assert x_fit.dtype == np.float32
    assert x_hold.dtype == np.float32

    assert np.isfinite(x_fit).all()
    assert np.isfinite(x_hold).all()

    TCN_FOLD_DATA.append({
        "fold": fold_id,
        "x_fit": x_fit,
        "x_hold": x_hold,
        "y_fit": np.ascontiguousarray(
            JOINT_TARGET[fit_idx],
            dtype=np.int64
        ),
        "y_hold": np.ascontiguousarray(
            JOINT_TARGET[hold_idx],
            dtype=np.int64
        ),
        "horizons_hold": Y_HORIZONS[hold_idx],
        "onset_hold": ONSET[hold_idx],
        "family_hold": FAMILY[hold_idx]
    })

print("✓ Five fold-local scalers fitted without holdout leakage")

# =============================================================================
# 5E — METRIC FUNCTION
# =============================================================================

def evaluate_tcn_joint_logits(
    logits_numpy,
    horizon_truth,
    onset_truth,
    family_truth
):

    logits_t = torch.from_numpy(
        logits_numpy.astype(np.float32)
    )

    with torch.inference_mode():
        joint = torch.softmax(
            logits_t,
            dim=1
        ).numpy()

    assert joint.shape[1] == 76

    event_matrix = joint[:, 1:].reshape(
        -1, 15, 5
    )

    onset_probs = event_matrix.sum(axis=2)
    family_probs = event_matrix.sum(axis=1)

    horizon_probs = (
        onset_probs.cumsum(axis=1)[:, [4, 9, 14]]
    )

    assert np.all(
        horizon_probs[:, 0]
        <= horizon_probs[:, 1] + 1e-7
    )

    assert np.all(
        horizon_probs[:, 1]
        <= horizon_probs[:, 2] + 1e-7
    )

    ap_values = [
        float(
            average_precision_score(
                horizon_truth[:, col],
                horizon_probs[:, col]
            )
        )
        for col in range(3)
    ]

    event_mask = onset_truth > 0

    assert event_mask.any()

    # Family/onset are conditional on event within H15.
    event_p = np.maximum(
        horizon_probs[:, 2],
        1e-12
    )

    family_cond = (
        family_probs / event_p[:, None]
    )

    onset_cond = (
        onset_probs / event_p[:, None]
    )

    predicted_family = (
        family_cond.argmax(axis=1) + 1
    )

    family_accuracy = float(
        np.mean(
            predicted_family[event_mask]
            == family_truth[event_mask]
        )
    )

    family_macro_f1 = float(
        f1_score(
            family_truth[event_mask],
            predicted_family[event_mask],
            labels=[1, 2, 3, 4, 5],
            average="macro",
            zero_division=0
        )
    )

    expected_eta = (
        onset_cond
        @ np.arange(1, 16, dtype=np.float64)
    )

    eta_mae = float(
        np.mean(
            np.abs(
                expected_eta[event_mask]
                - onset_truth[event_mask]
            )
        )
    )

    return {
        "H5_AP": ap_values[0],
        "H10_AP": ap_values[1],
        "H15_AP": ap_values[2],
        "Family_Accuracy": family_accuracy,
        "Family_Macro_F1": family_macro_f1,
        "ETA_MAE": eta_mae
    }

# =============================================================================
# 5F — TRAIN A SINGLE CONFIGURATION/FOLD
# =============================================================================

def fit_tcn_cv_fold(config, config_index, data):

    fold_id = data["fold"]

    seed = (
        TCN_SEARCH_SEED
        + config_index * 100
        + fold_id
    )

    tcn_seed_everything(seed)

    model_config = {
        "input_dim": 14,
        "channels": config["channels"],
        "num_blocks": config["num_blocks"],
        "kernel_size": 3,
        "dropout": config["dropout"],
        "dense_dim": 32,
        "num_classes": 76
    }

    model = TCNCompetingRisk(
        **model_config
    ).to(TCN_DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["learning_rate"],
        weight_decay=TCN_WEIGHT_DECAY
    )

    generator = torch.Generator()
    generator.manual_seed(seed)

    fit_loader = DataLoader(
        TensorDataset(
            torch.from_numpy(data["x_fit"]),
            torch.from_numpy(data["y_fit"])
        ),
        batch_size=TCN_BATCH_SIZE,
        shuffle=True,
        generator=generator,
        num_workers=0,
        pin_memory=True
    )

    hold_loader = DataLoader(
        TensorDataset(
            torch.from_numpy(data["x_hold"]),
            torch.from_numpy(data["y_hold"])
        ),
        batch_size=1024,
        shuffle=False,
        num_workers=0,
        pin_memory=True
    )

    best_nll = float("inf")
    best_epoch = None
    best_metrics = None

    bad_epochs = 0
    epoch_history = []

    started = time.perf_counter()

    for epoch in range(1, TCN_MAX_EPOCHS + 1):

        # ---------------------------------------------------------
        # TRAIN
        # ---------------------------------------------------------

        model.train()

        train_loss_sum = 0.0
        train_seen = 0

        for xb, yb in fit_loader:

            xb = xb.to(
                TCN_DEVICE,
                non_blocking=True
            )

            yb = yb.to(
                TCN_DEVICE,
                non_blocking=True
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            logits = model(xb)

            loss = F.cross_entropy(
                logits, yb
            )

            if not torch.isfinite(loss):
                raise RuntimeError(
                    f"Nonfinite train loss: "
                    f"{config['id']} fold {fold_id}"
                )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=TCN_GRAD_CLIP,
                error_if_nonfinite=True
            )

            optimizer.step()

            train_loss_sum += (
                float(loss.item()) * len(xb)
            )

            train_seen += len(xb)

        assert train_seen == len(data["y_fit"])

        train_nll = (
            train_loss_sum / train_seen
        )

        # ---------------------------------------------------------
        # GROUP-DISJOINT HOLDOUT
        # ---------------------------------------------------------

        model.eval()

        hold_loss_sum = 0.0
        hold_seen = 0

        logits_parts = []

        with torch.inference_mode():

            for xb, yb in hold_loader:

                xb = xb.to(
                    TCN_DEVICE,
                    non_blocking=True
                )

                yb = yb.to(
                    TCN_DEVICE,
                    non_blocking=True
                )

                logits = model(xb)

                hold_loss_sum += float(
                    F.cross_entropy(
                        logits,
                        yb,
                        reduction="sum"
                    ).item()
                )

                hold_seen += len(yb)

                logits_parts.append(
                    logits.cpu().numpy()
                )

        assert hold_seen == len(data["y_hold"])

        hold_nll = (
            hold_loss_sum / hold_seen
        )

        if not np.isfinite(hold_nll):
            raise RuntimeError(
                "Nonfinite holdout joint NLL."
            )

        logits_all = np.concatenate(
            logits_parts, axis=0
        )

        metrics = evaluate_tcn_joint_logits(
            logits_all,
            data["horizons_hold"],
            data["onset_hold"],
            data["family_hold"]
        )

        row = {
            "epoch": epoch,
            "train_nll": float(train_nll),
            "hold_nll": float(hold_nll),
            **metrics
        }

        epoch_history.append(row)

        print(
            f"{config['id']} | "
            f"Fold {fold_id} | "
            f"Epoch {epoch:02d} | "
            f"Train NLL={train_nll:.5f} | "
            f"Hold NLL={hold_nll:.5f} | "
            f"H5 AP={metrics['H5_AP']:.5f} | "
            f"H10 AP={metrics['H10_AP']:.5f} | "
            f"H15 AP={metrics['H15_AP']:.5f} | "
            f"Family F1={metrics['Family_Macro_F1']:.4f} | "
            f"ETA MAE={metrics['ETA_MAE']:.3f}"
        )

        # Best fold checkpoint is selected ONLY by joint NLL.
        if hold_nll < best_nll - TCN_MIN_DELTA:

            best_nll = float(hold_nll)
            best_epoch = int(epoch)
            best_metrics = dict(metrics)

            bad_epochs = 0

        else:
            bad_epochs += 1

        if bad_epochs >= TCN_PATIENCE:

            print(
                f"  Early stopping at epoch {epoch}; "
                f"best epoch={best_epoch}, "
                f"best NLL={best_nll:.6f}"
            )

            break

    if best_epoch is None:
        raise RuntimeError(
            "No valid model-selection epoch."
        )

    result = {
        "configuration_id": config["id"],
        "configuration": config,
        "fold": fold_id,
        "seed": seed,
        "protocol_hash": TCN_PROTOCOL_HASH,
        "fit_windows": int(len(data["y_fit"])),
        "holdout_windows": int(len(data["y_hold"])),
        "best_epoch": best_epoch,
        "best_holdout_joint_nll": best_nll,
        "selected_metrics": best_metrics,
        "epochs_run": len(epoch_history),
        "history": epoch_history,
        "elapsed_seconds": float(
            time.perf_counter() - started
        ),
        "official_validation_used": False,
        "official_test_used": False
    }

    del model, optimizer, fit_loader, hold_loader

    gc.collect()
    torch.cuda.empty_cache()

    return result

# =============================================================================
# 5G — RUN OR RESUME ALL 40 EXPERIMENTS
# =============================================================================

TCN_ALL_FOLD_RESULTS = []

for config_index, config in enumerate(
    TCN_SEARCH_CONFIGS,
    start=1
):

    config_id = config["id"]

    config_dir = (
        TCN_CV_ROOT / config_id
    )

    config_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    print("\n" + "=" * 100)
    print(
        f"CONFIGURATION {config_index}/8: {config_id}"
    )
    print("=" * 100)
    print(config)

    for data in TCN_FOLD_DATA:

        fold_id = data["fold"]

        result_path = (
            config_dir
            / f"FOLD_{fold_id:02d}_RESULT.json"
        )

        if result_path.exists():

            result = tcn_load_json(result_path)

            assert (
                result["protocol_hash"]
                == TCN_PROTOCOL_HASH
            )

            assert (
                result["configuration_id"]
                == config_id
            )

            assert result["fold"] == fold_id

            print(
                f"✓ Resumed {config_id} fold {fold_id}"
            )

        else:

            result = fit_tcn_cv_fold(
                config,
                config_index,
                data
            )

            tcn_atomic_json(
                result_path,
                result
            )

            print(
                f"✓ Saved {config_id} fold {fold_id}"
            )

        TCN_ALL_FOLD_RESULTS.append(result)

assert len(TCN_ALL_FOLD_RESULTS) == 40

# =============================================================================
# 5H — AGGREGATE CV RESULTS
# =============================================================================

fold_rows = []

for result in TCN_ALL_FOLD_RESULTS:

    fold_rows.append({
        "Config_ID": result["configuration_id"],
        "Fold": result["fold"],
        "Best_Epoch": result["best_epoch"],
        "Holdout_Joint_NLL": (
            result["best_holdout_joint_nll"]
        ),
        **result["selected_metrics"]
    })

TCN_FOLD_RESULTS_DF = pd.DataFrame(
    fold_rows
)

summary_rows = []

for config in TCN_SEARCH_CONFIGS:

    subset = TCN_FOLD_RESULTS_DF.loc[
        TCN_FOLD_RESULTS_DF["Config_ID"]
        == config["id"]
    ].sort_values("Fold")

    assert len(subset) == 5

    summary_rows.append({
        "Config_ID": config["id"],
        "Mean_Joint_NLL": float(
            subset["Holdout_Joint_NLL"].mean()
        ),
        "SD_Joint_NLL": float(
            subset["Holdout_Joint_NLL"].std(ddof=1)
        ),
        "Mean_H5_AP": float(
            subset["H5_AP"].mean()
        ),
        "Mean_H10_AP": float(
            subset["H10_AP"].mean()
        ),
        "Mean_H15_AP": float(
            subset["H15_AP"].mean()
        ),
        "Mean_Family_Accuracy": float(
            subset["Family_Accuracy"].mean()
        ),
        "Mean_Family_Macro_F1": float(
            subset["Family_Macro_F1"].mean()
        ),
        "Mean_ETA_MAE": float(
            subset["ETA_MAE"].mean()
        ),
        "Final_Epochs": int(
            np.median(
                subset["Best_Epoch"].to_numpy()
            )
        )
    })

TCN_CV_SUMMARY_DF = pd.DataFrame(
    summary_rows
).sort_values(
    by=[
        "Mean_Joint_NLL",
        "Mean_Family_Macro_F1",
        "Config_ID"
    ],
    ascending=[True, False, True],
    kind="stable"
).reset_index(drop=True)

# =============================================================================
# 5I — LOCK WINNER
# =============================================================================

winner = TCN_CV_SUMMARY_DF.iloc[0]

winner_id = str(
    winner["Config_ID"]
)

winner_config = next(
    config
    for config in TCN_SEARCH_CONFIGS
    if config["id"] == winner_id
)

TCN_SELECTION = {
    "experiment": "TCN_V2_COMPETING_RISK",
    "dataset": "Synthetic_NAF_v2",
    "release_version": "2.0.0",
    "protocol_hash": TCN_PROTOCOL_HASH,
    "selection_rule": "MINIMUM_MEAN_HOLDOUT_JOINT_NLL",
    "checkpoint_rule": "MINIMUM_FOLD_HOLDOUT_JOINT_NLL",
    "selected_config_id": winner_id,
    "selected_model_config": winner_config,
    "selected_final_epochs": int(
        winner["Final_Epochs"]
    ),
    "selected_mean_joint_nll": float(
        winner["Mean_Joint_NLL"]
    ),
    "selected_mean_H5_AP": float(
        winner["Mean_H5_AP"]
    ),
    "selected_mean_H10_AP": float(
        winner["Mean_H10_AP"]
    ),
    "selected_mean_H15_AP": float(
        winner["Mean_H15_AP"]
    ),
    "selected_mean_family_macro_f1": float(
        winner["Mean_Family_Macro_F1"]
    ),
    "selected_mean_family_accuracy": float(
        winner["Mean_Family_Accuracy"]
    ),
    "selected_mean_eta_mae": float(
        winner["Mean_ETA_MAE"]
    ),
    "final_training_performed": False,
    "official_validation_used": False,
    "official_test_used": False
}

selection_path = (
    TCN_CV_ROOT
    / "SELECTED_TCN_CONFIGURATION.json"
)

if selection_path.exists():

    previous = tcn_load_json(
        selection_path
    )

    stable_fields = [
        "protocol_hash",
        "selected_config_id",
        "selected_model_config",
        "selected_final_epochs"
    ]

    for field in stable_fields:
        if previous[field] != TCN_SELECTION[field]:
            raise RuntimeError(
                "An existing locked TCN selection differs. "
                "Refusing overwrite."
            )

else:

    TCN_SELECTION["created_utc"] = (
        datetime.now(timezone.utc).isoformat()
    )

    tcn_atomic_json(
        selection_path,
        TCN_SELECTION
    )

# Save summary tables without silently replacing prior results.
summary_outputs = {
    "ALL_CONFIGURATIONS_CV_SUMMARY.csv": (
        TCN_CV_SUMMARY_DF
    ),
    "ALL_FOLD_RESULTS.csv": (
        TCN_FOLD_RESULTS_DF
    )
}

for filename, df in summary_outputs.items():

    output_path = TCN_CV_ROOT / filename

    csv_data = df.to_csv(
        index=False,
        float_format="%.12g"
    )

    if output_path.exists():
        if output_path.read_text(
            encoding="utf-8"
        ) != csv_data:
            raise RuntimeError(
                f"Existing report differs: {output_path}"
            )

    else:
        output_path.write_text(
            csv_data,
            encoding="utf-8"
        )

# =============================================================================
# 5J — FINAL REPORT
# =============================================================================

print("\n" + "=" * 110)
print("TCN V2 COMPETING-RISK — FIVE-FOLD MODEL SELECTION COMPLETE")
print("=" * 110)

print("\nALL CONFIGURATION RESULTS")
print(
    TCN_CV_SUMMARY_DF.round(6).to_string(
        index=False
    )
)

print("\nSELECTED CONFIGURATION")
print(
    json.dumps(
        TCN_SELECTION,
        indent=2
    )
)

print("\nCompleted fold runs:", len(TCN_ALL_FOLD_RESULTS))
print("Saved in:", TCN_CV_ROOT)

print("\nFinal full-TRAIN TCN: NOT TRAINED")
print("Official VALIDATION: NOT ACCESSED")
print("Official TEST: NOT ACCESSED")
print("Frozen dataset: UNCHANGED")

print("\nNEXT: Review CV selection before final full-TRAIN fitting.")
print("=" * 110)


In [ ]:

# =============================================================================
# TCN V2 COMPETING-RISK — CELL 6
# FINAL FULL-TRAIN FITTING OF LOCKED C02 CONFIGURATION
#
# Fresh model, 21 epochs, all 88,186 TRAIN windows.
# No validation/test access, early stopping, or hyperparameter tuning.
# Preserves existing model artifacts and frozen dataset.
# =============================================================================

import os
import gc
import json
import time
import random
import hashlib
import platform
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from torch.utils.data import TensorDataset, DataLoader

# =============================================================================
# 6A — PATHS
# =============================================================================

ROOT_6 = Path("/content/drive/MyDrive/network-attack-forecasting")
DATA_DIR_6 = ROOT_6 / "Synthetic_NAF_v2/release_candidate_corrected"
FREEZE_DIR_6 = ROOT_6 / "Synthetic_NAF_v2/FINAL_RELEASE_FREEZE"

TRAIN_FILE_6 = DATA_DIR_6 / "windows_W10_TRAIN.npz"
MANIFEST_FILE_6 = (
    FREEZE_DIR_6 / "Synthetic_NAF_v2_RELEASE_MANIFEST.json"
)
FREEZE_LOCK_FILE_6 = (
    FREEZE_DIR_6 / "Synthetic_NAF_v2_FREEZE_LOCK.json"
)

CV_ROOT_6 = (
    ROOT_6
    / "Synthetic_NAF_v2/models/TCN_V2_CompetingRisk"
    / "train_cv_selection_GPU"
)

PROTOCOL_FILE_6 = CV_ROOT_6 / "SEARCH_PROTOCOL.json"
SELECTION_FILE_6 = CV_ROOT_6 / "SELECTED_TCN_CONFIGURATION.json"

FINAL_DIR_6 = (
    ROOT_6
    / "Synthetic_NAF_v2/models/TCN_V2_CompetingRisk"
    / "final_train_model"
)

STAGING_DIR_6 = FINAL_DIR_6.with_name("final_train_model_STAGING")

# Refuse to overwrite completed or partially saved artifacts.
if FINAL_DIR_6.exists() or STAGING_DIR_6.exists():
    raise RuntimeError(
        "Existing final or staging directory detected.\n"
        "Refusing to overwrite. Inspect existing artifacts first.\n"
        f"Final: {FINAL_DIR_6}\n"
        f"Staging: {STAGING_DIR_6}"
    )

# =============================================================================
# 6B — HELPERS
# =============================================================================

def read_json_6(path):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(path)
    return json.loads(path.read_text(encoding="utf-8"))


def sha256_file_6(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def canonical_hash_6(obj):
    payload = json.dumps(
        obj,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False
    ).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()


def save_json_6(path, obj):
    Path(path).write_text(
        json.dumps(obj, indent=2, sort_keys=True, allow_nan=False),
        encoding="utf-8"
    )


def seed_everything_6(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


# =============================================================================
# 6C — VERIFY PREPARATION, DATASET AND SAVED CV SELECTION
# =============================================================================

required_6 = [
    "X_RAW", "JOINT_TARGET", "Y_HORIZONS",
    "ONSET", "FAMILY", "EPISODE_ID", "CV_GROUPS",
    "CV_FOLDS", "FEATURES", "FAMILIES",
    "TemporalRobustScaler", "TCNCompetingRisk"
]

missing_6 = [name for name in required_6 if name not in globals()]
if missing_6:
    raise RuntimeError(
        f"Rerun TCN Cells 1–3. Missing: {missing_6}"
    )

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA unavailable. Select T4 GPU and rerun setup cells."
    )

manifest_6 = read_json_6(MANIFEST_FILE_6)
freeze_lock_6 = read_json_6(FREEZE_LOCK_FILE_6)
protocol_6 = read_json_6(PROTOCOL_FILE_6)
selection_6 = read_json_6(SELECTION_FILE_6)

assert manifest_6["release_version"] == "2.0.0"

assert canonical_hash_6(manifest_6) == (
    freeze_lock_6["manifest_sha256_canonical"]
)

expected_train_hash_6 = (
    manifest_6["data_integrity"]["file_sha256"]
    ["windows_W10_TRAIN.npz"]
)

assert sha256_file_6(TRAIN_FILE_6) == expected_train_hash_6
assert protocol_6["train_sha256"] == expected_train_hash_6

assert canonical_hash_6(protocol_6) == selection_6["protocol_hash"]
assert selection_6["selected_config_id"] == "C02"
assert selection_6["selected_final_epochs"] == 21

assert selection_6["final_training_performed"] is False
assert selection_6["official_validation_used"] is False
assert selection_6["official_test_used"] is False

assert selection_6["selection_rule"] == (
    "MINIMUM_MEAN_HOLDOUT_JOINT_NLL"
)
assert protocol_6["loss"] == (
    "UNWEIGHTED_76_CLASS_CROSS_ENTROPY"
)
assert protocol_6["final_epoch_rule"] == (
    "MEDIAN_SELECTED_FOLD_EPOCH"
)

EXPECTED_C02_6 = {
    "id": "C02",
    "channels": 32,
    "num_blocks": 3,
    "dropout": 0.2,
    "learning_rate": 0.001
}

assert selection_6["selected_model_config"] == EXPECTED_C02_6
assert len(protocol_6["candidate_configurations"]) == 8
assert len(protocol_6["folds"]) == 5

assert X_RAW.shape == (88186, 10, 14)
assert JOINT_TARGET.shape == (88186,)
assert Y_HORIZONS.shape == (88186, 3)

assert np.isfinite(X_RAW).all()
assert np.all(X_RAW >= 0)
assert set(np.unique(JOINT_TARGET)) == set(range(76))
assert len(np.unique(EPISODE_ID)) == 1400
assert len(np.unique(CV_GROUPS)) == 35
assert len(FEATURES) == 14
assert len(FAMILIES) == 5

expected_joint_6 = np.where(
    ONSET == 0,
    0,
    5 * (ONSET - 1) + FAMILY
).astype(np.int64)

assert np.array_equal(expected_joint_6, JOINT_TARGET)

for col, horizon in enumerate([5, 10, 15]):
    expected_h_6 = (
        (ONSET >= 1) & (ONSET <= horizon)
    ).astype(np.int8)
    assert np.array_equal(expected_h_6, Y_HORIZONS[:, col])

# Independently check that notebook arrays match frozen TRAIN.
with np.load(TRAIN_FILE_6, allow_pickle=False) as frozen_6:
    assert np.array_equal(X_RAW, frozen_6["X"])
    assert np.array_equal(ONSET, frozen_6["onset_class"])
    assert np.array_equal(FAMILY, frozen_6["family_class"])

# Verify fold definitions and all 40 saved CV records.
all_fold_records_6 = []
fold_medians_6 = {}

for fold_number_6, (fit_idx_6, hold_idx_6) in enumerate(
    CV_FOLDS, start=1
):
    assert set(CV_GROUPS[fit_idx_6]).isdisjoint(
        set(CV_GROUPS[hold_idx_6])
    )
    assert set(EPISODE_ID[fit_idx_6]).isdisjoint(
        set(EPISODE_ID[hold_idx_6])
    )

    saved_fold_6 = protocol_6["folds"][fold_number_6 - 1]

    assert saved_fold_6["fold"] == fold_number_6
    assert saved_fold_6["fit_windows"] == len(fit_idx_6)
    assert saved_fold_6["holdout_windows"] == len(hold_idx_6)

    index_hash_6 = hashlib.sha256(
        np.asarray(hold_idx_6, dtype="<i8").tobytes()
    ).hexdigest()

    assert index_hash_6 == saved_fold_6["holdout_indices_sha256"]

for config_6 in protocol_6["candidate_configurations"]:
    for fold_number_6 in range(1, 6):
        path_6 = (
            CV_ROOT_6
            / config_6["id"]
            / f"FOLD_{fold_number_6:02d}_RESULT.json"
        )
        result_6 = read_json_6(path_6)

        assert result_6["protocol_hash"] == (
            selection_6["protocol_hash"]
        )
        assert result_6["configuration_id"] == config_6["id"]
        assert result_6["configuration"] == config_6
        assert result_6["fold"] == fold_number_6
        assert result_6["official_validation_used"] is False
        assert result_6["official_test_used"] is False

        assert np.isfinite(result_6["best_holdout_joint_nll"])
        assert 1 <= result_6["best_epoch"] <= 35
        all_fold_records_6.append(result_6)

assert len(all_fold_records_6) == 40

ranking_6 = []
for config_6 in protocol_6["candidate_configurations"]:
    subset_6 = [
        r for r in all_fold_records_6
        if r["configuration_id"] == config_6["id"]
    ]
    ranking_6.append({
        "id": config_6["id"],
        "mean_joint_nll": float(np.mean([
            r["best_holdout_joint_nll"] for r in subset_6
        ])),
        "mean_family_f1": float(np.mean([
            r["selected_metrics"]["Family_Macro_F1"]
            for r in subset_6
        ])),
        "median_epoch": int(np.median([
            r["best_epoch"] for r in subset_6
        ]))
    })

ranking_6.sort(
    key=lambda item: (
        item["mean_joint_nll"],
        -item["mean_family_f1"],
        item["id"]
    )
)

winner_6 = ranking_6[0]

assert winner_6["id"] == "C02"
assert winner_6["median_epoch"] == 21
assert np.isclose(
    winner_6["mean_joint_nll"],
    selection_6["selected_mean_joint_nll"],
    rtol=0,
    atol=1e-12
)

print("✓ Frozen release integrity verified")
print("✓ Full-TRAIN arrays verified")
print("✓ All 40 CV records verified")
print("✓ C02 selection independently confirmed")
print("✓ Locked 21-epoch budget verified")

# =============================================================================
# 6D — LOCK FINAL TRAINING PARAMETERS
# =============================================================================

FINAL_SEED_6 = 20261024
FINAL_EPOCHS_6 = 21
FINAL_BATCH_SIZE_6 = 256
FINAL_LR_6 = 0.001
FINAL_WEIGHT_DECAY_6 = 0.001
FINAL_GRAD_CLIP_6 = 1.0

DEVICE_6 = torch.device("cuda")
seed_everything_6(FINAL_SEED_6)

FINAL_TCN_CONFIG_6 = {
    "input_dim": 14,
    "channels": 32,
    "num_blocks": 3,
    "kernel_size": 3,
    "dropout": 0.2,
    "dense_dim": 32,
    "num_classes": 76
}

print("\n" + "=" * 100)
print("TCN V2 COMPETING-RISK — FINAL FULL-TRAIN FITTING")
print("=" * 100)

print("Device:", DEVICE_6)
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)
print("Selected configuration: C02")
print("Final epochs:", FINAL_EPOCHS_6)
print("Batch size:", FINAL_BATCH_SIZE_6)
print("Learning rate:", FINAL_LR_6)
print("Weight decay:", FINAL_WEIGHT_DECAY_6)
print("Training windows:", len(X_RAW))
print("TRAIN episodes:", len(np.unique(EPISODE_ID)))
print("Official VALIDATION: NOT LOADED")
print("Official TEST: NOT LOADED")

# =============================================================================
# 6E — FIT SCALER ON ALL TRAIN, NEVER ON VAL/TEST
# =============================================================================

FINAL_TCN_SCALER = TemporalRobustScaler(
    clip_value=8.0
).fit(X_RAW)

X_FINAL_6 = FINAL_TCN_SCALER.transform(X_RAW)
Y_FINAL_6 = np.ascontiguousarray(
    JOINT_TARGET,
    dtype=np.int64
)

assert X_FINAL_6.shape == (88186, 10, 14)
assert X_FINAL_6.dtype == np.float32
assert np.isfinite(X_FINAL_6).all()

SCALER_STATE_6 = FINAL_TCN_SCALER.state_dict()

assert SCALER_STATE_6["feature_names"] == list(FEATURES)
assert len(SCALER_STATE_6["median"]) == 14
assert len(SCALER_STATE_6["scale"]) == 14
assert SCALER_STATE_6["clip_value"] == 8.0

print("\n✓ Full-TRAIN scaler fitted")

# =============================================================================
# 6F — NEW MODEL AND TRAIN DATALOADER
# =============================================================================

train_generator_6 = torch.Generator()
train_generator_6.manual_seed(FINAL_SEED_6)

train_loader_6 = DataLoader(
    TensorDataset(
        torch.from_numpy(X_FINAL_6),
        torch.from_numpy(Y_FINAL_6)
    ),
    batch_size=FINAL_BATCH_SIZE_6,
    shuffle=True,
    generator=train_generator_6,
    num_workers=0,
    pin_memory=True
)

FINAL_TCN_MODEL = TCNCompetingRisk(
    **FINAL_TCN_CONFIG_6
).to(DEVICE_6)

FINAL_TCN_OPTIMIZER = torch.optim.AdamW(
    FINAL_TCN_MODEL.parameters(),
    lr=FINAL_LR_6,
    weight_decay=FINAL_WEIGHT_DECAY_6
)

PARAM_COUNT_6 = sum(
    p.numel() for p in FINAL_TCN_MODEL.parameters()
)

# Expected from the recorded architecture:
# C01 (64 channels) has 70,060 parameters.
# C02 has 32 channels, so verify this is a smaller architecture.
assert PARAM_COUNT_6 < 70060

print("Trainable parameters:", PARAM_COUNT_6)
print("\nModel architecture:")
print(FINAL_TCN_MODEL)

# =============================================================================
# 6G — TRAIN EXACTLY 21 EPOCHS
#
# No early stopping, holdout, validation, or metric-based checkpoint selection.
# =============================================================================

FINAL_TCN_HISTORY = []
training_start_6 = time.perf_counter()

for epoch_6 in range(1, FINAL_EPOCHS_6 + 1):

    FINAL_TCN_MODEL.train()

    total_loss_6 = 0.0
    seen_6 = 0

    for xb_6, yb_6 in train_loader_6:

        xb_6 = xb_6.to(DEVICE_6, non_blocking=True)
        yb_6 = yb_6.to(DEVICE_6, non_blocking=True)

        FINAL_TCN_OPTIMIZER.zero_grad(set_to_none=True)

        logits_6 = FINAL_TCN_MODEL(xb_6)
        loss_6 = F.cross_entropy(logits_6, yb_6)

        if not torch.isfinite(loss_6):
            raise RuntimeError(
                f"Nonfinite training loss at epoch {epoch_6}"
            )

        loss_6.backward()

        torch.nn.utils.clip_grad_norm_(
            FINAL_TCN_MODEL.parameters(),
            max_norm=FINAL_GRAD_CLIP_6,
            error_if_nonfinite=True
        )

        FINAL_TCN_OPTIMIZER.step()

        count_6 = int(len(yb_6))
        total_loss_6 += float(loss_6.item()) * count_6
        seen_6 += count_6

    if seen_6 != 88186:
        raise RuntimeError("An epoch did not cover all TRAIN windows.")

    train_nll_6 = total_loss_6 / seen_6

    if not np.isfinite(train_nll_6):
        raise RuntimeError("Nonfinite epoch NLL.")

    elapsed_6 = time.perf_counter() - training_start_6

    FINAL_TCN_HISTORY.append({
        "epoch": int(epoch_6),
        "train_joint_nll": float(train_nll_6),
        "windows_seen": int(seen_6),
        "elapsed_seconds": float(elapsed_6)
    })

    print(
        f"Epoch {epoch_6:02d}/{FINAL_EPOCHS_6} | "
        f"TRAIN joint NLL={train_nll_6:.6f} | "
        f"Windows={seen_6:,}"
    )

assert len(FINAL_TCN_HISTORY) == 21

FINAL_TCN_HISTORY_DF = pd.DataFrame(FINAL_TCN_HISTORY)

print("\n✓ All 21 epochs completed")

# =============================================================================
# 6H — TRAIN-ONLY PROBABILITY SANITY CHECK
# =============================================================================

FINAL_TCN_MODEL.eval()

with torch.inference_mode():

    preview_input_6 = torch.from_numpy(
        X_FINAL_6[:256]
    ).to(DEVICE_6)

    preview_logits_6 = FINAL_TCN_MODEL(preview_input_6)

    preview_probs_6 = torch.softmax(
        preview_logits_6,
        dim=1
    )

    assert preview_probs_6.shape == (256, 76)
    assert torch.isfinite(preview_probs_6).all()

    assert torch.allclose(
        preview_probs_6.sum(dim=1),
        torch.ones(256, device=DEVICE_6),
        atol=1e-6
    )

    no_event_6 = preview_probs_6[:, 0]

    event_matrix_6 = preview_probs_6[:, 1:].reshape(
        -1, 15, 5
    )

    onset_probs_6 = event_matrix_6.sum(dim=2)
    family_probs_6 = event_matrix_6.sum(dim=1)

    horizons_6 = (
        onset_probs_6.cumsum(dim=1)[:, [4, 9, 14]]
    )

    assert torch.all(
        horizons_6[:, 0] <= horizons_6[:, 1] + 1e-7
    )
    assert torch.all(
        horizons_6[:, 1] <= horizons_6[:, 2] + 1e-7
    )

    assert torch.allclose(
        no_event_6 + horizons_6[:, 2],
        torch.ones(256, device=DEVICE_6),
        atol=1e-6
    )

    assert torch.allclose(
        family_probs_6.sum(dim=1),
        horizons_6[:, 2],
        atol=1e-6
    )

    assert torch.allclose(
        onset_probs_6.sum(dim=1),
        horizons_6[:, 2],
        atol=1e-6
    )

print("✓ 76-class probabilities valid")
print("✓ H5 ≤ H10 ≤ H15")
print("✓ Family and onset marginals valid")
print("✓ No-event complement valid")

# =============================================================================
# 6I — FINAL RECORD AND ARTIFACTS
# =============================================================================

FINAL_TCN_RECORD_6 = {
    "model_name": "TCN_V2_COMPETING_RISK",
    "dataset": "Synthetic_NAF_v2",
    "dataset_version": "2.0.0",
    "training_split": "TRAIN",
    "training_status": "COMPLETED",
    "training_windows": 88186,
    "training_episodes": 1400,
    "train_data_sha256": expected_train_hash_6,
    "dataset_manifest_sha256": canonical_hash_6(manifest_6),
    "input_shape": [10, 14],
    "output_classes": 76,
    "output_definition": {
        "class_0": "NO_ONSET_WITHIN_15_MINUTES",
        "class_1_to_75": (
            "CLASS = 5 * (ONSET_MINUTE - 1) + FAMILY_ID"
        ),
        "onset_minutes": list(range(1, 16)),
        "family_names": list(FAMILIES)
    },
    "feature_names": list(FEATURES),
    "model_architecture": FINAL_TCN_CONFIG_6,
    "trainable_parameters": int(PARAM_COUNT_6),
    "selected_configuration_id": "C02",
    "selected_configuration": EXPECTED_C02_6,
    "selection_protocol_hash": selection_6["protocol_hash"],
    "selected_cv_mean_joint_nll": float(
        selection_6["selected_mean_joint_nll"]
    ),
    "selected_cv_mean_H5_AP": float(
        selection_6["selected_mean_H5_AP"]
    ),
    "selected_cv_mean_H10_AP": float(
        selection_6["selected_mean_H10_AP"]
    ),
    "selected_cv_mean_H15_AP": float(
        selection_6["selected_mean_H15_AP"]
    ),
    "selected_cv_mean_family_macro_f1": float(
        selection_6["selected_mean_family_macro_f1"]
    ),
    "selected_cv_mean_eta_mae": float(
        selection_6["selected_mean_eta_mae"]
    ),
    "training_epochs": FINAL_EPOCHS_6,
    "training_seed": FINAL_SEED_6,
    "batch_size": FINAL_BATCH_SIZE_6,
    "optimizer": "AdamW",
    "learning_rate": FINAL_LR_6,
    "weight_decay": FINAL_WEIGHT_DECAY_6,
    "gradient_clip_norm": FINAL_GRAD_CLIP_6,
    "loss_function": "UNWEIGHTED_76_CLASS_CROSS_ENTROPY",
    "preprocessing": "FOLD_LOCAL_LOG1P_MEDIAN_IQR_CLIP8",
    "scaler_fit_split": "FULL_TRAIN_ONLY",
    "torch_version": torch.__version__,
    "python_version": platform.python_version(),
    "device": str(DEVICE_6),
    "official_validation_used": False,
    "official_test_used": False,
    "dataset_modified": False,
    "trained_utc": datetime.now(timezone.utc).isoformat()
}

# Keep staging artifacts separate until reload verification succeeds.
STAGING_DIR_6.mkdir(parents=True, exist_ok=False)

MODEL_PATH_6 = STAGING_DIR_6 / "TCN_V2_CR_STATE.pt"
SCALER_PATH_6 = STAGING_DIR_6 / "TCN_V2_CR_FULL_TRAIN_SCALER.json"
HISTORY_PATH_6 = STAGING_DIR_6 / "TCN_V2_CR_FINAL_TRAIN_HISTORY.csv"
RECORD_PATH_6 = STAGING_DIR_6 / "TCN_V2_CR_FINAL_TRAIN_RECORD.json"
REGISTRY_PATH_6 = STAGING_DIR_6 / "TCN_V2_CR_MODEL_REGISTRY.json"

# Save only CPU tensors: compatible with torch.load(weights_only=True).
cpu_weights_6 = {
    key: value.detach().cpu().clone()
    for key, value in FINAL_TCN_MODEL.state_dict().items()
}

torch.save(cpu_weights_6, MODEL_PATH_6)

save_json_6(SCALER_PATH_6, SCALER_STATE_6)

FINAL_TCN_HISTORY_DF.to_csv(
    HISTORY_PATH_6,
    index=False,
    float_format="%.17g"
)

FINAL_TCN_RECORD_6["model_sha256"] = sha256_file_6(
    MODEL_PATH_6
)
FINAL_TCN_RECORD_6["scaler_sha256"] = sha256_file_6(
    SCALER_PATH_6
)
FINAL_TCN_RECORD_6["history_sha256"] = sha256_file_6(
    HISTORY_PATH_6
)

save_json_6(RECORD_PATH_6, FINAL_TCN_RECORD_6)

TCN_REGISTRY_6 = {
    "model_name": "TCN_V2_COMPETING_RISK",
    "dataset": "Synthetic_NAF_v2",
    "dataset_version": "2.0.0",
    "training_status": "FINAL_FULL_TRAIN_COMPLETED",
    "validation_status": "NOT_STARTED",
    "test_status": "EMBARGOED",
    "selected_configuration": "C02",
    "training_epochs": 21,
    "model_file": MODEL_PATH_6.name,
    "model_sha256": sha256_file_6(MODEL_PATH_6),
    "scaler_file": SCALER_PATH_6.name,
    "scaler_sha256": sha256_file_6(SCALER_PATH_6),
    "history_file": HISTORY_PATH_6.name,
    "history_sha256": sha256_file_6(HISTORY_PATH_6),
    "training_record_file": RECORD_PATH_6.name,
    "training_record_sha256": sha256_file_6(RECORD_PATH_6),
    "selection_protocol_hash": selection_6["protocol_hash"],
    "feature_names": list(FEATURES),
    "output_classes": 76,
    "forecast_horizons": [5, 10, 15]
}

save_json_6(REGISTRY_PATH_6, TCN_REGISTRY_6)

# =============================================================================
# 6J — INDEPENDENT CHECKPOINT RELOAD
# =============================================================================

saved_weights_6 = torch.load(
    MODEL_PATH_6,
    map_location="cpu",
    weights_only=True
)

RELOADED_TCN_6 = TCNCompetingRisk(
    **FINAL_TCN_CONFIG_6
).cpu()

RELOADED_TCN_6.load_state_dict(
    saved_weights_6,
    strict=True
)

RELOADED_TCN_6.eval()

with torch.inference_mode():

    sample_cpu_6 = torch.from_numpy(X_FINAL_6[:32])

    original_p_6 = torch.softmax(
        FINAL_TCN_MODEL(sample_cpu_6.to(DEVICE_6)).cpu(),
        dim=1
    )

    reloaded_p_6 = torch.softmax(
        RELOADED_TCN_6(sample_cpu_6),
        dim=1
    )

    assert torch.allclose(
        original_p_6,
        reloaded_p_6,
        rtol=1e-5,
        atol=1e-5
    ), "Checkpoint reload predictions do not match."

assert sha256_file_6(MODEL_PATH_6) == (
    TCN_REGISTRY_6["model_sha256"]
)
assert sha256_file_6(SCALER_PATH_6) == (
    TCN_REGISTRY_6["scaler_sha256"]
)
assert sha256_file_6(RECORD_PATH_6) == (
    TCN_REGISTRY_6["training_record_sha256"]
)

# Validate scaler serialization without refitting.
saved_scaler_state_6 = read_json_6(SCALER_PATH_6)

assert saved_scaler_state_6 == SCALER_STATE_6

print("\n✓ Saved checkpoint reloaded successfully")
print("✓ Original and reloaded predictions match")
print("✓ Scaler serialization verified")
print("✓ Model/scaler/record SHA256 verified")

# =============================================================================
# 6K — ATOMICALLY FINALIZE DIRECTORY
# =============================================================================

if FINAL_DIR_6.exists():
    raise RuntimeError(
        "Final directory appeared during training. "
        "Staging files preserved; refusing overwrite."
    )

STAGING_DIR_6.rename(FINAL_DIR_6)

expected_files_6 = [
    "TCN_V2_CR_STATE.pt",
    "TCN_V2_CR_FULL_TRAIN_SCALER.json",
    "TCN_V2_CR_FINAL_TRAIN_HISTORY.csv",
    "TCN_V2_CR_FINAL_TRAIN_RECORD.json",
    "TCN_V2_CR_MODEL_REGISTRY.json"
]

for filename_6 in expected_files_6:
    assert (FINAL_DIR_6 / filename_6).is_file()

# Check frozen dataset again after training.
assert sha256_file_6(TRAIN_FILE_6) == expected_train_hash_6

# =============================================================================
# 6L — FINAL REPORT
# =============================================================================

print("\n" + "=" * 105)
print("TCN V2 COMPETING-RISK — FINAL FULL-TRAIN FITTING COMPLETE")
print("=" * 105)

print("\nSelected configuration: C02")
print("Training windows:", len(X_RAW))
print("Training episodes:", len(np.unique(EPISODE_ID)))
print("Trainable parameters:", PARAM_COUNT_6)
print("Output classes:", 76)
print("Epochs completed:", len(FINAL_TCN_HISTORY_DF))

print(
    "\nInitial TRAIN joint NLL:",
    f"{FINAL_TCN_HISTORY_DF.iloc[0]['train_joint_nll']:.6f}"
)

print(
    "Final TRAIN joint NLL:",
    f"{FINAL_TCN_HISTORY_DF.iloc[-1]['train_joint_nll']:.6f}"
)

print("\nModel saved:")
print(FINAL_DIR_6 / "TCN_V2_CR_STATE.pt")

print("\nScaler saved:")
print(FINAL_DIR_6 / "TCN_V2_CR_FULL_TRAIN_SCALER.json")

print("\nFinal model SHA256:")
print(TCN_REGISTRY_6["model_sha256"])

print("\nProbability checks: PASSED")
print("Checkpoint reload: PASSED")
print("Additional hyperparameter selection: NONE")
print("Official VALIDATION: NOT ACCESSED")
print("Official TEST: NOT ACCESSED")
print("Frozen dataset: UNCHANGED")

print("\nNEXT: CELL 7 — OFFICIAL TCN VALIDATION")
print("=" * 105)


In [ ]:

# =============================================================================
# TCN V2 COMPETING-RISK — CELL 7 (COMPLETE CORRECTED VERSION)
# OFFICIAL VALIDATION + MATCHED LSTM COMPARISON
#
# Uses:
#   - Frozen official VALIDATION (18,839 windows / 300 episodes)
#   - Saved final TCN C02 checkpoint
#   - Saved full-TRAIN scaler
#   - Verified pickle-free LSTM predictions archive
#
# Evaluates:
#   1. 76-class joint NLL and accuracy
#   2. H5/H10/H15 AP, ROC-AUC, Brier, ECE
#   3. Per-family precision/recall/F1 and confusion matrix
#   4. Conditional onset-time error
#   5. Fixed-threshold detection and end-to-end family forecasts
#   6. Hard/ordinary benign episode false alerts
#   7. Source-specific metrics
#   8. Episode-cluster bootstrap CIs
#   9. Matched paired-bootstrap TCN vs LSTM comparison
#
# NEVER loads official TEST.
# NEVER trains or changes model weights.
# NEVER refits the scaler.
# NEVER overwrites an existing completed validation report.
# =============================================================================

import os
import json
import hashlib
import platform
import time
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch

from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    accuracy_score,
    f1_score,
    confusion_matrix,
    precision_recall_fscore_support
)

# =============================================================================
# 7A — CONFIGURATION AND PATHS
# =============================================================================

ROOT_7 = Path("/content/drive/MyDrive/network-attack-forecasting")

BASE_7 = ROOT_7 / "Synthetic_NAF_v2"

DATA_DIR_7 = BASE_7 / "release_candidate_corrected"

FREEZE_DIR_7 = BASE_7 / "FINAL_RELEASE_FREEZE"

MODEL_DIR_7 = (
    BASE_7 / "models/TCN_V2_CompetingRisk/final_train_model"
)

CV_DIR_7 = (
    BASE_7 / "models/TCN_V2_CompetingRisk/train_cv_selection_GPU"
)

LSTM_VAL_DIR_7 = (
    BASE_7 / "models/LSTM_V2_CompetingRisk/official_validation"
)

OUTPUT_DIR_7 = (
    BASE_7 / "models/TCN_V2_CompetingRisk/official_validation"
)

STAGING_DIR_7 = OUTPUT_DIR_7.with_name(
    "official_validation_STAGING"
)

VAL_PATH_7 = DATA_DIR_7 / "windows_W10_VALIDATION.npz"

TRAIN_PATH_7 = DATA_DIR_7 / "windows_W10_TRAIN.npz"

META_PATH_7 = (
    DATA_DIR_7 / "episode_metadata_DO_NOT_USE_AS_PREDICTORS.csv"
)

MANIFEST_PATH_7 = (
    FREEZE_DIR_7 / "Synthetic_NAF_v2_RELEASE_MANIFEST.json"
)

LOCK_PATH_7 = (
    FREEZE_DIR_7 / "Synthetic_NAF_v2_FREEZE_LOCK.json"
)

MODEL_PATH_7 = MODEL_DIR_7 / "TCN_V2_CR_STATE.pt"

SCALER_PATH_7 = (
    MODEL_DIR_7 / "TCN_V2_CR_FULL_TRAIN_SCALER.json"
)

RECORD_PATH_7 = (
    MODEL_DIR_7 / "TCN_V2_CR_FINAL_TRAIN_RECORD.json"
)

REGISTRY_PATH_7 = (
    MODEL_DIR_7 / "TCN_V2_CR_MODEL_REGISTRY.json"
)

SELECTION_PATH_7 = (
    CV_DIR_7 / "SELECTED_TCN_CONFIGURATION.json"
)

PROTOCOL_PATH_7 = CV_DIR_7 / "SEARCH_PROTOCOL.json"

# IMPORTANT: use the already-created safe LSTM archive.
LSTM_SAFE_PATH_7 = (
    LSTM_VAL_DIR_7
    / "JOINT_76_CLASS_VALIDATION_PREDICTIONS_SAFE.npz"
)

LSTM_ORIGINAL_PATH_7 = (
    LSTM_VAL_DIR_7
    / "JOINT_76_CLASS_VALIDATION_PREDICTIONS.npz"
)

EXPECTED_TCN_SHA_7 = (
    "20b26d861dcab10357f7165877e0c5eb4121d511f4b6fca0d67f6eaef12c36dd"
)

EXPECTED_ORIGINAL_LSTM_ARCHIVE_SHA_7 = (
    "183a9c90efdd552099cfd61b71b3d3094c4083f02418622dc0eaf0db166552a0"
)

FAMILY_NAMES_7 = [
    "BRUTE_AUTH",
    "DOS_DDOS",
    "EXPLOIT_APP",
    "FUZZING",
    "RECON_SCAN"
]

HORIZONS_7 = [5, 10, 15]

FIXED_THRESHOLDS_7 = [0.10, 0.20, 0.50]

BOOTSTRAP_REPLICATES_7 = 300
BOOTSTRAP_SEED_7 = 20261024

# Safety: never overwrite prior completed evaluation.
if OUTPUT_DIR_7.exists():
    raise RuntimeError(
        "An official TCN validation directory already exists.\n"
        "Refusing to overwrite it:\n"
        f"{OUTPUT_DIR_7}"
    )

if STAGING_DIR_7.exists():
    raise RuntimeError(
        "A previous validation staging directory exists.\n"
        "Inspect it before rerunning:\n"
        f"{STAGING_DIR_7}"
    )

if "TCNCompetingRisk" not in globals():
    raise RuntimeError(
        "TCNCompetingRisk is undefined. "
        "Run the original TCN Cell 3 architecture definition first."
    )

# =============================================================================
# 7B — HELPERS
# =============================================================================

def read_json_7(path):
    if not path.is_file():
        raise FileNotFoundError(path)
    return json.loads(path.read_text(encoding="utf-8"))


def sha256_file_7(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def canonical_sha_7(obj):
    payload = json.dumps(
        obj,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False
    ).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()


def save_json_7(path, obj):
    Path(path).write_text(
        json.dumps(
            obj,
            indent=2,
            sort_keys=True,
            allow_nan=False
        ),
        encoding="utf-8"
    )


def parse_boolean_7(values):
    s = (
        pd.Series(values)
        .astype("string")
        .str.strip()
        .str.lower()
    )

    valid = {"true", "false", "1", "0", "yes", "no"}

    if not s.isin(valid).all():
        raise ValueError(
            "Unexpected Hard_Benign values in episode metadata."
        )

    return s.isin(
        {"true", "1", "yes"}
    ).to_numpy(dtype=bool)


def ece_7(truth, probabilities, bins=10):
    truth = np.asarray(truth, dtype=float)
    probabilities = np.asarray(probabilities, dtype=float)

    indices = np.minimum(
        (probabilities * bins).astype(int),
        bins - 1
    )

    rows = []
    score = 0.0

    for b in range(bins):
        mask = indices == b
        count = int(mask.sum())

        confidence = (
            float(probabilities[mask].mean())
            if count else None
        )

        observed = (
            float(truth[mask].mean())
            if count else None
        )

        if count:
            score += (
                count / len(truth)
                * abs(confidence - observed)
            )

        rows.append({
            "Bin": b,
            "Lower": b / bins,
            "Upper": (b + 1) / bins,
            "Count": count,
            "Mean_Prediction": confidence,
            "Observed_Rate": observed
        })

    return float(score), rows


def horizon_metrics_7(y, p):
    y = np.asarray(y, dtype=np.int8)
    p = np.asarray(p, dtype=np.float64)

    if len(np.unique(y)) != 2:
        raise ValueError(
            "Horizon evaluation subset must contain both classes."
        )

    calibration_error, calibration_rows = ece_7(y, p)

    metrics = {
        "Windows": int(len(y)),
        "Positive_Windows": int(y.sum()),
        "Prevalence": float(y.mean()),
        "AP": float(average_precision_score(y, p)),
        "ROC_AUC": float(roc_auc_score(y, p)),
        "Brier": float(brier_score_loss(y, p)),
        "ECE_10": calibration_error
    }

    return metrics, calibration_rows


def derive_predictions_7(joint):
    joint = np.asarray(joint, dtype=np.float64)

    if joint.ndim != 2 or joint.shape[1] != 76:
        raise ValueError("Expected a matrix with 76 joint classes.")

    if not np.isfinite(joint).all():
        raise ValueError("Nonfinite joint probabilities.")

    if (joint < -1e-12).any():
        raise ValueError("Negative joint probabilities.")

    if not np.allclose(
        joint.sum(axis=1), 1.0, atol=1e-6
    ):
        raise ValueError("Joint probabilities do not sum to one.")

    matrix = joint[:, 1:].reshape(-1, 15, 5)

    onset = matrix.sum(axis=2)
    family = matrix.sum(axis=1)

    horizon = onset.cumsum(axis=1)[:, [4, 9, 14]]

    assert np.all(
        horizon[:, 0] <= horizon[:, 1] + 1e-7
    )

    assert np.all(
        horizon[:, 1] <= horizon[:, 2] + 1e-7
    )

    assert np.allclose(
        joint[:, 0] + horizon[:, 2],
        1.0,
        atol=1e-6
    )

    assert np.allclose(
        family.sum(axis=1),
        horizon[:, 2],
        atol=1e-6
    )

    event_probability = np.maximum(
        horizon[:, 2],
        1e-30
    )

    family_cond = family / event_probability[:, None]
    onset_cond = onset / event_probability[:, None]

    # For normal nondegenerate softmax outputs, these should sum to 1.
    assert np.allclose(
        family_cond.sum(axis=1), 1.0, atol=1e-5
    )

    assert np.allclose(
        onset_cond.sum(axis=1), 1.0, atol=1e-5
    )

    predicted_family = family_cond.argmax(axis=1) + 1
    predicted_minute = onset_cond.argmax(axis=1) + 1

    expected_eta = (
        onset_cond
        @ np.arange(1, 16, dtype=np.float64)
    )

    return {
        "joint": joint,
        "horizon": horizon,
        "onset": onset,
        "family": family,
        "family_cond": family_cond,
        "onset_cond": onset_cond,
        "pred_family": predicted_family,
        "pred_minute": predicted_minute,
        "expected_eta": expected_eta
    }


def family_macro_f1_7(y, predicted):
    return float(
        f1_score(
            y,
            predicted,
            labels=[1, 2, 3, 4, 5],
            average="macro",
            zero_division=0
        )
    )


def subset_metric_7(metric_name, idx, pred, true_joint,
                    true_horizons, true_onset, true_family):
    mask_event = true_onset[idx] > 0

    if metric_name == "Joint_NLL":
        values = pred["joint"][idx, true_joint[idx]]
        return float(
            -np.log(np.clip(values, 1e-30, 1)).mean()
        )

    if metric_name in {"H5_AP", "H10_AP", "H15_AP"}:
        col = {
            "H5_AP": 0,
            "H10_AP": 1,
            "H15_AP": 2
        }[metric_name]

        return float(
            average_precision_score(
                true_horizons[idx, col],
                pred["horizon"][idx, col]
            )
        )

    if metric_name == "Family_Macro_F1":
        return family_macro_f1_7(
            true_family[idx][mask_event],
            pred["pred_family"][idx][mask_event]
        )

    if metric_name == "Family_Accuracy":
        return float(
            accuracy_score(
                true_family[idx][mask_event],
                pred["pred_family"][idx][mask_event]
            )
        )

    if metric_name == "ETA_MAE":
        return float(
            np.abs(
                pred["expected_eta"][idx][mask_event]
                - true_onset[idx][mask_event]
            ).mean()
        )

    raise ValueError(f"Unknown metric: {metric_name}")


# =============================================================================
# 7C — INTEGRITY AND MODEL SELECTION VERIFICATION
# =============================================================================

required_files_7 = [
    VAL_PATH_7,
    TRAIN_PATH_7,
    META_PATH_7,
    MANIFEST_PATH_7,
    LOCK_PATH_7,
    MODEL_PATH_7,
    SCALER_PATH_7,
    RECORD_PATH_7,
    REGISTRY_PATH_7,
    SELECTION_PATH_7,
    PROTOCOL_PATH_7,
    LSTM_SAFE_PATH_7,
    LSTM_ORIGINAL_PATH_7
]

for path in required_files_7:
    if not path.is_file():
        raise FileNotFoundError(path)

manifest_7 = read_json_7(MANIFEST_PATH_7)
lock_7 = read_json_7(LOCK_PATH_7)

record_7 = read_json_7(RECORD_PATH_7)
registry_7 = read_json_7(REGISTRY_PATH_7)

selection_7 = read_json_7(SELECTION_PATH_7)
protocol_7 = read_json_7(PROTOCOL_PATH_7)

scaler_state_7 = read_json_7(SCALER_PATH_7)

assert manifest_7["release_version"] == "2.0.0"

assert canonical_sha_7(manifest_7) == (
    lock_7["manifest_sha256_canonical"]
)

assert sha256_file_7(MODEL_PATH_7) == EXPECTED_TCN_SHA_7
assert registry_7["model_sha256"] == EXPECTED_TCN_SHA_7
assert record_7["model_sha256"] == EXPECTED_TCN_SHA_7

assert sha256_file_7(SCALER_PATH_7) == (
    registry_7["scaler_sha256"]
)

assert sha256_file_7(RECORD_PATH_7) == (
    registry_7["training_record_sha256"]
)

assert sha256_file_7(
    MODEL_DIR_7 / registry_7["history_file"]
) == registry_7["history_sha256"]

assert sha256_file_7(TRAIN_PATH_7) == (
    record_7["train_data_sha256"]
)

expected_val_sha_7 = (
    manifest_7["data_integrity"]["file_sha256"]
    ["windows_W10_VALIDATION.npz"]
)

assert sha256_file_7(VAL_PATH_7) == expected_val_sha_7

expected_meta_sha_7 = (
    manifest_7["data_integrity"]["file_sha256"]
    [META_PATH_7.name]
)

assert sha256_file_7(META_PATH_7) == expected_meta_sha_7

assert canonical_sha_7(protocol_7) == (
    selection_7["protocol_hash"]
)

assert selection_7["selected_config_id"] == "C02"
assert selection_7["selected_final_epochs"] == 21

assert record_7["selected_configuration_id"] == "C02"
assert record_7["training_epochs"] == 21
assert record_7["trainable_parameters"] == 20940

assert record_7["official_validation_used"] is False
assert record_7["official_test_used"] is False

assert registry_7["validation_status"] == "NOT_STARTED"
assert registry_7["test_status"] == "EMBARGOED"

assert (
    sha256_file_7(LSTM_ORIGINAL_PATH_7)
    == EXPECTED_ORIGINAL_LSTM_ARCHIVE_SHA_7
), "Original LSTM archive unexpectedly changed."

assert record_7["feature_names"] == (
    scaler_state_7["feature_names"]
)

assert record_7["output_definition"]["family_names"] == (
    FAMILY_NAMES_7
)

print("=" * 108)
print("TCN V2 COMPETING-RISK — OFFICIAL VALIDATION")
print("=" * 108)

print("Selected TCN configuration: C02")
print("TCN SHA256: VERIFIED")
print("Frozen dataset manifest: VERIFIED")
print("Official validation checksum: VERIFIED")
print("Full-TRAIN scaler: VERIFIED")
print("LSTM original archive unchanged: VERIFIED")
print("LSTM safe archive: FOUND")
print("Official TEST: NOT ACCESSED")

# =============================================================================
# 7D — LOAD FROZEN VALIDATION
# =============================================================================

with np.load(VAL_PATH_7, allow_pickle=False) as pack:
    X_VAL_7 = pack["X"].astype(np.float32)
    Y_VAL_7 = pack["y_horizons"].astype(np.int8)
    ONSET_VAL_7 = pack["onset_class"].astype(np.int16)
    FAMILY_VAL_7 = pack["family_class"].astype(np.int16)
    EPISODE_VAL_7 = (
        pd.Series(pack["episode_id"])
        .astype(str)
        .to_numpy()
    )
    MINUTE_VAL_7 = pack["minute"].astype(np.int16)

assert X_VAL_7.shape == (18839, 10, 14)
assert Y_VAL_7.shape == (18839, 3)

assert len(np.unique(EPISODE_VAL_7)) == 300
assert np.isfinite(X_VAL_7).all()
assert np.all(X_VAL_7 >= 0)

assert np.array_equal(
    ONSET_VAL_7 == 0,
    FAMILY_VAL_7 == 0
)

JOINT_VAL_7 = np.where(
    ONSET_VAL_7 == 0,
    0,
    5 * (ONSET_VAL_7 - 1) + FAMILY_VAL_7
).astype(np.int64)

assert set(np.unique(JOINT_VAL_7)) == set(range(76))

for j, horizon in enumerate(HORIZONS_7):
    reconstructed = (
        (ONSET_VAL_7 >= 1)
        & (ONSET_VAL_7 <= horizon)
    ).astype(np.int8)

    assert np.array_equal(
        reconstructed,
        Y_VAL_7[:, j]
    )

assert int((JOINT_VAL_7 > 0).sum()) == 3375

# Metadata is used only for reporting and subgroup checks.
metadata_7 = pd.read_csv(META_PATH_7)
metadata_7["Episode_ID"] = (
    metadata_7["Episode_ID"].astype(str)
)

assert metadata_7["Episode_ID"].is_unique

aligned_metadata_7 = (
    metadata_7
    .set_index("Episode_ID")
    .loc[EPISODE_VAL_7]
)

assert set(
    aligned_metadata_7["Split"].astype(str)
) == {"VALIDATION"}

SOURCE_VAL_7 = (
    aligned_metadata_7["Source"]
    .astype(str)
    .to_numpy()
)

META_FAMILY_7 = (
    aligned_metadata_7["Family"]
    .astype(str)
    .to_numpy()
)

HARD_BENIGN_7 = parse_boolean_7(
    aligned_metadata_7["Hard_Benign"]
)

WHOLE_BENIGN_7 = META_FAMILY_7 == "BENIGN"

assert not np.any(
    HARD_BENIGN_7 & ~WHOLE_BENIGN_7
)

assert np.all(
    JOINT_VAL_7[WHOLE_BENIGN_7] == 0
)

assert set(SOURCE_VAL_7) == {
    "CIC_IDS2017",
    "UNSW_NB15"
}

EVENT_MASK_7 = ONSET_VAL_7 > 0

print("\nVALIDATION DATASET")
print("Windows:", len(X_VAL_7))
print("Episodes:", len(np.unique(EPISODE_VAL_7)))
print("Future-event windows:", int(EVENT_MASK_7.sum()))
print("No-event windows:", int((~EVENT_MASK_7).sum()))

# =============================================================================
# 7E — APPLY SAVED FULL-TRAIN SCALER (NO FIT)
# =============================================================================

median_7 = np.asarray(
    scaler_state_7["median"],
    dtype=np.float32
)

scale_7 = np.asarray(
    scaler_state_7["scale"],
    dtype=np.float32
)

clip_7 = float(scaler_state_7["clip_value"])

assert median_7.shape == (14,)
assert scale_7.shape == (14,)
assert np.all(scale_7 > 0)
assert clip_7 == 8.0

X_VAL_SCALED_7 = np.ascontiguousarray(
    np.clip(
        (
            np.log1p(X_VAL_7)
            - median_7[None, None, :]
        ) / scale_7[None, None, :],
        -clip_7,
        clip_7
    ),
    dtype=np.float32
)

assert np.isfinite(X_VAL_SCALED_7).all()

print("\n✓ Saved TRAIN scaler applied to validation")
print("✓ No scaler refitting")

# =============================================================================
# 7F — LOAD EXACT SAVED C02 MODEL AND RUN INFERENCE
# =============================================================================

architecture_7 = record_7["model_architecture"]

expected_architecture_7 = {
    "input_dim": 14,
    "channels": 32,
    "num_blocks": 3,
    "kernel_size": 3,
    "dropout": 0.2,
    "dense_dim": 32,
    "num_classes": 76
}

assert architecture_7 == expected_architecture_7

DEVICE_7 = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

MODEL_7 = TCNCompetingRisk(
    **architecture_7
).to(DEVICE_7)

weights_7 = torch.load(
    MODEL_PATH_7,
    map_location="cpu",
    weights_only=True
)

MODEL_7.load_state_dict(
    weights_7,
    strict=True
)

MODEL_7.eval()
MODEL_7.requires_grad_(False)

assert sum(
    p.numel() for p in MODEL_7.parameters()
) == 20940

loader_7 = DataLoader(
    TensorDataset(
        torch.from_numpy(X_VAL_SCALED_7)
    ),
    batch_size=1024,
    shuffle=False,
    num_workers=0,
    pin_memory=(DEVICE_7.type == "cuda")
)

joint_parts_7 = []

with torch.inference_mode():

    for (xb,) in loader_7:

        xb = xb.to(
            DEVICE_7,
            non_blocking=True
        )

        logits = MODEL_7(xb)

        assert logits.shape[1] == 76
        assert torch.isfinite(logits).all()

        joint_parts_7.append(
            torch.softmax(
                logits,
                dim=1
            ).cpu().numpy().astype(np.float64)
        )

TCN_JOINT_7 = np.concatenate(
    joint_parts_7,
    axis=0
)

assert TCN_JOINT_7.shape == (18839, 76)

TCN_PRED_7 = derive_predictions_7(
    TCN_JOINT_7
)

print("\n✓ TCN checkpoint inference complete")
print("✓ Joint probabilities normalized")
print("✓ H5 <= H10 <= H15")
print("✓ Family/onset probability identities passed")

# =============================================================================
# 7G — LOAD SAFE LSTM PREDICTIONS AND VERIFY ROW ALIGNMENT
# =============================================================================

# No pickle required; safe copy created and verified previously.
with np.load(
    LSTM_SAFE_PATH_7,
    allow_pickle=False
) as archive:

    expected_keys = {
        "joint_probabilities",
        "episode_id",
        "minute",
        "joint_true",
        "horizon_true",
        "onset_true",
        "family_true",
        "horizon_probabilities"
    }

    assert expected_keys.issubset(
        set(archive.files)
    )

    LSTM_JOINT_7 = (
        archive["joint_probabilities"]
        .astype(np.float64)
    )

    LSTM_EPISODE_7 = (
        pd.Series(archive["episode_id"])
        .astype(str)
        .to_numpy()
    )

    LSTM_MINUTE_7 = archive["minute"]
    LSTM_JOINT_TRUE_7 = archive["joint_true"]
    LSTM_Y_TRUE_7 = archive["horizon_true"]
    LSTM_ONSET_TRUE_7 = archive["onset_true"]
    LSTM_FAMILY_TRUE_7 = archive["family_true"]

    LSTM_SAVED_HORIZON_7 = (
        archive["horizon_probabilities"]
        .astype(np.float64)
    )

assert LSTM_JOINT_7.shape == (18839, 76)

# CRITICAL: refuse to compare if any rows or labels differ.
assert np.array_equal(
    LSTM_EPISODE_7,
    EPISODE_VAL_7
)

assert np.array_equal(
    LSTM_MINUTE_7,
    MINUTE_VAL_7
)

assert np.array_equal(
    LSTM_JOINT_TRUE_7,
    JOINT_VAL_7
)

assert np.array_equal(
    LSTM_Y_TRUE_7,
    Y_VAL_7
)

assert np.array_equal(
    LSTM_ONSET_TRUE_7,
    ONSET_VAL_7
)

assert np.array_equal(
    LSTM_FAMILY_TRUE_7,
    FAMILY_VAL_7
)

LSTM_PRED_7 = derive_predictions_7(
    LSTM_JOINT_7
)

assert np.allclose(
    LSTM_PRED_7["horizon"],
    LSTM_SAVED_HORIZON_7,
    rtol=1e-6,
    atol=1e-6
)

# Independently confirm previously published LSTM results.
LSTM_REFERENCE_7 = {
    "Joint_NLL": 0.649275,
    "H5_AP": 0.724836,
    "H10_AP": 0.899137,
    "H15_AP": 0.936059,
    "Family_Accuracy": 0.808889,
    "Family_Macro_F1": 0.795823,
    "ETA_MAE": 2.536534
}

ALL_INDICES_7 = np.arange(
    len(JOINT_VAL_7)
)

for metric, expected in LSTM_REFERENCE_7.items():

    measured = subset_metric_7(
        metric,
        ALL_INDICES_7,
        LSTM_PRED_7,
        JOINT_VAL_7,
        Y_VAL_7,
        ONSET_VAL_7,
        FAMILY_VAL_7
    )

    assert abs(measured - expected) < 1e-5, (
        f"LSTM metric mismatch for {metric}: "
        f"measured={measured}, reference={expected}"
    )

print("\n✓ Safe LSTM archive loaded without pickle")
print("✓ Validation episode/minute rows match exactly")
print("✓ All six saved target arrays and labels verified")
print("✓ Previously reported LSTM metrics reproduced")
print("✓ Paired model comparison authorized")

# =============================================================================
# 7H — JOINT NLL AND HORIZON FORECASTING
# =============================================================================

TCN_JOINT_NLL_7 = subset_metric_7(
    "Joint_NLL",
    ALL_INDICES_7,
    TCN_PRED_7,
    JOINT_VAL_7,
    Y_VAL_7,
    ONSET_VAL_7,
    FAMILY_VAL_7
)

TCN_TOP1_7 = float(
    accuracy_score(
        JOINT_VAL_7,
        TCN_JOINT_7.argmax(axis=1)
    )
)

HORIZON_ROWS_7 = []
CALIBRATION_ROWS_7 = []

for j, h in enumerate(HORIZONS_7):

    scores, calibration = horizon_metrics_7(
        Y_VAL_7[:, j],
        TCN_PRED_7["horizon"][:, j]
    )

    HORIZON_ROWS_7.append({
        "Horizon": f"H{h}",
        **scores
    })

    for row in calibration:
        CALIBRATION_ROWS_7.append({
            "Horizon": f"H{h}",
            **row
        })

HORIZON_DF_7 = pd.DataFrame(
    HORIZON_ROWS_7
)

CALIBRATION_DF_7 = pd.DataFrame(
    CALIBRATION_ROWS_7
)

# =============================================================================
# 7I — CONDITIONAL FAMILY METRICS
# =============================================================================

true_f_7 = FAMILY_VAL_7[EVENT_MASK_7]

pred_f_7 = (
    TCN_PRED_7["pred_family"][EVENT_MASK_7]
)

FAMILY_ACCURACY_7 = float(
    accuracy_score(true_f_7, pred_f_7)
)

FAMILY_MACRO_F1_7 = family_macro_f1_7(
    true_f_7,
    pred_f_7
)

precision_7, recall_7, f1_7, supports_7 = (
    precision_recall_fscore_support(
        true_f_7,
        pred_f_7,
        labels=[1, 2, 3, 4, 5],
        zero_division=0
    )
)

PER_FAMILY_DF_7 = pd.DataFrame({
    "Family_ID": [1, 2, 3, 4, 5],
    "Family": FAMILY_NAMES_7,
    "Support": supports_7,
    "Precision": precision_7,
    "Recall": recall_7,
    "F1": f1_7
})

FAMILY_CM_DF_7 = pd.DataFrame(
    confusion_matrix(
        true_f_7,
        pred_f_7,
        labels=[1, 2, 3, 4, 5]
    ),
    index=[
        f"True_{name}"
        for name in FAMILY_NAMES_7
    ],
    columns=[
        f"Pred_{name}"
        for name in FAMILY_NAMES_7
    ]
)

# Paired LSTM vs TCN per-family results.
family_comparison_rows_7 = []

for family_id, family_name in enumerate(
    FAMILY_NAMES_7,
    start=1
):

    mask = true_f_7 == family_id

    tcn_recall = float(
        np.mean(pred_f_7[mask] == family_id)
    )

    lstm_recall = float(
        np.mean(
            LSTM_PRED_7["pred_family"][EVENT_MASK_7][mask]
            == family_id
        )
    )

    family_comparison_rows_7.append({
        "Family": family_name,
        "Support": int(mask.sum()),
        "TCN_Recall": tcn_recall,
        "LSTM_Recall": lstm_recall,
        "TCN_Minus_LSTM_Recall": tcn_recall - lstm_recall
    })

FAMILY_COMPARISON_DF_7 = pd.DataFrame(
    family_comparison_rows_7
)

# =============================================================================
# 7J — CONDITIONAL ONSET-TIME METRICS
# =============================================================================

true_onset_7 = ONSET_VAL_7[EVENT_MASK_7]

expected_eta_7 = (
    TCN_PRED_7["expected_eta"][EVENT_MASK_7]
)

predicted_minute_7 = (
    TCN_PRED_7["pred_minute"][EVENT_MASK_7]
)

ETA_MAE_7 = float(
    np.abs(expected_eta_7 - true_onset_7).mean()
)

ETA_RMSE_7 = float(
    np.sqrt(
        np.mean(
            (expected_eta_7 - true_onset_7) ** 2
        )
    )
)

ETA_EXACT_7 = float(
    np.mean(
        predicted_minute_7 == true_onset_7
    )
)

ETA_WITHIN_1_7 = float(
    np.mean(
        np.abs(predicted_minute_7 - true_onset_7) <= 1
    )
)

ETA_WITHIN_2_7 = float(
    np.mean(
        np.abs(predicted_minute_7 - true_onset_7) <= 2
    )
)

ETA_MINUTE_ROWS_7 = []

for minute in range(1, 16):

    mask = ONSET_VAL_7 == minute

    assert mask.any()

    ETA_MINUTE_ROWS_7.append({
        "True_Onset_Minute": minute,
        "Windows": int(mask.sum()),
        "Expected_ETA_MAE": float(
            np.mean(
                np.abs(
                    TCN_PRED_7["expected_eta"][mask]
                    - minute
                )
            )
        ),
        "Exact_Minute_Accuracy": float(
            np.mean(
                TCN_PRED_7["pred_minute"][mask]
                == minute
            )
        )
    })

ETA_BY_MINUTE_DF_7 = pd.DataFrame(
    ETA_MINUTE_ROWS_7
)

# =============================================================================
# 7K — FIXED-THRESHOLD DETECTION AND END-TO-END FAMILY
# =============================================================================

threshold_rows_7 = []
e2e_rows_7 = []

for j, h in enumerate(HORIZONS_7):

    y = Y_VAL_7[:, j]

    for threshold in FIXED_THRESHOLDS_7:

        alert = (
            TCN_PRED_7["horizon"][:, j] >= threshold
        )

        tn, fp, fn, tp = confusion_matrix(
            y,
            alert.astype(np.int8),
            labels=[0, 1]
        ).ravel()

        threshold_rows_7.append({
            "Horizon": f"H{h}",
            "Threshold": threshold,
            "TP": int(tp),
            "FP": int(fp),
            "TN": int(tn),
            "FN": int(fn),
            "Precision": (
                float(tp / (tp + fp))
                if tp + fp else 0.0
            ),
            "Recall": (
                float(tp / (tp + fn))
                if tp + fn else 0.0
            ),
            "FPR": (
                float(fp / (fp + tn))
                if fp + tn else 0.0
            )
        })

        correct_family = int(
            np.sum(
                alert
                & (y == 1)
                & (
                    TCN_PRED_7["pred_family"]
                    == FAMILY_VAL_7
                )
            )
        )

        e2e_rows_7.append({
            "Horizon": f"H{h}",
            "Threshold": threshold,
            "True_Event_Windows": int(y.sum()),
            "Detected_Event_Windows": int(tp),
            "Detected_With_Correct_Family": correct_family,
            "Correct_Family_Event_Recall": float(
                correct_family / y.sum()
            ),
            "Family_Accuracy_Among_Detected": (
                float(correct_family / tp)
                if tp else 0.0
            )
        })

THRESHOLD_DF_7 = pd.DataFrame(threshold_rows_7)
E2E_DF_7 = pd.DataFrame(e2e_rows_7)

# =============================================================================
# 7L — FALSE ALERTS ON WHOLLY BENIGN EPISODES
# =============================================================================

benign_rows_7 = []

categories_7 = {
    "HARD_BENIGN": (
        WHOLE_BENIGN_7 & HARD_BENIGN_7
    ),
    "ORDINARY_BENIGN": (
        WHOLE_BENIGN_7 & ~HARD_BENIGN_7
    ),
    "ALL_BENIGN": WHOLE_BENIGN_7
}

for j, h in enumerate(HORIZONS_7):

    for threshold in FIXED_THRESHOLDS_7:

        alerted = (
            TCN_PRED_7["horizon"][:, j] >= threshold
        )

        for category, mask in categories_7.items():

            episodes = np.unique(
                EPISODE_VAL_7[mask]
            )

            alerted_episodes = np.unique(
                EPISODE_VAL_7[mask & alerted]
            )

            benign_rows_7.append({
                "Horizon": f"H{h}",
                "Threshold": threshold,
                "Category": category,
                "Episodes": int(len(episodes)),
                "Episodes_With_Alert": int(
                    len(alerted_episodes)
                ),
                "Alert_Windows": int(
                    np.sum(mask & alerted)
                ),
                "Episode_Alert_Fraction": (
                    float(
                        len(alerted_episodes)
                        / len(episodes)
                    )
                    if len(episodes) else 0.0
                )
            })

BENIGN_DF_7 = pd.DataFrame(benign_rows_7)

assert len(np.unique(
    EPISODE_VAL_7[WHOLE_BENIGN_7]
)) == 75

# =============================================================================
# 7M — SOURCE-SPECIFIC PERFORMANCE
# =============================================================================

source_horizon_rows_7 = []
source_family_rows_7 = []
source_class_rows_7 = []

for source in sorted(np.unique(SOURCE_VAL_7)):

    mask = SOURCE_VAL_7 == source
    event_mask = mask & EVENT_MASK_7

    for j, h in enumerate(HORIZONS_7):

        scores, _ = horizon_metrics_7(
            Y_VAL_7[mask, j],
            TCN_PRED_7["horizon"][mask, j]
        )

        source_horizon_rows_7.append({
            "Source": source,
            "Horizon": f"H{h}",
            "Episodes": int(
                len(np.unique(EPISODE_VAL_7[mask]))
            ),
            **scores
        })

    true_family_source = FAMILY_VAL_7[event_mask]
    pred_family_source = (
        TCN_PRED_7["pred_family"][event_mask]
    )

    present_ids = sorted(
        set(true_family_source.tolist())
    )

    source_family_rows_7.append({
        "Source": source,
        "Event_Windows": int(event_mask.sum()),
        "True_Families_Present": ",".join(
            FAMILY_NAMES_7[i - 1]
            for i in present_ids
        ),
        "Family_Accuracy": float(
            accuracy_score(
                true_family_source,
                pred_family_source
            )
        ),
        "Family_Macro_F1_All5": family_macro_f1_7(
            true_family_source,
            pred_family_source
        ),
        "ETA_MAE": float(
            np.abs(
                TCN_PRED_7["expected_eta"][event_mask]
                - ONSET_VAL_7[event_mask]
            ).mean()
        )
    })

    pr, rc, fs, support = (
        precision_recall_fscore_support(
            true_family_source,
            pred_family_source,
            labels=[1, 2, 3, 4, 5],
            zero_division=0
        )
    )

    for i, name in enumerate(FAMILY_NAMES_7):

        source_class_rows_7.append({
            "Source": source,
            "Family": name,
            "Support": int(support[i]),
            "Precision": float(pr[i]),
            "Recall": float(rc[i]),
            "F1": float(fs[i])
        })

SOURCE_HORIZON_DF_7 = pd.DataFrame(
    source_horizon_rows_7
)

SOURCE_FAMILY_DF_7 = pd.DataFrame(
    source_family_rows_7
)

SOURCE_CLASS_DF_7 = pd.DataFrame(
    source_class_rows_7
)

# =============================================================================
# 7N — MATCHED OFFICIAL METRIC COMPARISON
# =============================================================================

METRIC_NAMES_7 = [
    "Joint_NLL",
    "H5_AP",
    "H10_AP",
    "H15_AP",
    "Family_Accuracy",
    "Family_Macro_F1",
    "ETA_MAE"
]

tcn_metric_values_7 = {}
lstm_metric_values_7 = {}

for metric in METRIC_NAMES_7:

    tcn_metric_values_7[metric] = subset_metric_7(
        metric,
        ALL_INDICES_7,
        TCN_PRED_7,
        JOINT_VAL_7,
        Y_VAL_7,
        ONSET_VAL_7,
        FAMILY_VAL_7
    )

    lstm_metric_values_7[metric] = subset_metric_7(
        metric,
        ALL_INDICES_7,
        LSTM_PRED_7,
        JOINT_VAL_7,
        Y_VAL_7,
        ONSET_VAL_7,
        FAMILY_VAL_7
    )

COMPARISON_DF_7 = pd.DataFrame([
    {
        "Metric": metric,
        "TCN": tcn_metric_values_7[metric],
        "LSTM": lstm_metric_values_7[metric],
        "TCN_Minus_LSTM": (
            tcn_metric_values_7[metric]
            - lstm_metric_values_7[metric]
        ),
        "Preferred_Direction": (
            "LOWER"
            if metric in {"Joint_NLL", "ETA_MAE"}
            else "HIGHER"
        )
    }
    for metric in METRIC_NAMES_7
])

# =============================================================================
# 7O — EPISODE-CLUSTER BOOTSTRAP AND PAIRED CIs
# =============================================================================

rng_7 = np.random.default_rng(
    BOOTSTRAP_SEED_7
)

unique_episodes_7 = np.unique(
    EPISODE_VAL_7
)

episode_to_rows_7 = [
    np.flatnonzero(
        EPISODE_VAL_7 == episode
    )
    for episode in unique_episodes_7
]

assert len(episode_to_rows_7) == 300

bootstrap_tcn_7 = {
    metric: [] for metric in METRIC_NAMES_7
}

bootstrap_diff_7 = {
    metric: [] for metric in METRIC_NAMES_7
}

valid_bootstrap_samples_7 = 0

for replicate in range(BOOTSTRAP_REPLICATES_7):

    sampled_episode_ids = rng_7.integers(
        low=0,
        high=len(unique_episodes_7),
        size=len(unique_episodes_7)
    )

    indices = np.concatenate([
        episode_to_rows_7[i]
        for i in sampled_episode_ids
    ])

    # Guard against degenerate resamples.
    if not np.any(ONSET_VAL_7[indices] > 0):
        continue

    if any(
        len(np.unique(
            Y_VAL_7[indices, j]
        )) < 2
        for j in range(3)
    ):
        continue

    valid_bootstrap_samples_7 += 1

    for metric in METRIC_NAMES_7:

        tcn_score = subset_metric_7(
            metric,
            indices,
            TCN_PRED_7,
            JOINT_VAL_7,
            Y_VAL_7,
            ONSET_VAL_7,
            FAMILY_VAL_7
        )

        lstm_score = subset_metric_7(
            metric,
            indices,
            LSTM_PRED_7,
            JOINT_VAL_7,
            Y_VAL_7,
            ONSET_VAL_7,
            FAMILY_VAL_7
        )

        bootstrap_tcn_7[metric].append(
            tcn_score
        )

        bootstrap_diff_7[metric].append(
            tcn_score - lstm_score
        )

assert valid_bootstrap_samples_7 >= 250

CI_DICT_7 = {}
PAIRED_ROWS_7 = []

for metric in METRIC_NAMES_7:

    values = np.asarray(
        bootstrap_tcn_7[metric]
    )

    differences = np.asarray(
        bootstrap_diff_7[metric]
    )

    tcn_lower, tcn_upper = np.percentile(
        values,
        [2.5, 97.5]
    )

    diff_lower, diff_upper = np.percentile(
        differences,
        [2.5, 97.5]
    )

    CI_DICT_7[metric] = {
        "estimate": tcn_metric_values_7[metric],
        "lower_95": float(tcn_lower),
        "upper_95": float(tcn_upper),
        "replicates": int(len(values))
    }

    PAIRED_ROWS_7.append({
        "Metric": metric,
        "TCN_Minus_LSTM": float(
            tcn_metric_values_7[metric]
            - lstm_metric_values_7[metric]
        ),
        "Lower_95": float(diff_lower),
        "Upper_95": float(diff_upper),
        "Bootstrap_Replicates": int(
            len(differences)
        )
    })

PAIRED_DF_7 = pd.DataFrame(
    PAIRED_ROWS_7
)

print("\n✓ Episode-cluster bootstrap complete")
print("Valid bootstrap resamples:", valid_bootstrap_samples_7)
print("✓ Matched paired-bootstrap intervals calculated")

# =============================================================================
# 7P — SAVE ALL RESULTS TO A NEW STAGING DIRECTORY
# =============================================================================

PREDICTIONS_DF_7 = pd.DataFrame({
    "Episode_ID": EPISODE_VAL_7,
    "Minute": MINUTE_VAL_7,
    "Source": SOURCE_VAL_7,
    "True_Joint_Class": JOINT_VAL_7,
    "True_Onset_Minute": ONSET_VAL_7,
    "True_Family_ID": FAMILY_VAL_7,
    "True_H5": Y_VAL_7[:, 0],
    "True_H10": Y_VAL_7[:, 1],
    "True_H15": Y_VAL_7[:, 2],
    "P_No_Event_Within_15": TCN_JOINT_7[:, 0],
    "P_H5": TCN_PRED_7["horizon"][:, 0],
    "P_H10": TCN_PRED_7["horizon"][:, 1],
    "P_H15": TCN_PRED_7["horizon"][:, 2],
    "Predicted_Family_If_Event": TCN_PRED_7["pred_family"],
    "Predicted_Onset_Minute_If_Event": TCN_PRED_7["pred_minute"],
    "Expected_ETA_If_Event": TCN_PRED_7["expected_eta"],
    "True_Joint_Class_Probability": (
        TCN_JOINT_7[
            ALL_INDICES_7,
            JOINT_VAL_7
        ]
    )
})

assert not PREDICTIONS_DF_7.duplicated(
    ["Episode_ID", "Minute"]
).any()

OFFICIAL_REPORT_7 = {
    "experiment": "TCN_V2_COMPETING_RISK_OFFICIAL_VALIDATION",
    "dataset": "Synthetic_NAF_v2",
    "release_version": "2.0.0",
    "configuration": "C02",
    "model_sha256": EXPECTED_TCN_SHA_7,
    "original_lstm_archive_sha256": (
        EXPECTED_ORIGINAL_LSTM_ARCHIVE_SHA_7
    ),
    "safe_lstm_archive_sha256": sha256_file_7(
        LSTM_SAFE_PATH_7
    ),
    "validation_sha256": expected_val_sha_7,
    "training_epochs": 21,
    "trainable_parameters": 20940,
    "validation_windows": int(len(JOINT_VAL_7)),
    "validation_episodes": int(len(unique_episodes_7)),
    "future_event_windows": int(EVENT_MASK_7.sum()),
    "joint_nll": TCN_JOINT_NLL_7,
    "joint_top1_accuracy": TCN_TOP1_7,
    "horizon_metrics": HORIZON_ROWS_7,
    "family_metrics": {
        "evaluation_condition": "TRUE_EVENT_WITHIN_15_MINUTES",
        "accuracy": FAMILY_ACCURACY_7,
        "macro_f1": FAMILY_MACRO_F1_7,
        "event_windows": int(EVENT_MASK_7.sum())
    },
    "eta_metrics": {
        "evaluation_condition": "TRUE_EVENT_WITHIN_15_MINUTES",
        "expected_eta_mae": ETA_MAE_7,
        "expected_eta_rmse": ETA_RMSE_7,
        "exact_minute_accuracy": ETA_EXACT_7,
        "within_one_minute": ETA_WITHIN_1_7,
        "within_two_minutes": ETA_WITHIN_2_7
    },
    "bootstrap": {
        "unit": "episode",
        "seed": BOOTSTRAP_SEED_7,
        "requested_replicates": BOOTSTRAP_REPLICATES_7,
        "valid_replicates": valid_bootstrap_samples_7,
        "confidence_intervals": CI_DICT_7
    },
    "paired_comparison": {
        "reference": "LSTM_V2_CompetingRisk",
        "row_alignment": "VERIFIED_EXACT",
        "same_bootstrap_episode_resamples": True,
        "metrics": PAIRED_ROWS_7
    },
    "fixed_thresholds": FIXED_THRESHOLDS_7,
    "model_retrained": False,
    "scaler_refitted": False,
    "hyperparameters_tuned": False,
    "official_test_accessed": False,
    "frozen_dataset_modified": False,
    "python_version": platform.python_version(),
    "torch_version": torch.__version__,
    "evaluated_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "limitations": [
        "Synthetic controlled benchmark, not live-network validation.",
        "Overlapping time-series windows are dependent.",
        "Episode bootstrap does not fully model calibration-group dependence.",
        "Family and ETA metrics are conditional on true future onset.",
        "Fixed thresholds are diagnostic and were not tuned.",
        "Validation comparisons are descriptive, not independent TEST results."
    ]
}

STAGING_DIR_7.mkdir(
    parents=True,
    exist_ok=False
)

TABLES_7 = {
    "HORIZON_METRICS.csv": HORIZON_DF_7,
    "CALIBRATION_10_BINS.csv": CALIBRATION_DF_7,
    "FAMILY_PER_CLASS_METRICS.csv": PER_FAMILY_DF_7,
    "FAMILY_CONFUSION_MATRIX.csv": FAMILY_CM_DF_7,
    "TCN_VS_LSTM_PER_FAMILY.csv": FAMILY_COMPARISON_DF_7,
    "ETA_BY_TRUE_MINUTE.csv": ETA_BY_MINUTE_DF_7,
    "FIXED_THRESHOLD_METRICS.csv": THRESHOLD_DF_7,
    "END_TO_END_FAMILY_METRICS.csv": E2E_DF_7,
    "BENIGN_FALSE_ALERTS.csv": BENIGN_DF_7,
    "SOURCE_HORIZON_METRICS.csv": SOURCE_HORIZON_DF_7,
    "SOURCE_FAMILY_ETA_METRICS.csv": SOURCE_FAMILY_DF_7,
    "SOURCE_FAMILY_PER_CLASS.csv": SOURCE_CLASS_DF_7,
    "TCN_VS_LSTM_COMPARISON.csv": COMPARISON_DF_7,
    "PAIRED_LSTM_BOOTSTRAP_CI.csv": PAIRED_DF_7,
    "VALIDATION_PREDICTIONS.csv": PREDICTIONS_DF_7
}

for filename, dataframe in TABLES_7.items():

    dataframe.to_csv(
        STAGING_DIR_7 / filename,
        index=(
            filename == "FAMILY_CONFUSION_MATRIX.csv"
        ),
        float_format="%.17g"
    )

np.savez_compressed(
    STAGING_DIR_7 / "JOINT_76_CLASS_VALIDATION_PREDICTIONS.npz",
    joint_probabilities=TCN_JOINT_7,
    episode_id=np.asarray(
        EPISODE_VAL_7,
        dtype=str
    ),
    minute=MINUTE_VAL_7,
    joint_true=JOINT_VAL_7,
    horizon_true=Y_VAL_7,
    onset_true=ONSET_VAL_7,
    family_true=FAMILY_VAL_7,
    horizon_probabilities=TCN_PRED_7["horizon"]
)

save_json_7(
    STAGING_DIR_7 / "BOOTSTRAP_CONFIDENCE_INTERVALS.json",
    CI_DICT_7
)

save_json_7(
    STAGING_DIR_7 / "OFFICIAL_VALIDATION_REPORT.json",
    OFFICIAL_REPORT_7
)

artifact_hashes_7 = {
    path.name: sha256_file_7(path)
    for path in sorted(STAGING_DIR_7.iterdir())
    if path.is_file()
}

save_json_7(
    STAGING_DIR_7 / "ARTIFACT_SHA256SUMS.json",
    artifact_hashes_7
)

# Verify prediction archive can be safely reloaded.
with np.load(
    STAGING_DIR_7 / "JOINT_76_CLASS_VALIDATION_PREDICTIONS.npz",
    allow_pickle=False
) as saved:

    assert np.array_equal(
        saved["joint_true"],
        JOINT_VAL_7
    )

    assert np.array_equal(
        saved["episode_id"],
        EPISODE_VAL_7
    )

    assert np.allclose(
        saved["joint_probabilities"],
        TCN_JOINT_7,
        rtol=0,
        atol=0
    )

for filename, expected_hash in artifact_hashes_7.items():

    assert sha256_file_7(
        STAGING_DIR_7 / filename
    ) == expected_hash

# Make completed results available atomically.
if OUTPUT_DIR_7.exists():
    raise RuntimeError(
        "Official output directory appeared during evaluation. "
        "Refusing to overwrite it."
    )

STAGING_DIR_7.rename(OUTPUT_DIR_7)

# Recheck existing model and original archive after saving.
assert sha256_file_7(MODEL_PATH_7) == EXPECTED_TCN_SHA_7

assert sha256_file_7(LSTM_ORIGINAL_PATH_7) == (
    EXPECTED_ORIGINAL_LSTM_ARCHIVE_SHA_7
)

assert sha256_file_7(VAL_PATH_7) == expected_val_sha_7

# =============================================================================
# 7Q — PRINT COMPLETE EVALUATION REPORT
# =============================================================================

print("\n" + "=" * 110)
print("TCN V2 COMPETING-RISK — OFFICIAL VALIDATION COMPLETE")
print("=" * 110)

print("\nJOINT DISTRIBUTION")
print(f"Joint NLL                 : {TCN_JOINT_NLL_7:.6f}")
print(f"Joint top-1 accuracy      : {TCN_TOP1_7:.6f}")

print("\nH5 / H10 / H15 FORECASTING")
print(
    HORIZON_DF_7.round(6).to_string(
        index=False
    )
)

print("\nCONDITIONAL FAMILY PERFORMANCE")
print(f"Family accuracy          : {FAMILY_ACCURACY_7:.6f}")
print(f"Family macro-F1          : {FAMILY_MACRO_F1_7:.6f}")

print("\nPER-FAMILY METRICS")
print(
    PER_FAMILY_DF_7.round(6).to_string(
        index=False
    )
)

print("\nFAMILY CONFUSION MATRIX")
print(
    FAMILY_CM_DF_7.to_string()
)

print("\nTCN VS LSTM — PER-FAMILY RECALL")
print(
    FAMILY_COMPARISON_DF_7.round(6).to_string(
        index=False
    )
)

print("\nCONDITIONAL ONSET-TIME PERFORMANCE")
print(f"Expected ETA MAE         : {ETA_MAE_7:.6f} minutes")
print(f"Expected ETA RMSE        : {ETA_RMSE_7:.6f} minutes")
print(f"Exact-minute accuracy    : {ETA_EXACT_7:.6f}")
print(f"Within 1 minute          : {ETA_WITHIN_1_7:.6f}")
print(f"Within 2 minutes         : {ETA_WITHIN_2_7:.6f}")

print("\nFIXED-THRESHOLD ATTACK DETECTION")
print(
    THRESHOLD_DF_7.round(6).to_string(
        index=False
    )
)

print("\nEND-TO-END FAMILY FORECASTING")
print(
    E2E_DF_7.round(6).to_string(
        index=False
    )
)

print("\nBENIGN FALSE ALERTS")
print(
    BENIGN_DF_7.round(6).to_string(
        index=False
    )
)

print("\nSOURCE-SPECIFIC HORIZON FORECASTING")
print(
    SOURCE_HORIZON_DF_7.round(6).to_string(
        index=False
    )
)

print("\nSOURCE-SPECIFIC FAMILY AND ETA")
print(
    SOURCE_FAMILY_DF_7.round(6).to_string(
        index=False
    )
)

print("\nSOURCE-SPECIFIC PER-FAMILY METRICS")
print(
    SOURCE_CLASS_DF_7.round(6).to_string(
        index=False
    )
)

print("\nEPISODE-BOOTSTRAP 95% CONFIDENCE INTERVALS")
for metric, ci in CI_DICT_7.items():

    print(
        f"{metric:22s}: "
        f"[{ci['lower_95']:.6f}, {ci['upper_95']:.6f}]"
    )

print("\nTCN VERSUS LSTM — MATCHED VALIDATION METRICS")
print(
    COMPARISON_DF_7.round(6).to_string(
        index=False
    )
)

print("\nPAIRED EPISODE-BOOTSTRAP DIFFERENCES (TCN - LSTM)")
print(
    PAIRED_DF_7.round(6).to_string(
        index=False
    )
)

print("\nMODEL SHA256")
print(EXPECTED_TCN_SHA_7)

print("\nSAVED OFFICIAL VALIDATION REPORTS")
print(OUTPUT_DIR_7)

print("\nFINAL INTEGRITY STATUS")
print("TCN weights: UNCHANGED")
print("LSTM weights: UNCHANGED")
print("Original LSTM prediction archive: UNCHANGED")
print("Training: NONE")
print("Hyperparameter tuning: NONE")
print("Scaler refitting: NONE")
print("Official TEST: NOT ACCESSED")
print("Frozen dataset: UNCHANGED")

print("=" * 110)
print("CELL 7 — SUCCESSFULLY COMPLETED")
print("=" * 110)


In [ ]:

# =============================================================================
# NETWORK ATTACK FORECASTING — CELL 8
# FINAL VALIDATION AUDIT AND FOUR-MODEL COMPARISON
#
# READ-ONLY INPUTS:
#   - Frozen official VALIDATION
#   - Saved TCN and competing-risk LSTM validation predictions
#   - Existing reported XGBoost and Stacked LSTM benchmark metrics
#
# OUTPUT:
#   Synthetic_NAF_v2/models/FINAL_VALIDATION_AUDIT/
#
# No fitting, retraining, calibration or TEST access.
# =============================================================================

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    f1_score,
    confusion_matrix,
    precision_recall_fscore_support
)

ROOT8 = Path("/content/drive/MyDrive/network-attack-forecasting")
BASE8 = ROOT8 / "Synthetic_NAF_v2"
DATA8 = BASE8 / "release_candidate_corrected"
MODELS8 = BASE8 / "models"

TCN8 = MODELS8 / "TCN_V2_CompetingRisk/official_validation"
LSTM8 = MODELS8 / "LSTM_V2_CompetingRisk/official_validation"

OUTPUT8 = MODELS8 / "FINAL_VALIDATION_AUDIT"

VAL8 = DATA8 / "windows_W10_VALIDATION.npz"
META8 = DATA8 / "episode_metadata_DO_NOT_USE_AS_PREDICTORS.csv"

TCN_FILE8 = TCN8 / "JOINT_76_CLASS_VALIDATION_PREDICTIONS.npz"
LSTM_FILE8 = (
    LSTM8 / "JOINT_76_CLASS_VALIDATION_PREDICTIONS_SAFE.npz"
)
TCN_REPORT8 = TCN8 / "OFFICIAL_VALIDATION_REPORT.json"

FAMILIES8 = [
    "BRUTE_AUTH",
    "DOS_DDOS",
    "EXPLOIT_APP",
    "FUZZING",
    "RECON_SCAN"
]

HORIZONS8 = [5, 10, 15]
THRESHOLDS8 = [0.10, 0.20, 0.50]
SEED8 = 20261024
BOOTSTRAPS8 = 500

# Never overwrite any existing final audit.
if OUTPUT8.exists():
    raise RuntimeError(
        "FINAL_VALIDATION_AUDIT already exists. "
        "Inspect the existing report before running again."
    )

for file8 in [VAL8, META8, TCN_FILE8, LSTM_FILE8, TCN_REPORT8]:
    if not file8.is_file():
        raise FileNotFoundError(file8)

def sha8(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def read_json8(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def write_json8(path, data):
    Path(path).write_text(
        json.dumps(data, indent=2, sort_keys=True, allow_nan=False),
        encoding="utf-8"
    )

def calc_joint8(joint):
    joint = np.asarray(joint, dtype=np.float64)

    assert joint.ndim == 2 and joint.shape[1] == 76
    assert np.isfinite(joint).all()
    assert (joint >= 0).all()
    assert np.allclose(joint.sum(axis=1), 1, atol=1e-6)

    matrix = joint[:, 1:].reshape(-1, 15, 5)
    onset = matrix.sum(axis=2)
    family = matrix.sum(axis=1)
    horizon = onset.cumsum(axis=1)[:, [4, 9, 14]]

    assert np.all(horizon[:, 0] <= horizon[:, 1] + 1e-7)
    assert np.all(horizon[:, 1] <= horizon[:, 2] + 1e-7)
    assert np.allclose(
        horizon[:, 2] + joint[:, 0], 1, atol=1e-6
    )

    event = np.maximum(horizon[:, 2], 1e-30)

    return {
        "joint": joint,
        "horizon": horizon,
        "pred_family": family.argmax(axis=1) + 1,
        "eta": (onset / event[:, None])
               @ np.arange(1, 16, dtype=np.float64)
    }

def metrics8(pred, idx, joint_true, h_true, onset_true, family_true):
    ev = onset_true[idx] > 0
    if not ev.any():
        raise ValueError("No positive event windows.")

    p_true = pred["joint"][idx, joint_true[idx]]
    result = {
        "Joint_NLL": float(
            -np.log(np.maximum(p_true, 1e-30)).mean()
        ),
        "Family_Accuracy": float(np.mean(
            pred["pred_family"][idx][ev] == family_true[idx][ev]
        )),
        "Family_Macro_F1": float(f1_score(
            family_true[idx][ev],
            pred["pred_family"][idx][ev],
            labels=[1, 2, 3, 4, 5],
            average="macro",
            zero_division=0
        )),
        "ETA_MAE": float(np.mean(np.abs(
            pred["eta"][idx][ev] - onset_true[idx][ev]
        )))
    }

    for j, h in enumerate(HORIZONS8):
        y = h_true[idx, j]
        p = pred["horizon"][idx, j]
        if len(np.unique(y)) == 2:
            result[f"H{h}_AP"] = float(
                average_precision_score(y, p)
            )
            result[f"H{h}_ROC_AUC"] = float(
                roc_auc_score(y, p)
            )
        else:
            result[f"H{h}_AP"] = np.nan
            result[f"H{h}_ROC_AUC"] = np.nan

        result[f"H{h}_Brier"] = float(
            brier_score_loss(y, p)
        )

    return result

print("=" * 105)
print("CELL 8 — FINAL VALIDATION AUDIT")
print("=" * 105)
print("Mode: READ-ONLY ANALYSIS")
print("Official TEST: NOT ACCESSED")

# =============================================================================
# A — RECONSTRUCT OFFICIAL VALIDATION TRUTH
# =============================================================================

with np.load(VAL8, allow_pickle=False) as z:
    eid8 = np.asarray(z["episode_id"]).astype(str)
    minute8 = np.asarray(z["minute"])
    onset8 = np.asarray(z["onset_class"])
    family8 = np.asarray(z["family_class"])
    truth_h8 = np.asarray(z["y_horizons"])

joint_true8 = np.where(
    onset8 == 0, 0, 5 * (onset8 - 1) + family8
).astype(np.int64)

n8 = len(joint_true8)
idx8 = np.arange(n8)

assert n8 == 18839
assert len(np.unique(eid8)) == 300
assert int(np.sum(joint_true8 > 0)) == 3375

meta8 = pd.read_csv(META8)
meta8["Episode_ID"] = meta8["Episode_ID"].astype(str)

assert meta8["Episode_ID"].is_unique
aligned8 = meta8.set_index("Episode_ID").loc[eid8]
assert set(aligned8["Split"].astype(str)) == {"VALIDATION"}

source8 = aligned8["Source"].astype(str).to_numpy()
meta_family8 = aligned8["Family"].astype(str).to_numpy()

assert set(source8) == {"CIC_IDS2017", "UNSW_NB15"}

# =============================================================================
# B — LOAD SAVED JOINT PREDICTIONS, EXACT ROW ALIGNMENT
# =============================================================================

def load_predictions8(path):
    with np.load(path, allow_pickle=False) as z:
        keys = set(z.files)
        required = {
            "joint_probabilities",
            "episode_id",
            "minute",
            "joint_true",
            "horizon_true",
            "onset_true",
            "family_true"
        }
        assert required.issubset(keys), required - keys

        joint = np.asarray(z["joint_probabilities"])
        episodes = np.asarray(z["episode_id"]).astype(str)
        minutes = np.asarray(z["minute"])

        assert np.array_equal(episodes, eid8)
        assert np.array_equal(minutes, minute8)
        assert np.array_equal(z["joint_true"], joint_true8)
        assert np.array_equal(z["horizon_true"], truth_h8)
        assert np.array_equal(z["onset_true"], onset8)
        assert np.array_equal(z["family_true"], family8)

        predictions = calc_joint8(joint)

        if "horizon_probabilities" in keys:
            assert np.allclose(
                predictions["horizon"],
                z["horizon_probabilities"],
                atol=1e-6
            )

        return predictions

preds8 = {
    "Competing-Risk LSTM": load_predictions8(LSTM_FILE8),
    "Competing-Risk TCN": load_predictions8(TCN_FILE8)
}

print("✓ TCN and LSTM prediction arrays verified")
print("✓ Exact episode/minute/target alignment")

# =============================================================================
# C — REPRODUCE EXISTING JOINT-FORECAST METRICS
# =============================================================================

joint_rows8 = []

for name, pred in preds8.items():
    m = metrics8(
        pred, idx8, joint_true8, truth_h8, onset8, family8
    )
    joint_rows8.append({"Model": name, **m})

JOINT_RESULTS8 = pd.DataFrame(joint_rows8)

report8 = read_json8(TCN_REPORT8)
tcn_row8 = JOINT_RESULTS8.set_index("Model").loc[
    "Competing-Risk TCN"
]

assert abs(
    tcn_row8["Joint_NLL"] - report8["joint_nll"]
) < 1e-9

assert abs(
    tcn_row8["Family_Macro_F1"]
    - report8["family_metrics"]["macro_f1"]
) < 1e-9

# =============================================================================
# D — FOUR-MODEL COMPARISON
#
# XGBoost and Stacked LSTM figures are previous reported
# benchmark values, NOT recomputed from prediction archives.
# Keep unavailable metrics as NaN.
# =============================================================================

FOUR_MODEL8 = pd.DataFrame([
    {
        "Model": "XGBoost V2",
        "H5_AP": 0.676304,
        "H10_AP": 0.875130,
        "H15_AP": np.nan,
        "Joint_NLL": np.nan,
        "Family_Macro_F1": np.nan,
        "ETA_MAE": np.nan,
        "Evidence": "Previously reported official validation"
    },
    {
        "Model": "Stacked Horizon LSTM V2",
        "H5_AP": 0.761005,
        "H10_AP": 0.906018,
        "H15_AP": 0.935854,
        "Joint_NLL": np.nan,
        "Family_Macro_F1": np.nan,
        "ETA_MAE": np.nan,
        "Evidence": "Previously reported official validation"
    }
] + [
    {
        "Model": name,
        "H5_AP": float(row["H5_AP"]),
        "H10_AP": float(row["H10_AP"]),
        "H15_AP": float(row["H15_AP"]),
        "Joint_NLL": float(row["Joint_NLL"]),
        "Family_Macro_F1": float(row["Family_Macro_F1"]),
        "ETA_MAE": float(row["ETA_MAE"]),
        "Evidence": "Recomputed from saved aligned validation predictions"
    }
    for name, row in JOINT_RESULTS8.set_index("Model").iterrows()
])

# =============================================================================
# E — SOURCE-BY-FAMILY RECALL AND CONFUSION
# =============================================================================

family_rows8 = []
source_rows8 = []
confusion_frames8 = {}

for name, pred in preds8.items():
    event8 = onset8 > 0

    for source in sorted(np.unique(source8)):
        mask = event8 & (source8 == source)

        source_metrics = metrics8(
            pred,
            np.flatnonzero(source8 == source),
            joint_true8, truth_h8, onset8, family8
        )

        source_rows8.append({
            "Model": name,
            "Source": source,
            "Event_Windows": int(mask.sum()),
            **source_metrics
        })

        true = family8[mask]
        guessed = pred["pred_family"][mask]

        precision, recall, f1, support = (
            precision_recall_fscore_support(
                true, guessed,
                labels=[1, 2, 3, 4, 5],
                zero_division=0
            )
        )

        for j, fname in enumerate(FAMILIES8):
            family_rows8.append({
                "Model": name,
                "Source": source,
                "Family": fname,
                "Support": int(support[j]),
                "Precision": float(precision[j]),
                "Recall": float(recall[j]),
                "F1": float(f1[j])
            })

        confusion_frames8[(name, source)] = pd.DataFrame(
            confusion_matrix(
                true, guessed, labels=[1, 2, 3, 4, 5]
            ),
            index=[f"True_{f}" for f in FAMILIES8],
            columns=[f"Pred_{f}" for f in FAMILIES8]
        )

SOURCE_RESULTS8 = pd.DataFrame(source_rows8)
FAMILY_SOURCE8 = pd.DataFrame(family_rows8)

# Explicitly measure the known EXPLOIT_APP failure.
EXPLOIT_AUDIT8 = FAMILY_SOURCE8[
    FAMILY_SOURCE8["Family"] == "EXPLOIT_APP"
].copy()

# =============================================================================
# F — FIXED-THRESHOLD ALERT AUDIT
#
# Diagnostics only. No threshold is selected or optimized.
# =============================================================================

alert_rows8 = []

for name, pred in preds8.items():
    for j, horizon in enumerate(HORIZONS8):
        y = truth_h8[:, j]
        p = pred["horizon"][:, j]

        for threshold in THRESHOLDS8:
            alerts = p >= threshold
            tn, fp, fn, tp = confusion_matrix(
                y, alerts.astype(int), labels=[0, 1]
            ).ravel()

            benign = meta_family8 == "BENIGN"
            benign_ep = np.unique(eid8[benign])
            benign_alert_ep = np.unique(
                eid8[benign & alerts]
            )

            alert_rows8.append({
                "Model": name,
                "Horizon": f"H{horizon}",
                "Threshold": threshold,
                "TP": int(tp),
                "FP": int(fp),
                "TN": int(tn),
                "FN": int(fn),
                "Precision": float(tp / (tp + fp)) if tp + fp else 0,
                "Recall": float(tp / (tp + fn)) if tp + fn else 0,
                "False_Positive_Rate": (
                    float(fp / (fp + tn)) if fp + tn else 0
                ),
                "Whole_Benign_Episodes": int(len(benign_ep)),
                "Benign_Episodes_Alerted": int(len(benign_alert_ep)),
                "Benign_Episode_Alert_Fraction": (
                    float(len(benign_alert_ep) / len(benign_ep))
                    if len(benign_ep) else 0
                )
            })

ALERT_RESULTS8 = pd.DataFrame(alert_rows8)

# =============================================================================
# G — PAIRED EPISODE BOOTSTRAP, JOINT MODELS
#
# Same resampled episodes for both models.
# Differences are TCN minus LSTM.
# =============================================================================

rng8 = np.random.default_rng(SEED8)
episodes8 = np.unique(eid8)
episode_rows8 = [
    np.flatnonzero(eid8 == e) for e in episodes8
]

metric_names8 = [
    "Joint_NLL",
    "H5_AP",
    "H10_AP",
    "H15_AP",
    "Family_Accuracy",
    "Family_Macro_F1",
    "ETA_MAE"
]

paired_samples8 = {k: [] for k in metric_names8}

for iteration in range(BOOTSTRAPS8):
    selected = rng8.integers(
        0, len(episodes8), size=len(episodes8)
    )
    ix = np.concatenate([episode_rows8[i] for i in selected])

    if any(
        len(np.unique(truth_h8[ix, j])) != 2
        for j in range(3)
    ):
        continue

    a = metrics8(
        preds8["Competing-Risk TCN"], ix,
        joint_true8, truth_h8, onset8, family8
    )
    b = metrics8(
        preds8["Competing-Risk LSTM"], ix,
        joint_true8, truth_h8, onset8, family8
    )

    for metric in metric_names8:
        paired_samples8[metric].append(
            a[metric] - b[metric]
        )

paired_rows8 = []

tcn_full8 = JOINT_RESULTS8.set_index("Model").loc[
    "Competing-Risk TCN"
]
lstm_full8 = JOINT_RESULTS8.set_index("Model").loc[
    "Competing-Risk LSTM"
]

for metric in metric_names8:
    values = paired_samples8[metric]
    assert len(values) >= 400

    lo, hi = np.percentile(values, [2.5, 97.5])

    paired_rows8.append({
        "Metric": metric,
        "TCN": float(tcn_full8[metric]),
        "LSTM": float(lstm_full8[metric]),
        "TCN_Minus_LSTM": float(
            tcn_full8[metric] - lstm_full8[metric]
        ),
        "CI_Lower_95": float(lo),
        "CI_Upper_95": float(hi),
        "Bootstrap_Replicates": len(values),
        "CI_Includes_Zero": bool(lo <= 0 <= hi)
    })

PAIRED_RESULTS8 = pd.DataFrame(paired_rows8)

# =============================================================================
# H — SOURCE-SPECIFIC EXPLOIT RECALL COMPARISON
#
# Episode-cluster paired bootstrap in each source.
# Do not treat 330 overlapping windows as independent samples.
# =============================================================================

exploit_ci_rows8 = []

for source in sorted(np.unique(source8)):
    eligible = np.unique(
        eid8[(source8 == source) & (family8 == 3) & (onset8 > 0)]
    )

    if not len(eligible):
        continue

    grouped = [
        np.flatnonzero(
            (eid8 == e) & (family8 == 3) & (onset8 > 0)
        )
        for e in eligible
    ]

    differences = []

    for _ in range(BOOTSTRAPS8):
        sampled = rng8.integers(
            0, len(grouped), size=len(grouped)
        )
        ix = np.concatenate([grouped[i] for i in sampled])

        t = np.mean(
            preds8["Competing-Risk TCN"]["pred_family"][ix] == 3
        )
        l = np.mean(
            preds8["Competing-Risk LSTM"]["pred_family"][ix] == 3
        )
        differences.append(float(t - l))

    target = EXPLOIT_AUDIT8[
        EXPLOIT_AUDIT8["Source"] == source
    ]

    tcn_recall = float(target.loc[
        target["Model"] == "Competing-Risk TCN", "Recall"
    ].iloc[0])

    lstm_recall = float(target.loc[
        target["Model"] == "Competing-Risk LSTM", "Recall"
    ].iloc[0])

    lo, hi = np.percentile(differences, [2.5, 97.5])

    exploit_ci_rows8.append({
        "Source": source,
        "True_Exploit_Windows": int(
            ((source8 == source) & (family8 == 3)
             & (onset8 > 0)).sum()
        ),
        "True_Exploit_Episodes": int(len(eligible)),
        "TCN_Recall": tcn_recall,
        "LSTM_Recall": lstm_recall,
        "TCN_Minus_LSTM": tcn_recall - lstm_recall,
        "CI_Lower_95": float(lo),
        "CI_Upper_95": float(hi)
    })

EXPLOIT_CI8 = pd.DataFrame(exploit_ci_rows8)

# =============================================================================
# I — FINAL AUDIT VERDICT
# =============================================================================

issues8 = []

if float(EXPLOIT_AUDIT8.loc[
    (EXPLOIT_AUDIT8["Model"] == "Competing-Risk TCN")
    & (EXPLOIT_AUDIT8["Source"] == "UNSW_NB15"),
    "Recall"
].iloc[0]) < 0.10:
    issues8.append(
        "CRITICAL: TCN EXPLOIT_APP recall below 10% on "
        "UNSW_NB15 validation windows."
    )

issues8.extend([
    "XGBoost and Stacked LSTM figures are historical reported "
    "validation metrics, not independently replayed in this cell.",
    "Family and ETA metrics condition on a true future event.",
    "Substantial window overlap makes naive window-level "
    "independence assumptions invalid.",
    "Episode bootstrap may not capture dependence between "
    "shared calibration groups.",
    "All reported performance is on a synthetic forecasting "
    "benchmark, not deployment traffic."
])

AUDIT_SUMMARY8 = {
    "experiment": "FINAL_VALIDATION_AUDIT",
    "dataset": "Synthetic_NAF_v2",
    "validation_windows": n8,
    "validation_episodes": int(len(episodes8)),
    "TCN_prediction_sha256": sha8(TCN_FILE8),
    "LSTM_prediction_sha256": sha8(LSTM_FILE8),
    "validation_sha256": sha8(VAL8),
    "metadata_sha256": sha8(META8),
    "compared_joint_models": list(preds8.keys()),
    "historical_baselines": [
        "XGBoost V2",
        "Stacked Horizon LSTM V2"
    ],
    "paired_bootstrap_replicates_requested": BOOTSTRAPS8,
    "official_test_accessed": False,
    "models_retrained": False,
    "calibration_performed": False,
    "hyperparameters_tuned": False,
    "final_model_chosen_for_test": False,
    "findings_requiring_attention": issues8,
    "created_utc": datetime.now(timezone.utc).isoformat()
}

# =============================================================================
# J — SAVE REPORTS (NEW DIRECTORY ONLY)
# =============================================================================

OUTPUT8.mkdir(parents=True, exist_ok=False)

tables8 = {
    "FOUR_MODEL_VALIDATION_COMPARISON.csv": FOUR_MODEL8,
    "COMPETING_RISK_RECOMPUTED_METRICS.csv": JOINT_RESULTS8,
    "SOURCE_SPECIFIC_METRICS.csv": SOURCE_RESULTS8,
    "SOURCE_FAMILY_PER_CLASS.csv": FAMILY_SOURCE8,
    "EXPLOIT_APP_SOURCE_AUDIT.csv": EXPLOIT_AUDIT8,
    "EXPLOIT_APP_PAIRED_SOURCE_CI.csv": EXPLOIT_CI8,
    "FIXED_THRESHOLD_ALERT_AUDIT.csv": ALERT_RESULTS8,
    "PAIRED_COMPETING_RISK_BOOTSTRAP.csv": PAIRED_RESULTS8
}

for name, table in tables8.items():
    table.to_csv(
        OUTPUT8 / name,
        index=False,
        float_format="%.12g"
    )

for (model, source), table in confusion_frames8.items():
    safe_model = (
        "TCN" if model == "Competing-Risk TCN" else "LSTM"
    )
    table.to_csv(
        OUTPUT8 / f"CONFUSION_{safe_model}_{source}.csv"
    )

write_json8(
    OUTPUT8 / "FINAL_VALIDATION_AUDIT_SUMMARY.json",
    AUDIT_SUMMARY8
)

hashes8 = {
    path.name: sha8(path)
    for path in sorted(OUTPUT8.iterdir())
    if path.is_file()
}
write_json8(OUTPUT8 / "ARTIFACT_SHA256SUMS.json", hashes8)

# =============================================================================
# K — PRINT RESULTS
# =============================================================================

print("\n" + "=" * 105)
print("CELL 8 — FINAL VALIDATION AUDIT COMPLETED")
print("=" * 105)

print("\n1. FOUR-MODEL VALIDATION SUMMARY")
print(FOUR_MODEL8.round(6).to_string(index=False))

print("\n2. COMPETING-RISK JOINT METRICS")
print(JOINT_RESULTS8.round(6).to_string(index=False))

print("\n3. PAIRED EPISODE-BOOTSTRAP COMPARISON")
print(PAIRED_RESULTS8.round(6).to_string(index=False))

print("\n4. EXPLOIT_APP SOURCE FAILURE ANALYSIS")
print(EXPLOIT_AUDIT8.round(6).to_string(index=False))

print("\n5. EXPLOIT_APP PAIRED SOURCE CONFIDENCE INTERVALS")
print(EXPLOIT_CI8.round(6).to_string(index=False))

print("\n6. SOURCE-SPECIFIC JOINT MODEL PERFORMANCE")
print(SOURCE_RESULTS8.round(6).to_string(index=False))

print("\n7. FIXED-THRESHOLD FALSE ALERT AUDIT")
print(ALERT_RESULTS8.round(6).to_string(index=False))

print("\n8. KEY AUDIT FINDINGS")
for number, issue in enumerate(issues8, start=1):
    print(f"{number}. {issue}")

print("\nSaved outputs:", OUTPUT8)
print("Models modified: NO")
print("Frozen dataset modified: NO")
print("Official TEST accessed: NO")
print("Final test model selected: NO")

print("=" * 105)
print("NEXT: REVIEW AUDIT AND LOCK THE FINAL EVALUATION PROTOCOL")
print("=" * 105)


In [ ]:

# =============================================================================
# NETWORK ATTACK FORECASTING — CELL 9
# PRE-TEST PROTOCOL FREEZE AND ARTIFACT INTEGRITY AUDIT
#
# READ-ONLY INPUTS:
#   Frozen release manifest and lock
#   TRAIN and VALIDATION checksums
#   Saved official validation evidence
#   Existing model checkpoints and preprocessing artifacts
#
# NEVER:
#   Loads, hashes or evaluates TEST
#   Trains, tunes or recalibrates models
#   Modifies existing checkpoints or dataset files
#
# Outputs:
#   Synthetic_NAF_v2/models/FINAL_TEST_PROTOCOL_FREEZE/
#
# STATUS:
#   LOCKED only if every required model artifact is verified.
#   Otherwise PRECHECK_INCOMPLETE; TEST remains embargoed.
# =============================================================================

import json
import hashlib
import os
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

# =============================================================================
# 9A — PATHS
# =============================================================================

ROOT9 = Path("/content/drive/MyDrive/network-attack-forecasting")
BASE9 = ROOT9 / "Synthetic_NAF_v2"
DATA9 = BASE9 / "release_candidate_corrected"
MODELS9 = BASE9 / "models"
FREEZE9 = BASE9 / "FINAL_RELEASE_FREEZE"

OUTPUT9 = MODELS9 / "FINAL_TEST_PROTOCOL_FREEZE"

MANIFEST9 = (
    FREEZE9 / "Synthetic_NAF_v2_RELEASE_MANIFEST.json"
)
DATA_LOCK9 = (
    FREEZE9 / "Synthetic_NAF_v2_FREEZE_LOCK.json"
)

TRAIN9 = DATA9 / "windows_W10_TRAIN.npz"
VALIDATION9 = DATA9 / "windows_W10_VALIDATION.npz"
METADATA9 = DATA9 / "episode_metadata_DO_NOT_USE_AS_PREDICTORS.csv"

AUDIT9 = (
    MODELS9
    / "FINAL_VALIDATION_AUDIT"
    / "FINAL_VALIDATION_AUDIT_SUMMARY.json"
)

FOUR_MODEL_REPORT9 = (
    MODELS9
    / "FINAL_VALIDATION_AUDIT"
    / "FOUR_MODEL_VALIDATION_COMPARISON.csv"
)

TCN_MODEL9 = (
    MODELS9
    / "TCN_V2_CompetingRisk"
    / "final_train_model"
    / "TCN_V2_CR_STATE.pt"
)

TCN_SCALER9 = (
    MODELS9
    / "TCN_V2_CompetingRisk"
    / "final_train_model"
    / "TCN_V2_CR_FULL_TRAIN_SCALER.json"
)

TCN_REGISTRY9 = (
    MODELS9
    / "TCN_V2_CompetingRisk"
    / "final_train_model"
    / "TCN_V2_CR_MODEL_REGISTRY.json"
)

TCN_PRED9 = (
    MODELS9
    / "TCN_V2_CompetingRisk"
    / "official_validation"
    / "JOINT_76_CLASS_VALIDATION_PREDICTIONS.npz"
)

LSTM_PRED9 = (
    MODELS9
    / "LSTM_V2_CompetingRisk"
    / "official_validation"
    / "JOINT_76_CLASS_VALIDATION_PREDICTIONS_SAFE.npz"
)

TCN_SELECTION9 = (
    MODELS9
    / "TCN_V2_CompetingRisk"
    / "train_cv_selection_GPU"
    / "SELECTED_TCN_CONFIGURATION.json"
)

EXPECTED_TCN_SHA9 = (
    "20b26d861dcab10357f7165877e0c5eb4121d511f4b6fca0d67f6eaef12c36dd"
)

# OPTIONAL EXPLICIT PATH OVERRIDES.
#
# Populate only with VERIFIED paths from your previous
# Colab training outputs. Do not guess paths.
#
# Keys:
# "XGBoost V2"
# "Stacked Horizon LSTM V2"
# "Competing-Risk LSTM V2"
#
# Each model needs:
#   "checkpoints": [Path(...), ...]
#   "preprocessors": [Path(...), ...]
#
# Leave empty to inspect/discover candidate artifacts.

MODEL_ARTIFACT_OVERRIDES9 = {}

if OUTPUT9.exists():
    raise RuntimeError(
        "Final protocol-freeze directory already exists. "
        "Refusing to overwrite:\n"
        f"{OUTPUT9}"
    )

# =============================================================================
# 9B — FUNCTIONS
# =============================================================================

def read_json9(path):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(path)
    return json.loads(path.read_text(encoding="utf-8"))


def hash_file9(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def canonical_hash9(obj):
    payload = json.dumps(
        obj,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False
    ).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()


def write_json9(path, data):
    Path(path).write_text(
        json.dumps(
            data,
            indent=2,
            sort_keys=True,
            allow_nan=False
        ),
        encoding="utf-8"
    )


def evidence_record9(path):
    path = Path(path).resolve(strict=True)
    if not path.is_file():
        raise ValueError(f"Not a file: {path}")

    # Every selected model artifact must be inside project models/.
    if MODELS9.resolve() not in path.parents:
        raise ValueError(
            "Model artifacts must be inside the project models directory: "
            f"{path}"
        )

    return {
        "path": str(path),
        "size_bytes": int(path.stat().st_size),
        "sha256": hash_file9(path)
    }


# =============================================================================
# 9C — RELEASE AND VALIDATION AUDIT INTEGRITY
# =============================================================================

required9 = [
    MANIFEST9, DATA_LOCK9, TRAIN9,
    VALIDATION9, METADATA9,
    AUDIT9, FOUR_MODEL_REPORT9,
    TCN_MODEL9, TCN_SCALER9,
    TCN_REGISTRY9, TCN_PRED9,
    LSTM_PRED9, TCN_SELECTION9
]

for p in required9:
    if not p.is_file():
        raise FileNotFoundError(p)

manifest9 = read_json9(MANIFEST9)
lock9 = read_json9(DATA_LOCK9)
audit9 = read_json9(AUDIT9)
tcn_registry9 = read_json9(TCN_REGISTRY9)
tcn_selection9 = read_json9(TCN_SELECTION9)

assert manifest9["release_version"] == "2.0.0"

assert canonical_hash9(manifest9) == (
    lock9["manifest_sha256_canonical"]
)

expected_hashes9 = manifest9["data_integrity"]["file_sha256"]

for filename, path in [
    ("windows_W10_TRAIN.npz", TRAIN9),
    ("windows_W10_VALIDATION.npz", VALIDATION9),
    (METADATA9.name, METADATA9)
]:
    assert hash_file9(path) == expected_hashes9[filename], (
        f"Frozen dataset checksum mismatch: {filename}"
    )

assert hash_file9(TCN_MODEL9) == EXPECTED_TCN_SHA9
assert tcn_registry9["model_sha256"] == EXPECTED_TCN_SHA9

assert hash_file9(TCN_SCALER9) == (
    tcn_registry9["scaler_sha256"]
)

assert tcn_selection9["selected_config_id"] == "C02"
assert tcn_selection9["selected_final_epochs"] == 21

assert audit9["validation_windows"] == 18839
assert audit9["validation_episodes"] == 300
assert audit9["official_test_accessed"] is False
assert audit9["models_retrained"] is False
assert audit9["hyperparameters_tuned"] is False

assert audit9["TCN_prediction_sha256"] == hash_file9(
    TCN_PRED9
)
assert audit9["LSTM_prediction_sha256"] == hash_file9(
    LSTM_PRED9
)

assert audit9["validation_sha256"] == hash_file9(
    VALIDATION9
)

assert audit9["metadata_sha256"] == hash_file9(
    METADATA9
)

comparison9 = pd.read_csv(FOUR_MODEL_REPORT9)

assert set(comparison9["Model"]) == {
    "XGBoost V2",
    "Stacked Horizon LSTM V2",
    "Competing-Risk LSTM",
    "Competing-Risk TCN"
}

print("=" * 105)
print("CELL 9 — PRE-TEST PROTOCOL FREEZE")
print("=" * 105)

print("✓ Dataset release verified")
print("✓ TRAIN checksum verified")
print("✓ VALIDATION checksum verified")
print("✓ Final validation audit verified")
print("✓ TCN C02 checkpoint and scaler verified")
print("✓ TCN/LSTM validation prediction evidence verified")
print("Official TEST: NOT ACCESSED")

# =============================================================================
# 9D — FIXED MODEL ROLES
# =============================================================================

MODEL_ROLES9 = {
    "XGBoost V2": {
        "role": "CLASSICAL_H5_H10_BASELINE",
        "outputs": ["P_H5", "P_H10"],
        "joint_family_eta": False,
        "status": "RETAIN_AS_BASELINE"
    },
    "Stacked Horizon LSTM V2": {
        "role": "PRIMARY_ATTACK_OCCURRENCE_FORECASTER",
        "outputs": ["P_H5", "P_H10", "P_H15"],
        "joint_family_eta": False,
        "status": "PRIMARY_FOR_H5_H10"
    },
    "Competing-Risk LSTM V2": {
        "role": "PRIMARY_JOINT_FAMILY_AND_ONSET_FORECASTER",
        "outputs": [
            "P_H5", "P_H10", "P_H15",
            "FAMILY_PROBABILITIES", "ONSET_ETA"
        ],
        "joint_family_eta": True,
        "status": "PRIMARY_JOINT_MODEL"
    },
    "Competing-Risk TCN V2": {
        "role": "COMPETING_RISK_ARCHITECTURAL_ALTERNATIVE",
        "outputs": [
            "P_H5", "P_H10", "P_H15",
            "FAMILY_PROBABILITIES", "ONSET_ETA"
        ],
        "joint_family_eta": True,
        "status": "RETAIN_AS_COMPARATOR"
    }
}

# =============================================================================
# 9E — REGISTER VERIFIED ARTIFACTS
# =============================================================================

artifact_records9 = {
    "Competing-Risk TCN V2": {
        "checkpoints": [evidence_record9(TCN_MODEL9)],
        "preprocessors": [evidence_record9(TCN_SCALER9)],
        "validation_predictions": [evidence_record9(TCN_PRED9)],
        "verification": "CHECKPOINT_AND_PREPROCESSOR_VERIFIED"
    }
}

# The other checkpoints need explicit evidence.
# Discover possible files only. Discovery DOES NOT authenticate
# a filename as the correct trained model.
SEARCH_TERMS9 = {
    "XGBoost V2": [
        "xgboost", "xgb"
    ],
    "Stacked Horizon LSTM V2": [
        "stacked", "horizon_lstm", "horizon-lstm"
    ],
    "Competing-Risk LSTM V2": [
        "competingrisklstm", "competing_risk_lstm",
        "lstm_v2_competingrisk", "lstm_v2_competing_risk"
    ]
}

CHECKPOINT_SUFFIXES9 = {
    ".pt", ".pth", ".pkl", ".joblib",
    ".json", ".ubj", ".bin", ".model"
}

PREPROCESSOR_TOKENS9 = {
    "scaler", "preprocess", "normaliz",
    "transformer", "encoder"
}

def discover_candidates9(model_name):
    tokens = SEARCH_TERMS9[model_name]

    candidates = []

    # Search only existing models directory.
    for path in MODELS9.rglob("*"):
        if not path.is_file():
            continue

        if path.suffix.lower() not in CHECKPOINT_SUFFIXES9:
            continue

        text = str(path.relative_to(MODELS9)).lower()

        if not any(token in text for token in tokens):
            continue

        # Exclude validation prediction files and reports.
        if "official_validation" in text:
            continue
        if "train_cv_selection_gpu" in text:
            continue

        candidates.append(path)

    candidates = sorted(set(candidates))

    return {
        "checkpoint_candidates": [
            str(p) for p in candidates
            if not any(t in p.name.lower()
                       for t in PREPROCESSOR_TOKENS9)
            and "report" not in p.name.lower()
            and "record" not in p.name.lower()
            and "manifest" not in p.name.lower()
        ],
        "preprocessor_candidates": [
            str(p) for p in candidates
            if any(t in p.name.lower()
                   for t in PREPROCESSOR_TOKENS9)
        ]
    }


CANDIDATE_INVENTORY9 = {}
UNRESOLVED9 = []

for model_name in SEARCH_TERMS9:

    discovered = discover_candidates9(model_name)
    CANDIDATE_INVENTORY9[model_name] = discovered

    override = MODEL_ARTIFACT_OVERRIDES9.get(model_name)

    if override is None:
        UNRESOLVED9.append({
            "model": model_name,
            "reason": (
                "Exact checkpoint and preprocessing paths not "
                "independently identified and approved."
            )
        })
        continue

    checkpoints = override.get("checkpoints", [])
    preprocessors = override.get("preprocessors", [])

    if not checkpoints or not preprocessors:
        UNRESOLVED9.append({
            "model": model_name,
            "reason": (
                "Explicit artifact override lacks checkpoint "
                "or preprocessing evidence."
            )
        })
        continue

    artifact_records9[model_name] = {
        "checkpoints": [
            evidence_record9(p) for p in checkpoints
        ],
        "preprocessors": [
            evidence_record9(p) for p in preprocessors
        ],
        "verification": (
            "EXPLICIT_PATHS_HASHED; "
            "MODEL_SEMANTICS_REQUIRE_RELOAD_VERIFICATION"
        )
    }

# Saved LSTM validation prediction evidence is verified even if
# its training checkpoint has not yet been located.
artifact_records9["Competing-Risk LSTM V2"] = {
    **artifact_records9.get("Competing-Risk LSTM V2", {}),
    "validation_predictions": [
        evidence_record9(LSTM_PRED9)
    ]
}

# =============================================================================
# 9F — PRE-REGISTER FINAL TEST EVALUATION RULES
# =============================================================================

PROTOCOL_RULES9 = {
    "benchmark": "Synthetic_NAF_v2",
    "release_version": "2.0.0",
    "split": "OFFICIAL_TEST",
    "test_data_embargo": True,
    "test_access_before_lock": False,
    "model_roles": MODEL_ROLES9,
    "prediction_tasks": {
        "attack_occurrence": {
            "horizons_minutes": [5, 10, 15],
            "primary_metric": "Average_Precision",
            "secondary_metrics": [
                "ROC_AUC", "Brier", "ECE_10"
            ],
            "comparison_rule": (
                "Compare only horizons emitted by each model."
            )
        },
        "competing_risks": {
            "output_classes": 76,
            "no_event_class": 0,
            "family_names": [
                "BRUTE_AUTH",
                "DOS_DDOS",
                "EXPLOIT_APP",
                "FUZZING",
                "RECON_SCAN"
            ],
            "encoding": (
                "class = 5 * (onset_minute - 1) + family_id"
            ),
            "primary_metric": "JOINT_NLL",
            "secondary_metrics": [
                "Conditional_Family_Accuracy",
                "Conditional_Family_Macro_F1",
                "Expected_ETA_MAE"
            ],
            "family_eta_condition": (
                "TRUE_ONSET_WITHIN_15_MINUTES"
            )
        }
    },
    "fixed_thresholds": [0.10, 0.20, 0.50],
    "threshold_selection_after_test": False,
    "subgroups": [
        "CIC_IDS2017",
        "UNSW_NB15"
    ],
    "mandatory_family_analysis": [
        "Precision",
        "Recall",
        "F1",
        "Confusion_Matrix",
        "EXPLOIT_APP_By_Source"
    ],
    "benign_alert_metrics": [
        "Window_False_Positive_Rate",
        "Wholly_Benign_Episodes_With_Alert"
    ],
    "uncertainty": {
        "method": "Paired_Episode_Cluster_Bootstrap",
        "replicates": 1000,
        "seed": 20261024,
        "confidence_interval": "Percentile_95",
        "limitations": (
            "Episode bootstrap does not fully capture "
            "calibration-group dependence."
        )
    },
    "prohibited_after_test": [
        "Retraining",
        "Hyperparameter_Changes",
        "Validation_Reselection",
        "Threshold_Tuning",
        "Calibration_Fitting",
        "Changing_Primary_Metrics",
        "Dropping_Unfavorable_Subgroups"
    ],
    "data_interpretation": {
        "synthetic_benchmark": True,
        "live_network_claims_permitted": False,
        "overlapping_windows_independent": False
    },
    "model_selection_policy": (
        "Fixed pre-test roles; no post-test model reselection."
    )
}

# =============================================================================
# 9G — CREATE IMMUTABLE CONTENT SNAPSHOT
# =============================================================================

fully_resolved9 = len(UNRESOLVED9) == 0

# Explicit paths are not by themselves enough to prove that
# the checkpoint is the model that produced historical results.
# Require a later reload/semantic verification for 3 models.
SEMANTIC_VERIFICATION_REQUIRED9 = [
    "XGBoost V2",
    "Stacked Horizon LSTM V2",
    "Competing-Risk LSTM V2"
]

LOCK_STATUS9 = (
    "PRECHECK_INCOMPLETE"
    if not fully_resolved9
    else "ARTIFACT_PATHS_HASHED_SEMANTIC_VERIFICATION_PENDING"
)

FREEZE_RECORD9 = {
    "protocol_name": "NAF_V2_FINAL_TEST_PROTOCOL",
    "status": LOCK_STATUS9,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "dataset": {
        "release_version": "2.0.0",
        "manifest_sha256_canonical": canonical_hash9(manifest9),
        "train_sha256": hash_file9(TRAIN9),
        "validation_sha256": hash_file9(VALIDATION9),
        "metadata_sha256": hash_file9(METADATA9)
    },
    "validation_evidence": {
        "audit_summary_sha256": hash_file9(AUDIT9),
        "four_model_report_sha256": hash_file9(FOUR_MODEL_REPORT9),
        "tcn_validation_predictions_sha256": hash_file9(TCN_PRED9),
        "lstm_validation_predictions_sha256": hash_file9(LSTM_PRED9)
    },
    "registered_model_artifacts": artifact_records9,
    "protocol_rules": PROTOCOL_RULES9,
    "unresolved_models": UNRESOLVED9,
    "semantic_verification_required": (
        SEMANTIC_VERIFICATION_REQUIRED9
    ),
    "ready_to_run_test": False,
    "official_test_accessed": False,
    "models_modified": False
}

CONTENT_HASH9 = canonical_hash9(FREEZE_RECORD9)

OUTPUT9.mkdir(parents=True, exist_ok=False)

write_json9(
    OUTPUT9 / "PRE_TEST_PROTOCOL_RECORD.json",
    FREEZE_RECORD9
)

write_json9(
    OUTPUT9 / "MODEL_ARTIFACT_CANDIDATES.json",
    CANDIDATE_INVENTORY9
)

write_json9(
    OUTPUT9 / "PRE_TEST_LOCK_STATUS.json",
    {
        "status": LOCK_STATUS9,
        "protocol_record_sha256_canonical": CONTENT_HASH9,
        "ready_to_run_test": False,
        "official_test_accessed": False,
        "unresolved_model_count": len(UNRESOLVED9)
    }
)

# Verify the newly written protocol hasn't changed.
assert canonical_hash9(
    read_json9(
        OUTPUT9 / "PRE_TEST_PROTOCOL_RECORD.json"
    )
) == CONTENT_HASH9

# =============================================================================
# 9H — FINAL REPORT
# =============================================================================

print("\n" + "=" * 105)
print("PRE-TEST PROTOCOL AUDIT COMPLETE")
print("=" * 105)

print("\nProtocol status:", LOCK_STATUS9)
print("Protocol content SHA256:", CONTENT_HASH9)

print("\nFIXED MODEL ROLES")
for model, info in MODEL_ROLES9.items():
    print(f"  {model}: {info['role']}")

print("\nVERIFIED TCN ARTIFACT")
print("  Model SHA256:", EXPECTED_TCN_SHA9)
print("  Configuration: C02")
print("  Epochs: 21")

print("\nUNRESOLVED MODEL ARTIFACTS")
if UNRESOLVED9:
    for item in UNRESOLVED9:
        print(f"  - {item['model']}: {item['reason']}")
else:
    print("  None; semantic reload checks still required.")

print("\nMODEL ARTIFACT CANDIDATES")
for model, inventory in CANDIDATE_INVENTORY9.items():
    print(f"\n  {model}")
    print("    Candidate checkpoints:")
    for path in inventory["checkpoint_candidates"][:15]:
        print("     ", path)
    print("    Candidate preprocessors:")
    for path in inventory["preprocessor_candidates"][:15]:
        print("     ", path)

print("\nSaved to:", OUTPUT9)

print("\nOfficial TEST accessed: NO")
print("Model training: NO")
print("Model tuning: NO")
print("Dataset modified: NO")
print("Ready to run TEST:", False)

print("=" * 105)
print(
    "NEXT: VERIFY REMAINING MODEL CHECKPOINTS AND PREPROCESSORS "
    "BEFORE SEALING THE FINAL TEST LOCK."
)
print("=" * 105)


In [ ]:

from pathlib import Path
import json
import hashlib

root = Path("/content/drive/MyDrive/network-attack-forecasting")
folder = root / "Synthetic_NAF_v2/models/FINAL_TEST_PROTOCOL_FREEZE"

def file_sha256(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

print("=" * 90)
print("CELL 9 — EXISTING PROTOCOL DIRECTORY DIAGNOSTIC")
print("=" * 90)

if not folder.exists():
    print("STATUS: Directory does not exist. Original Cell 9 can run.")
else:
    print("Directory:", folder)
    print("\nEXISTING FILES:")

    for path in sorted(folder.iterdir()):
        if path.is_file():
            print(f"  {path.name} ({path.stat().st_size:,} bytes)")
        else:
            print(f"  {path.name}/")

    record_path = folder / "PRE_TEST_PROTOCOL_RECORD.json"
    status_path = folder / "PRE_TEST_LOCK_STATUS.json"

    if record_path.is_file() and status_path.is_file():
        record = json.loads(record_path.read_text())
        status = json.loads(status_path.read_text())

        canonical = json.dumps(
            record,
            sort_keys=True,
            separators=(",", ":"),
            ensure_ascii=False
        ).encode("utf-8")

        calculated_hash = hashlib.sha256(canonical).hexdigest()
        stored_hash = status.get("protocol_record_sha256_canonical")

        print("\nPROTOCOL DETAILS:")
        print("Recorded status:", record.get("status"))
        print("Lock status:", status.get("status"))
        print("Hash verified:", calculated_hash == stored_hash)
        print("Ready for TEST:", status.get("ready_to_run_test"))
        print("Official TEST accessed:", status.get("official_test_accessed"))

        print("\nUNRESOLVED MODELS:")
        for item in record.get("unresolved_models", []):
            print(" -", item.get("model"), "|", item.get("reason"))

        print("\nPROTOCOL HASH:")
        print(calculated_hash)

        if calculated_hash == stored_hash:
            print("\n✓ Existing protocol record is internally consistent.")
            print("No need to rerun the original Cell 9.")
        else:
            print("\nWARNING: Protocol hash mismatch. Do not proceed to TEST.")

    else:
        print("\nProtocol files are missing or incomplete.")
        print("Do not delete the directory until its contents are reviewed.")

print("=" * 90)


In [ ]:

# =============================================================================
# CELL 9A — INSPECT UNRESOLVED MODEL ARTIFACTS
# READ-ONLY: no training, no file modifications, no TEST access
# =============================================================================

import json
import hashlib
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/network-attack-forecasting")

FREEZE_DIR = (
    ROOT
    / "Synthetic_NAF_v2/models/FINAL_TEST_PROTOCOL_FREEZE"
)

CANDIDATES_FILE = FREEZE_DIR / "MODEL_ARTIFACT_CANDIDATES.json"
STATUS_FILE = FREEZE_DIR / "PRE_TEST_LOCK_STATUS.json"

def sha256_file(path):
    h = hashlib.sha256()

    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)

    return h.hexdigest()

if not CANDIDATES_FILE.is_file():
    raise FileNotFoundError(CANDIDATES_FILE)

with open(CANDIDATES_FILE, "r") as f:
    inventory = json.load(f)

with open(STATUS_FILE, "r") as f:
    status = json.load(f)

assert status["ready_to_run_test"] is False

print("=" * 105)
print("CELL 9A — UNRESOLVED MODEL ARTIFACT INVENTORY")
print("=" * 105)

for model_name, candidates in inventory.items():

    print(f"\n{'=' * 90}")
    print("MODEL:", model_name)
    print(f"{'=' * 90}")

    for category, paths in candidates.items():

        print(f"\n{category.upper()}: {len(paths)} candidate(s)")

        if not paths:
            print("  NONE FOUND")

        for path_string in paths:

            path = Path(path_string)

            print("\n  File:", path.name)
            print("  Full path:", path)

            if not path.is_file():
                print("  STATUS: MISSING")
                continue

            print(
                "  Size:",
                f"{path.stat().st_size / 1024 / 1024:.3f} MB"
            )
            print("  SHA256:", sha256_file(path))

print("\n" + "=" * 105)
print("INVENTORY COMPLETE")
print("Files modified: NONE")
print("Official TEST accessed: NO")
print("Ready to run TEST: NO")
print("=" * 105)


In [ ]:

# =============================================================================
# CELL 9B — MODEL ARTIFACT DISCOVERY AND REGISTRY AUDIT
# READ-ONLY. NO TEST ACCESS. NO MODIFICATIONS.
# =============================================================================

import json
import hashlib
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/network-attack-forecasting")
MODELS = ROOT / "Synthetic_NAF_v2/models"

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def show_json(path, max_chars=12000):
    print(f"\nFILE: {path}")
    data = json.loads(path.read_text(encoding="utf-8"))
    rendered = json.dumps(data, indent=2, ensure_ascii=False)
    print(rendered[:max_chars])
    if len(rendered) > max_chars:
        print("... [display truncated]")
    return data

print("=" * 105)
print("CELL 9B — BROAD MODEL ARTIFACT DISCOVERY")
print("=" * 105)

# -------------------------------------------------------------------------
# A. Discover top-level model directories.
# -------------------------------------------------------------------------

print("\n1. MODEL DIRECTORIES")

for path in sorted(MODELS.iterdir()):
    if path.is_dir():
        print(" ", path.name)

# -------------------------------------------------------------------------
# B. Inspect known registries.
# -------------------------------------------------------------------------

registry_paths = {
    "Competing-Risk LSTM": (
        MODELS
        / "LSTM_V2_CompetingRisk/final_train_model"
        / "LSTM_V2_CR_MODEL_REGISTRY.json"
    ),
    "XGBoost": (
        MODELS
        / "XGBoost_V2_FLAT140/final_train_models"
        / "XGBOOST_V2_FINAL_TRAIN_REGISTRY.json"
    )
}

registries = {}

print("\n" + "=" * 105)
print("2. TRAINING REGISTRIES")
print("=" * 105)

for model_name, path in registry_paths.items():
    print("\nMODEL:", model_name)
    if path.is_file():
        registries[model_name] = show_json(path)
        print("Registry SHA256:", sha256(path))
    else:
        print("Registry not found:", path)

# -------------------------------------------------------------------------
# C. Search all model subdirectories for Stacked/Horizon LSTM.
# -------------------------------------------------------------------------

print("\n" + "=" * 105)
print("3. BROAD LSTM/HORIZON ARTIFACT DISCOVERY")
print("=" * 105)

tokens = (
    "lstm", "stack", "horizon",
    "multihead", "multi_head", "sequence"
)

extensions = {
    ".pt", ".pth", ".ckpt",
    ".pkl", ".joblib", ".safetensors",
    ".json", ".npz"
}

matches = []

for path in MODELS.rglob("*"):
    if not path.is_file():
        continue

    relative = str(path.relative_to(MODELS)).lower()

    if not any(word in relative for word in tokens):
        continue

    if path.suffix.lower() not in extensions:
        continue

    # Exclude huge sets of CV fold records and ordinary
    # validation predictions. We need final train artifacts.
    if "train_cv_selection" in relative:
        continue
    if "official_validation" in relative:
        continue
    if "bootstrap" in relative:
        continue

    matches.append(path)

matches = sorted(set(matches))

print("Candidate files:", len(matches))

for path in matches:
    print(
        f"{path.stat().st_size / 1024 / 1024:9.3f} MB | "
        f"{path.relative_to(MODELS)}"
    )

# -------------------------------------------------------------------------
# D. Verify known checkpoint files are still intact.
# -------------------------------------------------------------------------

print("\n" + "=" * 105)
print("4. KNOWN CHECKPOINT HASHES")
print("=" * 105)

known_files = {
    "Competing-Risk LSTM checkpoint": (
        MODELS
        / "LSTM_V2_CompetingRisk/final_train_model"
        / "LSTM_V2_CR_STATE.pt"
    ),
    "Competing-Risk LSTM scaler": (
        MODELS
        / "LSTM_V2_CompetingRisk/final_train_model"
        / "LSTM_V2_CR_FULL_TRAIN_SCALER.json"
    ),
    "XGBoost H5": (
        MODELS
        / "XGBoost_V2_FLAT140/final_train_models"
        / "XGBoost_V2_FLAT140_H5.json"
    ),
    "XGBoost H10": (
        MODELS
        / "XGBoost_V2_FLAT140/final_train_models"
        / "XGBoost_V2_FLAT140_H10.json"
    )
}

for label, path in known_files.items():
    if not path.is_file():
        print(f"MISSING: {label}")
        continue

    print(f"\n{label}")
    print(" Path:", path)
    print(" SHA256:", sha256(path))

# -------------------------------------------------------------------------
# E. Read only the metadata fields from XGBoost JSON models.
# This does not load models for inference.
# -------------------------------------------------------------------------

print("\n" + "=" * 105)
print("5. XGBOOST MODEL FORMAT VERIFICATION")
print("=" * 105)

for label in ["XGBoost H5", "XGBoost H10"]:
    path = known_files[label]

    if path.is_file():
        with path.open("r", encoding="utf-8") as f:
            data = json.load(f)

        print(f"\n{label}:")
        print(" Top-level keys:", list(data.keys()))
        print(
            " Has learner section:",
            "learner" in data
        )

        if "learner" in data:
            learner = data["learner"]
            print(
                " Objective:",
                learner.get("objective", {}).get("name")
            )
            print(
                " Number of features:",
                learner.get("learner_model_param", {}).get("num_feature")
            )

# -------------------------------------------------------------------------
# F. Lock status remains unchanged.
# -------------------------------------------------------------------------

lock_status = (
    MODELS
    / "FINAL_TEST_PROTOCOL_FREEZE"
    / "PRE_TEST_LOCK_STATUS.json"
)

if lock_status.is_file():
    status = json.loads(
        lock_status.read_text(encoding="utf-8")
    )
    print("\nCurrent protocol status:", status.get("status"))
    print("Ready for TEST:", status.get("ready_to_run_test"))

    assert status["ready_to_run_test"] is False

print("\n" + "=" * 105)
print("CELL 9B DISCOVERY COMPLETE")
print("Files modified: NONE")
print("Checkpoints modified: NONE")
print("Official TEST accessed: NO")
print("=" * 105)


In [ ]:

# =============================================================================
# CELL 9C — FINAL MODEL ARTIFACT VERIFICATION
# READ ONLY | NO TRAINING | NO VALIDATION RESELECTION | NO TEST
# =============================================================================

import json
import hashlib
from pathlib import Path
import numpy as np
import torch

ROOT = Path("/content/drive/MyDrive/network-attack-forecasting")
BASE = ROOT / "Synthetic_NAF_v2"
MODELS = BASE / "models"

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def verify(path, expected):
    if not path.is_file():
        raise FileNotFoundError(path)
    actual = sha256(path)
    if actual != expected:
        raise RuntimeError(f"SHA256 mismatch: {path}")
    return actual

dirs = {
    "XGBoost": MODELS / "XGBoost_V2_FLAT140/final_train_models",
    "Horizon_LSTM": MODELS / "LSTM_V2_Horizon/final_train_model",
    "CR_LSTM": MODELS / "LSTM_V2_CompetingRisk/final_train_model",
    "CR_TCN": MODELS / "TCN_V2_CompetingRisk/final_train_model"
}

registry_names = {
    "XGBoost": "XGBOOST_V2_FINAL_TRAIN_REGISTRY.json",
    "Horizon_LSTM": "LSTM_V2_FINAL_MODEL_REGISTRY.json",
    "CR_LSTM": "LSTM_V2_CR_MODEL_REGISTRY.json",
    "CR_TCN": "TCN_V2_CR_MODEL_REGISTRY.json"
}

print("=" * 100)
print("CELL 9C — FINAL MODEL ARTIFACT VERIFICATION")
print("=" * 100)

registries = {}

for name, folder in dirs.items():
    path = folder / registry_names[name]
    if not path.is_file():
        raise FileNotFoundError(path)

    registries[name] = read_json(path)

    print(f"\n{'=' * 85}")
    print("MODEL:", name)
    print("REGISTRY SHA256:", sha256(path))
    print("REGISTRY CONTENT:")
    print(json.dumps(registries[name], indent=2))

# =============================================================================
# A — VERIFY XGBOOST FINAL MODELS
# =============================================================================

print("\n" + "=" * 100)
print("XGBOOST — FINAL H5/H10 VERIFICATION")
print("=" * 100)

xgb_registry = registries["XGBoost"]

assert xgb_registry["representation"] == "FLAT140_LOG1P"
assert xgb_registry["training_status"] == "FINAL_FULL_TRAIN_COMPLETED"

for horizon in ["H5", "H10"]:
    item = xgb_registry["models"][horizon]
    path = dirs["XGBoost"] / item["model_file"]

    actual = verify(path, item["model_sha256"])

    model_json = read_json(path)
    learner = model_json["learner"]

    assert learner["objective"]["name"] == "binary:logistic"
    assert int(
        learner["learner_model_param"]["num_feature"]
    ) == 140

    assert item["official_test_used"] is False

    print(f"\n{horizon}")
    print("Checkpoint:", path.name)
    print("SHA256:", actual)
    print("Features:", 140)
    print("Objective: binary:logistic")
    print("Boosting rounds:", item["boosting_rounds"])
    print("Representation:", xgb_registry["representation"])
    print("Integrity: VERIFIED")

# =============================================================================
# B — VERIFY ALL NEURAL CHECKPOINTS AND SAVED SCALERS
# =============================================================================

for name in ["Horizon_LSTM", "CR_LSTM", "CR_TCN"]:

    print("\n" + "=" * 100)
    print(name, "— CHECKPOINT AND SCALER")
    print("=" * 100)

    folder = dirs[name]
    registry = registries[name]

    model_path = folder / registry["model_file"]
    scaler_path = folder / registry["scaler_file"]

    print("Checkpoint SHA256:", verify(
        model_path, registry["model_sha256"]
    ))

    print("Scaler SHA256:", verify(
        scaler_path, registry["scaler_sha256"]
    ))

    for filename_key, hash_key in [
        ("history_file", "history_sha256"),
        ("training_record_file", "training_record_sha256")
    ]:
        if filename_key in registry and hash_key in registry:
            print(
                filename_key,
                "VERIFIED:",
                verify(
                    folder / registry[filename_key],
                    registry[hash_key]
                )
            )

    weights = torch.load(
        model_path,
        map_location="cpu",
        weights_only=True
    )

    if not isinstance(weights, dict):
        raise TypeError(
            f"Unexpected checkpoint object for {name}"
        )

    # Some checkpoints wrap the state_dict inside a dictionary.
    if "state_dict" in weights:
        state = weights["state_dict"]
    elif "model_state_dict" in weights:
        state = weights["model_state_dict"]
    else:
        state = weights

    if not isinstance(state, dict):
        raise TypeError(f"Invalid state dictionary: {name}")

    tensors = {
        k: v for k, v in state.items()
        if torch.is_tensor(v)
    }

    if not tensors:
        raise RuntimeError(f"No saved model tensors: {name}")

    nonfinite = [
        k for k, v in tensors.items()
        if v.is_floating_point()
        and not torch.isfinite(v).all()
    ]

    if nonfinite:
        raise RuntimeError(
            f"Nonfinite weights in {name}: {nonfinite}"
        )

    scaler = read_json(scaler_path)

    print("Checkpoint tensor count:", len(tensors))
    print("Total stored tensor elements:", sum(
        v.numel() for v in tensors.values()
    ))

    print("\nFirst 12 state dictionary entries:")
    for key, value in list(tensors.items())[:12]:
        print(" ", key, tuple(value.shape))

    print("\nScaler keys:")
    print(list(scaler.keys()))

    print("\nScaler state:")
    print(json.dumps(scaler, indent=2)[:5000])

    train_record = registry.get("training_record_file")

    if train_record:
        record = read_json(folder / train_record)

        print("\nTraining record architecture/configuration:")
        for key, value in record.items():
            if any(token in key.lower() for token in [
                "architecture", "config", "epoch",
                "feature", "preprocess", "seed", "loss"
            ]):
                rendered = json.dumps(
                    value, ensure_ascii=False, default=str
                )
                print(f" {key}: {rendered[:1200]}")

    print("\nIntegrity: VERIFIED")
    print("Architecture reload: NOT YET TESTED")

# =============================================================================
# C — DISCOVER SAVED VALIDATION ARTIFACTS FOR REPLAY
# =============================================================================

print("\n" + "=" * 100)
print("VALIDATION EVIDENCE INVENTORY")
print("=" * 100)

for name, parent in [
    ("XGBoost", MODELS / "XGBoost_V2_FLAT140"),
    ("Horizon_LSTM", MODELS / "LSTM_V2_Horizon"),
    ("CR_LSTM", MODELS / "LSTM_V2_CompetingRisk"),
    ("CR_TCN", MODELS / "TCN_V2_CompetingRisk")
]:
    print(f"\n{name}:")

    files = sorted(
        p for p in parent.rglob("*")
        if p.is_file()
        and (
            "validation" in str(p).lower()
            or "prediction" in p.name.lower()
        )
    )

    for path in files[:30]:
        print(" ", path.relative_to(parent))

    if len(files) > 30:
        print(f"  ... {len(files) - 30} additional files")

# =============================================================================
# D — ENSURE THE EXISTING PRE-TEST LOCK HAS NOT CHANGED
# =============================================================================

status_path = (
    MODELS
    / "FINAL_TEST_PROTOCOL_FREEZE"
    / "PRE_TEST_LOCK_STATUS.json"
)

status = read_json(status_path)

assert status["ready_to_run_test"] is False
assert status["official_test_accessed"] is False

print("\n" + "=" * 100)
print("CELL 9C COMPLETE")
print("=" * 100)
print("Checkpoint hashes: VERIFIED")
print("Neural tensor integrity: VERIFIED")
print("Full architecture/prediction replay: STILL REQUIRED")
print("Existing protocol status:", status["status"])
print("Ready for TEST: NO")
print("Models modified: NO")
print("Frozen dataset modified: NO")
print("Official TEST accessed: NO")
print("=" * 100)


In [ ]:

# =============================================================================
# CELL 9D — INFERENCE REPLAY PREFLIGHT
# READ ONLY | VALIDATION ONLY | NO TEST ACCESS
# =============================================================================

import inspect
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch

ROOT9D = Path("/content/drive/MyDrive/network-attack-forecasting")
BASE9D = ROOT9D / "Synthetic_NAF_v2"
MODELS9D = BASE9D / "models"

SOURCES9D = {
    "XGBoost": (
        MODELS9D / "XGBoost_V2_FLAT140"
        / "official_validation/VALIDATION_PREDICTIONS.parquet"
    ),
    "Horizon_LSTM": (
        MODELS9D / "LSTM_V2_Horizon"
        / "official_validation/VALIDATION_PREDICTIONS.csv"
    ),
    "CR_LSTM": (
        MODELS9D / "LSTM_V2_CompetingRisk"
        / "official_validation/JOINT_76_CLASS_VALIDATION_PREDICTIONS_SAFE.npz"
    ),
    "CR_TCN": (
        MODELS9D / "TCN_V2_CompetingRisk"
        / "official_validation/JOINT_76_CLASS_VALIDATION_PREDICTIONS.npz"
    )
}

RECORDS9D = {
    "Horizon_LSTM": (
        MODELS9D / "LSTM_V2_Horizon/final_train_model"
        / "LSTM_V2_FINAL_TRAIN_RECORD.json"
    ),
    "CR_LSTM": (
        MODELS9D / "LSTM_V2_CompetingRisk/final_train_model"
        / "LSTM_V2_CR_FINAL_TRAIN_RECORD.json"
    ),
    "CR_TCN": (
        MODELS9D / "TCN_V2_CompetingRisk/final_train_model"
        / "TCN_V2_CR_FINAL_TRAIN_RECORD.json"
    )
}

print("=" * 100)
print("CELL 9D — INFERENCE REPLAY PREFLIGHT")
print("=" * 100)

print("\n1. SAVED VALIDATION PREDICTION SCHEMAS")

for name, path in SOURCES9D.items():
    print(f"\n{name}")
    print("Path:", path)

    if not path.is_file():
        print("MISSING")
        continue

    if path.suffix == ".parquet":
        df = pd.read_parquet(path)
        print("Shape:", df.shape)
        print("Columns and dtypes:")
        for col in df.columns:
            print(f"  {col}: {df[col].dtype}")

    elif path.suffix == ".csv":
        df = pd.read_csv(path, nrows=10)
        print("Columns and dtypes:")
        for col in df.columns:
            print(f"  {col}: {df[col].dtype}")

    elif path.suffix == ".npz":
        with np.load(path, allow_pickle=False) as z:
            for key in z.files:
                arr = z[key]
                print(
                    f"  {key}: shape={arr.shape}, dtype={arr.dtype}"
                )

print("\n" + "=" * 100)
print("2. FROZEN NEURAL MODEL CONFIGURATIONS")
print("=" * 100)

for name, path in RECORDS9D.items():
    if not path.is_file():
        raise FileNotFoundError(path)

    record = json.loads(path.read_text(encoding="utf-8"))

    print(f"\n{name}")
    print("Architecture:", json.dumps(
        record.get("model_architecture"), indent=2
    ))
    print("Preprocessing:", record.get("preprocessing"))
    print("Training epochs:", record.get("training_epochs"))
    print("Selected configuration:", record.get("selected_configuration_id"))

print("\n" + "=" * 100)
print("3. AVAILABLE PYTORCH MODEL CLASSES")
print("=" * 100)

candidate_classes = []

for name, obj in list(globals().items()):
    if not inspect.isclass(obj):
        continue

    try:
        if obj is torch.nn.Module:
            continue

        if not issubclass(obj, torch.nn.Module):
            continue

        text = name.lower()
        if not any(
            token in text for token in
            ("lstm", "tcn", "horizon", "competing", "risk")
        ):
            continue

        candidate_classes.append((name, obj))
    except TypeError:
        continue

if not candidate_classes:
    print("No custom model classes found in notebook globals.")
else:
    for name, cls in sorted(candidate_classes):
        print(f"\nClass: {name}")

        try:
            print("Constructor:", inspect.signature(cls))
        except (TypeError, ValueError):
            print("Constructor signature unavailable.")

        try:
            print("Forward:", inspect.signature(cls.forward))
        except (TypeError, ValueError):
            print("Forward signature unavailable.")

print("\n" + "=" * 100)
print("4. EXISTING PROTOCOL LOCK")
print("=" * 100)

status_path = (
    MODELS9D / "FINAL_TEST_PROTOCOL_FREEZE"
    / "PRE_TEST_LOCK_STATUS.json"
)

status = json.loads(status_path.read_text(encoding="utf-8"))

print("Status:", status["status"])
print("Ready for TEST:", status["ready_to_run_test"])

assert status["ready_to_run_test"] is False
assert status["official_test_accessed"] is False

print("\nPrefight completed without changing files.")
print("Official TEST: NOT ACCESSED")
print("=" * 100)


In [ ]:

# =============================================================================
# CELL 9E — VALIDATION INFERENCE REPLAY
#
# No training, calibration, tuning, TEST access, or file modifications.
# Replays XGBoost, TCN and competing-risk LSTM.
# Horizon LSTM requires original class implementation.
# =============================================================================

import json
import hashlib
import inspect
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import xgboost as xgb
from sklearn.metrics import average_precision_score

ROOT_E = Path("/content/drive/MyDrive/network-attack-forecasting")
BASE_E = ROOT_E / "Synthetic_NAF_v2"
MODELS_E = BASE_E / "models"
DATA_E = BASE_E / "release_candidate_corrected"

VAL_E = DATA_E / "windows_W10_VALIDATION.npz"
LOCK_E = (
    MODELS_E
    / "FINAL_TEST_PROTOCOL_FREEZE/PRE_TEST_LOCK_STATUS.json"
)

def load_json_e(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def digest_e(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def verify_e(path, expected):
    assert Path(path).is_file(), path
    assert digest_e(path) == expected, f"Hash mismatch: {path}"

def compare_e(name, actual, expected, atol=1e-5):
    actual = np.asarray(actual, dtype=np.float64)
    expected = np.asarray(expected, dtype=np.float64)
    assert actual.shape == expected.shape
    assert np.isfinite(actual).all()
    assert np.isfinite(expected).all()

    maximum = float(np.max(np.abs(actual - expected)))
    mean = float(np.mean(np.abs(actual - expected)))

    passed = bool(np.allclose(
        actual, expected, rtol=1e-5, atol=atol
    ))

    print(
        f"{name:29s} | "
        f"{'PASS' if passed else 'FAIL':4s} | "
        f"max abs diff={maximum:.9g} | "
        f"mean abs diff={mean:.9g}"
    )

    return {
        "status": "PASS" if passed else "FAIL",
        "maximum_absolute_difference": maximum,
        "mean_absolute_difference": mean,
        "tolerance_atol": atol
    }

def scale_e(raw, state):
    median = np.asarray(state["median"], dtype=np.float32)
    scale = np.asarray(state["scale"], dtype=np.float32)
    clip = float(state["clip_value"])

    assert median.shape == scale.shape == (14,)
    assert (scale > 0).all()
    assert clip == 8.0

    return np.ascontiguousarray(
        np.clip(
            (np.log1p(raw) - median[None, None, :])
            / scale[None, None, :],
            -clip, clip
        ),
        dtype=np.float32
    )

def predict_torch_e(model, x, batch=1024):
    model.eval().cpu()
    model.requires_grad_(False)
    outputs = []

    with torch.inference_mode():
        for start in range(0, len(x), batch):
            inputs = torch.from_numpy(
                np.ascontiguousarray(x[start:start + batch])
            )
            out = model(inputs)
            if isinstance(out, (tuple, list)):
                raise RuntimeError(
                    "Unexpected multi-output forward return."
                )
            outputs.append(out.detach().cpu().numpy())

    return np.concatenate(outputs, axis=0)

assert load_json_e(LOCK_E)["ready_to_run_test"] is False

with np.load(VAL_E, allow_pickle=False) as data:
    X_E = np.asarray(data["X"], dtype=np.float32)
    Y_E = np.asarray(data["y_horizons"], dtype=np.int8)
    EP_E = np.asarray(data["episode_id"]).astype(str)
    MIN_E = np.asarray(data["minute"])
    ONSET_E = np.asarray(data["onset_class"])
    FAMILY_E = np.asarray(data["family_class"])

assert X_E.shape == (18839, 10, 14)

JOINT_TRUE_E = np.where(
    ONSET_E == 0,
    0,
    5 * (ONSET_E - 1) + FAMILY_E
).astype(np.int64)

RESULTS_E = {}

print("=" * 105)
print("CELL 9E — FROZEN VALIDATION INFERENCE REPLAY")
print("=" * 105)

# =============================================================================
# A — XGBOOST H5/H10
# =============================================================================

print("\n1. XGBOOST FLAT140 LOG1P REPLAY")

XGB_DIR_E = (
    MODELS_E / "XGBoost_V2_FLAT140/final_train_models"
)
XGB_VAL_E = (
    MODELS_E
    / "XGBoost_V2_FLAT140/official_validation"
    / "VALIDATION_PREDICTIONS.parquet"
)
XGB_REG_E = load_json_e(
    XGB_DIR_E / "XGBOOST_V2_FINAL_TRAIN_REGISTRY.json"
)

xgb_saved_e = pd.read_parquet(XGB_VAL_E)

assert np.array_equal(
    xgb_saved_e["Episode_ID"].astype(str).to_numpy(), EP_E
)
assert np.array_equal(
    xgb_saved_e["Minute"].to_numpy(), MIN_E
)
assert np.array_equal(
    xgb_saved_e["H5_Actual"].to_numpy(), Y_E[:, 0]
)
assert np.array_equal(
    xgb_saved_e["H10_Actual"].to_numpy(), Y_E[:, 1]
)

# The recorded representation is 10 x 14 log-transformed,
# flattened in the original time-major layout.
X_FLAT_E = np.ascontiguousarray(
    np.log1p(X_E).reshape(len(X_E), 140),
    dtype=np.float32
)

assert X_FLAT_E.shape == (18839, 140)

for horizon, col in [
    ("H5", "H5_Probability"),
    ("H10", "H10_Probability")
]:
    entry = XGB_REG_E["models"][horizon]
    path = XGB_DIR_E / entry["model_file"]

    verify_e(path, entry["model_sha256"])

    booster = xgb.Booster()
    booster.load_model(str(path))

    assert booster.num_features() == 140

    predictions = booster.predict(
        xgb.DMatrix(X_FLAT_E, feature_names=booster.feature_names),
        validate_features=True
    )

    RESULTS_E[f"XGBoost_{horizon}"] = compare_e(
        f"XGBoost {horizon}",
        predictions,
        xgb_saved_e[col].to_numpy(),
        atol=1e-6
    )

# =============================================================================
# B — COMPETING-RISK TCN
# =============================================================================

print("\n2. COMPETING-RISK TCN C02 REPLAY")

TCN_DIR_E = (
    MODELS_E / "TCN_V2_CompetingRisk/final_train_model"
)
TCN_VAL_E = (
    MODELS_E
    / "TCN_V2_CompetingRisk/official_validation"
    / "JOINT_76_CLASS_VALIDATION_PREDICTIONS.npz"
)

tcn_registry_e = load_json_e(
    TCN_DIR_E / "TCN_V2_CR_MODEL_REGISTRY.json"
)
tcn_record_e = load_json_e(
    TCN_DIR_E / "TCN_V2_CR_FINAL_TRAIN_RECORD.json"
)
tcn_scaler_e = load_json_e(
    TCN_DIR_E / tcn_registry_e["scaler_file"]
)

verify_e(
    TCN_DIR_E / tcn_registry_e["model_file"],
    tcn_registry_e["model_sha256"]
)
verify_e(
    TCN_DIR_E / tcn_registry_e["scaler_file"],
    tcn_registry_e["scaler_sha256"]
)

if "TCNCompetingRisk" not in globals():
    raise RuntimeError(
        "Original TCNCompetingRisk class missing. "
        "Rerun original model-definition cell."
    )

TCN_E = TCNCompetingRisk(
    **tcn_record_e["model_architecture"]
)

TCN_E.load_state_dict(
    torch.load(
        TCN_DIR_E / tcn_registry_e["model_file"],
        map_location="cpu",
        weights_only=True
    ),
    strict=True
)

tcn_logits_e = predict_torch_e(
    TCN_E, scale_e(X_E, tcn_scaler_e)
)

tcn_prob_e = torch.softmax(
    torch.from_numpy(tcn_logits_e), dim=1
).numpy()

with np.load(TCN_VAL_E, allow_pickle=False) as z:
    assert np.array_equal(
        z["episode_id"].astype(str), EP_E
    )
    assert np.array_equal(z["minute"], MIN_E)
    assert np.array_equal(z["joint_true"], JOINT_TRUE_E)
    tcn_saved_e = z["joint_probabilities"]

RESULTS_E["TCN_CompetingRisk"] = compare_e(
    "Competing-risk TCN",
    tcn_prob_e,
    tcn_saved_e,
    atol=2e-6
)

# =============================================================================
# C — COMPETING-RISK LSTM
#
# Reconstruct architecture directly from checkpoint tensor shapes.
# A successful probability replay is REQUIRED before treating it
# as equivalent to the original implementation.
# =============================================================================

print("\n3. COMPETING-RISK LSTM C01 REPLAY")

CR_DIR_E = (
    MODELS_E / "LSTM_V2_CompetingRisk/final_train_model"
)
CR_VAL_E = (
    MODELS_E
    / "LSTM_V2_CompetingRisk/official_validation"
    / "JOINT_76_CLASS_VALIDATION_PREDICTIONS_SAFE.npz"
)

cr_registry_e = load_json_e(
    CR_DIR_E / "LSTM_V2_CR_MODEL_REGISTRY.json"
)
cr_record_e = load_json_e(
    CR_DIR_E / "LSTM_V2_CR_FINAL_TRAIN_RECORD.json"
)
cr_scaler_e = load_json_e(
    CR_DIR_E / cr_registry_e["scaler_file"]
)

verify_e(
    CR_DIR_E / cr_registry_e["model_file"],
    cr_registry_e["model_sha256"]
)
verify_e(
    CR_DIR_E / cr_registry_e["scaler_file"],
    cr_registry_e["scaler_sha256"]
)

class CRLSTMReplayE(nn.Module):
    """
    Candidate inference architecture from the recorded tensor
    shapes and model configuration. Must reproduce saved scores.
    """
    def __init__(self, config):
        super().__init__()

        h = config["hidden_dim"]
        d = config["dense_dim"]
        drop = config["dropout"]

        self.lstm = nn.LSTM(
            input_size=config["input_dim"],
            hidden_size=h,
            num_layers=config["num_layers"],
            batch_first=True,
            dropout=drop if config["num_layers"] > 1 else 0.0
        )

        self.projection = nn.Sequential(
            nn.Linear(h, d),
            nn.ReLU(),
            nn.Dropout(drop)
        )

        self.classifier = nn.Linear(d, config["num_classes"])

    def forward(self, x):
        sequence, _ = self.lstm(x)
        features = self.projection(sequence[:, -1, :])
        return self.classifier(features)

CR_E = CRLSTMReplayE(
    cr_record_e["model_architecture"]
)

CR_E.load_state_dict(
    torch.load(
        CR_DIR_E / cr_registry_e["model_file"],
        map_location="cpu",
        weights_only=True
    ),
    strict=True
)

cr_logits_e = predict_torch_e(
    CR_E, scale_e(X_E, cr_scaler_e)
)

cr_prob_e = torch.softmax(
    torch.from_numpy(cr_logits_e), dim=1
).numpy()

with np.load(CR_VAL_E, allow_pickle=False) as z:
    assert np.array_equal(
        z["episode_id"].astype(str), EP_E
    )
    assert np.array_equal(z["minute"], MIN_E)
    assert np.array_equal(z["joint_true"], JOINT_TRUE_E)
    cr_saved_e = z["joint_probabilities"]

RESULTS_E["LSTM_CompetingRisk"] = compare_e(
    "Competing-risk LSTM",
    cr_prob_e,
    cr_saved_e,
    atol=2e-6
)

# =============================================================================
# D — HORIZON LSTM: REQUIRE ORIGINAL MODEL CLASS
# =============================================================================

print("\n4. STACKED HORIZON LSTM C05")

HOR_DIR_E = (
    MODELS_E / "LSTM_V2_Horizon/final_train_model"
)
HOR_VAL_E = (
    MODELS_E
    / "LSTM_V2_Horizon/official_validation"
    / "VALIDATION_PREDICTIONS.csv"
)

hor_registry_e = load_json_e(
    HOR_DIR_E / "LSTM_V2_FINAL_MODEL_REGISTRY.json"
)

verify_e(
    HOR_DIR_E / hor_registry_e["model_file"],
    hor_registry_e["model_sha256"]
)
verify_e(
    HOR_DIR_E / hor_registry_e["scaler_file"],
    hor_registry_e["scaler_sha256"]
)

hor_saved_e = pd.read_csv(HOR_VAL_E)

assert np.array_equal(
    hor_saved_e["Episode_ID"].astype(str).to_numpy(), EP_E
)
assert np.array_equal(
    hor_saved_e["Minute"].to_numpy(), MIN_E
)

for j, h in enumerate([5, 10, 15]):
    assert np.array_equal(
        hor_saved_e[f"Y_H{h}"].to_numpy(),
        Y_E[:, j]
    )

# In the original stacked-horizon model, the head has
# three trainable logits but uses a custom four-interval
# negative-log-likelihood and monotonic probability mapping.
#
# Its checkpoint shapes alone cannot establish that mapping.
# Therefore do not invent a replacement forward function.

RESULTS_E["LSTM_Horizon"] = {
    "status": "BLOCKED_ORIGINAL_CLASS_REQUIRED",
    "checkpoint_sha256": hor_registry_e["model_sha256"],
    "saved_validation_rows_verified": True,
    "reason": (
        "Original four-interval probability head implementation "
        "is not currently defined in Colab."
    )
}

print(
    "Horizon LSTM checkpoint integrity: PASS\n"
    "Saved validation row alignment: PASS\n"
    "Checkpoint inference replay: BLOCKED\n"
    "Original monotonic horizon model class is needed."
)

# =============================================================================
# E — FINAL RESULTS
# =============================================================================

print("\n" + "=" * 105)
print("CELL 9E — REPLAY SUMMARY")
print("=" * 105)

for name, result in RESULTS_E.items():
    print(
        f"{name:28s}: {result['status']}"
    )

completed_e = [
    result for result in RESULTS_E.values()
    if result["status"] in ("PASS", "FAIL")
]

all_pass_e = all(
    result["status"] == "PASS"
    for result in completed_e
)

print("\nExecuted model replays all passed:", all_pass_e)
print("Horizon LSTM full replay:", "NOT COMPLETED")
print("Pre-test protocol modified:", "NO")
print("Official TEST accessed:", "NO")
print("Ready for TEST:", "NO")
print("=" * 105)


In [ ]:
# READ-ONLY DIAGNOSTIC — DO NOT CHANGE MODEL FILES

import numpy as np
import pandas as pd
import xgboost as xgb

print("XGBoost runtime version:", xgb.__version__)

for horizon, col in [
    ("H5", "H5_Probability"),
    ("H10", "H10_Probability"),
]:
    entry = XGB_REG_E["models"][horizon]
    booster = xgb.Booster()
    booster.load_model(str(XGB_DIR_E / entry["model_file"]))

    current = booster.predict(
        xgb.DMatrix(
            X_FLAT_E,
            feature_names=booster.feature_names
        ),
        validate_features=True
    )
    saved = xgb_saved_e[col].to_numpy(dtype=np.float64)

    diff = np.abs(current.astype(np.float64) - saved)
    idx = np.argsort(diff)[-5:][::-1]

    print(f"\n{horizon}: largest discrepancies")
    print("Maximum difference:", diff.max())
    print("Mean difference:", diff.mean())

    for i in idx:
        print(
            f"row={i}, episode={EP_E[i]}, minute={MIN_E[i]}, "
            f"replayed={current[i]:.10f}, "
            f"saved={saved[i]:.10f}, "
            f"abs_diff={diff[i]:.10g}"
        )

print("\nCurrent Competing-Risk LSTM discrepancy:")
print("Maximum:", np.max(np.abs(cr_prob_e - cr_saved_e)))
print("Mean:", np.mean(np.abs(cr_prob_e - cr_saved_e)))

print("\nNo files modified. Official TEST accessed: NO.")

In [ ]:
# READ-ONLY: diagnose XGBoost replay mismatch
for horizon in ["H5", "H10"]:
    entry = XGB_REG_E["models"][horizon]
    booster = xgb.Booster()
    booster.load_model(str(XGB_DIR_E / entry["model_file"]))

    print(f"\n--- {horizon} ---")
    print("Registry keys:", list(entry.keys()))
    print("Boosted rounds:", booster.num_boosted_rounds())
    print("best_iteration attribute:", booster.attr("best_iteration"))
    print("best_ntree_limit attribute:", booster.attr("best_ntree_limit"))
    print("Saved model hash verified:", True)

    col = f"{horizon}_Probability"
    saved = xgb_saved_e[col].to_numpy(dtype=np.float64)

    full = booster.predict(
        xgb.DMatrix(
            X_FLAT_E,
            feature_names=booster.feature_names
        ),
        validate_features=True
    ).astype(np.float64)

    diff = np.abs(full - saved)
    bad = np.flatnonzero(diff > 1e-6)

    print("Rows differing by > 1e-6:", len(bad))
    print("First mismatching rows:", bad[:10].tolist())
    print("Max difference:", float(diff.max()))

print("\nNo files changed. Official TEST accessed: NO.")